# 05 — Insight: Model, Anomali, Risiko, dan Kecukupan Jaringan
**DA Gelar Rasa 2026 · Dashboard Kualitas Udara & Kesehatan**

Dibangun di atas `reports/findings/04_eda.md` dan memakai **hanya** `data/processed/` + `src/style.py` (fungsi di `src/models.py`, `src/eda.py`). Setiap bagian berisi grafik → uji statistik → interpretasi; ringkasan akhir ada di `reports/findings/05_insights.md`.

| Bagian | Isi |
|---|---|
| 0 | **Temuan kualitas data baru: sensor macet (flatline) pada BJB-01** dan data kerja yang dipakai seluruh bagian berikutnya |
| 1 | Faktor paling berpengaruh pada PM2.5 harian (RF/GB, time-split, permutation importance, SHAP vs regresi berganda) |
| 2 | Dampak kesehatan: quasi-Poisson/NB + efek tetap RS + spline waktu + distributed lag; RR per +10 µg/m³; kasus berlebih saat episode |
| 3 | Klaster stasiun (profil bulanan & per jam ternormalisasi) |
| 4 | Anomali: BJB-01 (uji hipotesis berbasis data) dan SM2-01/BJB-01 (polusi tinggi, kasus rendah) |
| 5 | Skor risiko wilayah + sensitivitas bobot |
| 6 | Kecukupan jaringan & prioritas penambahan/perbaikan stasiun + sensitivitas bobot |
| 7 | Matriks bukti per tipe wilayah (dasar rekomendasi) |
| 8 | Konsistensi dengan notebook 02–04, penyimpanan angka kunci |

**Aturan konsistensi.** Angka notebook 02–04 *tidak diubah*. Bila analisis di sini memakai data yang dikoreksi (bagian 0), selisihnya ditampilkan dan diberi label "bertopeng"; angka 04 tetap menjadi acuan "apa adanya". Bagian 7 mengulang perhitungan 04 dan memastikannya identik dengan `key_numbers.json`.

In [1]:
import sys, warnings, json
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import stats
from scipy.cluster import hierarchy as sch
import statsmodels.api as sm
from patsy import dmatrix
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, r2_score, silhouette_score
from IPython.display import display, Markdown

from src import config as cfg, style, eda, models as M, clean as C
from src.eda import rp, fl, fp, pv, bh

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60); pd.set_option("display.max_rows", 120); pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 110)
pio.renderers.default = "plotly_mimetype+notebook_connected"
PR = cfg.PROC_DIR

daily = pd.read_parquet(PR / "aq_daily.parquet"); hourly = pd.read_parquet(PR / "aq_hourly.parquet")
health = pd.read_parquet(PR / "health_aq_daily.parquet"); sites = pd.read_parquet(PR / "sites_clean.parquet"); qc = pd.read_parquet(PR / "qc_summary.parquet")
for df in (daily, hourly):
    df["kode_stasiun"] = df["kode_stasiun"].astype(str)
health["kode_stasiun"] = health["kode_stasiun"].astype(str)
st = sites.set_index("kode_stasiun"); status = qc.set_index("kode_stasiun").status_analisis
KN04 = json.loads((cfg.REPORTS_DIR / "key_numbers.json").read_text(encoding="utf-8"))      # acuan notebook 04 (harus sudah dijalankan)
assert "A" in KN04 and "D" in KN04, "Jalankan notebook 04 lebih dulu"
TIPE = cfg.URUTAN_KELOMPOK; PULAU = cfg.URUTAN_PULAU; WARNA = style.TIPE_LOKASI_WARNA
KN = {}; GAMBAR = []
def simpan(fig, nama, tinggi=800):
    style.simpan_gambar(fig, nama, tinggi); GAMBAR.append(nama)
def legenda_tipe(fig, row=None, col=None):
    for t in TIPE:
        fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", marker=dict(size=10, color=WARNA[t], symbol="square"), name=t), **({} if row is None else dict(row=row, col=col)))
def tafsir(*kalimat):
    display(Markdown("**Interpretasi.** " + " ".join(kalimat)))
def urut_stasiun(kodes):
    s = st.loc[list(kodes)].copy(); s["o1"] = s.pulau.map({k: i for i, k in enumerate(PULAU)}); s["o2"] = s.kelompok_lokasi.map({k: i for i, k in enumerate(TIPE)})
    return s.sort_values(["o1", "o2"]).index.tolist()
print(f"Tabel: jam {rp(len(hourly))} | harian {rp(len(daily))} | kesehatan {rp(len(health))} | stasiun {len(sites)}; status: {status.value_counts().to_dict()}")
print("Angka acuan 04 dimuat dari reports/key_numbers.json, bagian:", list(KN04))

Tabel: jam 591.047 | harian 26.468 | kesehatan 27.486 | stasiun 27; status: {'Layak': 22, 'Terbatas': 4, 'Dikecualikan': 1}
Angka acuan 04 dimuat dari reports/key_numbers.json, bagian: ['meta', 'A', 'B', 'C', 'D', 'E', 'insight']


## 0. Temuan kualitas data baru: sensor macet (flatline) dan data kerja
Nilai PM2.5 kontinu hampir tidak pernah **identik** pada jam-jam berurutan. Deret nilai identik ≥ 24 jam berturut-turut ditandai sebagai *flatline* (sensor macet/nilai tertahan). Ini pemeriksaan baru yang tidak ada pada notebook 02–03 (yang berfokus pada nilai teks, mustahil, dan `qc_flag`).

In [2]:
flat_semua = M.deteksi_flatline(hourly, min_jam=2)
terpanjang = flat_semua.groupby("kode_stasiun").n_jam.max().reindex(sorted(hourly.kode_stasiun.unique())).fillna(1).astype(int)
flat = M.deteksi_flatline(hourly, min_jam=24)
print("Deret nilai identik ≥ 24 jam berturut-turut (PM2.5 per jam):"); display(flat.round(2))
print("\nDeret identik terpanjang di stasiun LAIN (jam):", terpanjang.drop(flat.kode_stasiun.unique(), errors="ignore").describe().round(1).to_dict())
fig = go.Figure(go.Bar(x=terpanjang.index, y=terpanjang.values, marker_color=[style.KEPARAHAN_WARNA["Kritis"] if k in set(flat.kode_stasiun) else style.WARNA_UTAMA for k in terpanjang.index],
                       text=[rp(v) for v in terpanjang.values], textposition="outside", hovertemplate="%{x}: %{y} jam identik berturut-turut<extra></extra>"))
fig.add_hline(y=24, line=dict(dash="dash", color=style.WARNA_TEKS_REDUP), annotation_text="ambang flatline 24 jam", annotation_position="top left")
fig.update_yaxes(type="log", title_text="Deret nilai PM2.5 identik terpanjang (jam, sumbu log)"); fig.update_xaxes(tickangle=-60)
style.terapkan(fig, "Deret nilai identik terpanjang per stasiun: hanya BJB-01 yang macet", 520)
simpan(fig, "05_0_flatline_per_stasiun", 520); fig.show()
flat.to_csv(cfg.REPORTS_DIR / "flatline_terdeteksi.csv", index=False, encoding="utf-8-sig")

Deret nilai identik ≥ 24 jam berturut-turut (PM2.5 per jam):


,kode_stasiun,awal,akhir,n_jam,nilai,n_hari
0,BJB-01,2024-09-07 00:00:00,2025-03-29 13:00:00,4886,336.7,203.6
1,BJB-01,2025-03-29 14:00:00,2025-07-09 08:00:00,2443,3.5,101.8



Deret identik terpanjang di stasiun LAIN (jam): {'count': 25.0, 'mean': 2.7, 'std': 0.6, 'min': 2.0, '25%': 2.0, '50%': 3.0, '75%': 3.0, 'max': 4.0}


In [3]:
# --- data kerja bertopeng: jam flatline -> NaN; hari dengan >= 12 jam flatline -> PM2.5/PM10 harian NaN, bukan episode
hourly["flatline"] = False
for r in flat.itertuples():
    hourly.loc[(hourly.kode_stasiun == r.kode_stasiun) & (hourly.jam >= r.awal) & (hourly.jam <= r.akhir), "flatline"] = True
hourly_m = hourly.copy(); hourly_m.loc[hourly_m.flatline, ["pm25_ug_m3", "pm10_ug_m3"]] = np.nan
fh = hourly[hourly.flatline].groupby(["kode_stasiun", hourly.jam.dt.floor("D")]).size()
hari_flat = set(fh[fh >= 12].index)
daily_m = daily.copy()
daily_m["flatline"] = [(k, t) in hari_flat for k, t in zip(daily_m.kode_stasiun, daily_m.tanggal)]
kol_pm = ["pm25_ug_m3", "pm25_ug_m3_longgar", "pm25_maks_jam_ug_m3", "pm10_ug_m3"]
daily_m.loc[daily_m.flatline, kol_pm] = np.nan
daily_m["episode_m"] = daily_m.episode_asap.astype(bool) & ~daily_m.flatline
print(f"Jam bertopeng: {rp(hourly.flatline.sum())} | hari bertopeng: {rp(daily_m.flatline.sum())} ({', '.join(sorted(daily_m[daily_m.flatline].kode_stasiun.unique()))})")
# lag eksposur kesehatan dihitung ulang dari harian bertopeng; harus identik di luar stasiun terdampak
health_m = M.lag_dari_harian(health, daily_m)
kena = set(daily_m[daily_m.flatline].kode_stasiun)
lain = ~health.kode_stasiun.isin(kena)
for L in M.LAG:
    a, b = health.loc[lain, f"pm25_lag{L}"].to_numpy(float), health_m.loc[lain, f"pm25_lag{L}"].to_numpy(float)
    assert np.allclose(a, b, equal_nan=True), f"lag {L} berbeda pada stasiun tak terdampak"
em = daily_m.set_index(["kode_stasiun", "tanggal"]).episode_m
health_m["episode_m"] = [bool(em.get((k, d), False)) for k, d in zip(health_m.kode_stasiun, health_m.date)]
health_m["ada_paparan_pm25"] = health_m.pm25_lag0.notna()
print("Verifikasi: lag eksposur stasiun lain identik dengan tabel notebook 03 (lulus)")

Jam bertopeng: 7.329 | hari bertopeng: 305 (BJB-01)


Verifikasi: lag eksposur stasiun lain identik dengan tabel notebook 03 (lulus)


In [4]:
# --- dampak pada angka kunci notebook 04: hitung ULANG dengan definisi identik 04, versi 'apa adanya' dan 'bertopeng'
def ringkas_04(dl):
    n = dl.groupby("kode_stasiun").pm25_ug_m3.count(); stas = n[n >= 100].index
    d = dl[dl.pm25_ug_m3.notna() & dl.kode_stasiun.isin(stas)].copy()
    d["tipe"] = d.kode_stasiun.map(st.kelompok_lokasi)
    mean_st = d.groupby("kode_stasiun").pm25_ug_m3.mean()
    kat = C.kategori_ispu_pm25(d.pm25_ug_m3).astype(str)
    epi = dl[(dl["episode_m"] if "episode_m" in dl else dl.episode_asap) & dl.pm25_ug_m3.notna()]
    kw = eda.kruskal({t: d.loc[d.tipe == t, "pm25_ug_m3"].to_numpy() for t in TIPE})
    return {"n_stasiun": len(stas), "rata2_semua_hari_stasiun": d.pm25_ug_m3.mean(), "median_semua_hari_stasiun": d.pm25_ug_m3.median(), "n_stasiun_rata2_lebih_15": int((mean_st > 15).sum()),
            "persen_hari_lewat_bm55": 100 * (d.pm25_ug_m3 > 55).mean(), "persen_hari_lewat_who15": 100 * (d.pm25_ug_m3 > 15).mean(),
            "ispu_tidak_sehat_atau_lebih_persen": 100 * kat.isin(["Tidak Sehat", "Sangat Tidak Sehat", "Berbahaya"]).mean(), "hari_berbahaya": int((kat == "Berbahaya").sum()),
            "n_episode": int(epi.id_episode.nunique()), "n_hari_episode": int(len(epi)),
            "rata2_bjb01": float(mean_st.get("BJB-01", np.nan)), "rata2_tipe_gambut": float(d.loc[d.tipe == "Gambut", "pm25_ug_m3"].mean()), "kw_tipe_epsilon2": kw["epsilon2"], "stasiun_tertinggi": mean_st.idxmax()}
r_asli, r_m = ringkas_04(daily), ringkas_04(daily_m)
A = KN04["A"]
# 1) versi 'apa adanya' HARUS identik dengan 04
assert np.isclose(r_asli["rata2_semua_hari_stasiun"], A["rata2_semua_hari_stasiun"]) and np.isclose(r_asli["persen_hari_lewat_bm55"], A["persen_hari_lewat_bm55"])
assert r_asli["n_stasiun_rata2_lebih_15"] == A["n_stasiun_rata2_lebih_15"] and r_asli["n_episode"] == KN04["B"]["episode"]["n_episode"]
assert np.isclose(r_asli["kw_tipe_epsilon2"], A["kruskal_wallis"]["tipe"]["epsilon2"]) and np.isclose(r_asli["ispu_tidak_sehat_atau_lebih_persen"], A["ispu_tidak_sehat_atau_lebih_persen"])
dampak = pd.DataFrame({"apa_adanya (=04)": r_asli, "bertopeng (tanpa flatline)": r_m})
dampak["selisih"] = [(b - a) if isinstance(a, (int, float, np.integer, np.floating)) and not isinstance(a, bool) else "" for a, b in zip(dampak.iloc[:, 0], dampak.iloc[:, 1])]
print("Perhitungan ulang versi 'apa adanya' identik dengan key_numbers.json notebook 04 (lulus).\nDampak flatline BJB-01 pada angka kunci 04:")
display(dampak)
KN["0_flatline"] = {"n_run": int(len(flat)), "stasiun": sorted(flat.kode_stasiun.unique()), "jam_bertopeng": int(hourly.flatline.sum()), "hari_bertopeng": int(daily_m.flatline.sum()),
                    "run_terpanjang_stasiun_lain_jam": int(terpanjang.drop(flat.kode_stasiun.unique(), errors="ignore").max()), "runs": flat.assign(awal=flat.awal.dt.strftime("%Y-%m-%d %H:%M"), akhir=flat.akhir.dt.strftime("%Y-%m-%d %H:%M")).to_dict("records"),
                    "dampak_pada_04": {k: {"apa_adanya": (None if pd.isna(r_asli[k]) else (r_asli[k] if isinstance(r_asli[k], str) else float(r_asli[k]))), "bertopeng": (r_m[k] if isinstance(r_m[k], str) else float(r_m[k]))} for k in r_asli}}

Perhitungan ulang versi 'apa adanya' identik dengan key_numbers.json notebook 04 (lulus).
Dampak flatline BJB-01 pada angka kunci 04:


,apa_adanya (=04),bertopeng (tanpa flatline),selisih
n_stasiun,25,25,0
rata2_semua_hari_stasiun,41.720629,39.372989,-2.34764
median_semua_hari_stasiun,30.841667,30.766667,-0.075
n_stasiun_rata2_lebih_15,22,22,0
persen_hari_lewat_bm55,17.335149,16.703581,-0.631568
persen_hari_lewat_who15,82.611207,82.811651,0.200444
ispu_tidak_sehat_atau_lebih_persen,17.19485,16.561494,-0.633356
hari_berbahaya,301,98,-203
n_episode,21,20,-1
n_hari_episode,396,192,-204


In [5]:
tafsir(f"Ditemukan {len(flat)} deret nilai identik ≥ 24 jam, seluruhnya pada {', '.join(sorted(flat.kode_stasiun.unique()))}: " + "; ".join(f"{rp(r.n_jam)} jam ({fl(r.n_hari, 0)} hari) bernilai {fl(r.nilai, 2)} µg/m³ ({r.awal:%d-%m-%Y}…{r.akhir:%d-%m-%Y})" for r in flat.itertuples()) +
       f". Stasiun lain paling banyak {int(terpanjang.drop(flat.kode_stasiun.unique(), errors='ignore').max())} jam identik berturut-turut.",
       f"Dengan flatline dibuang, rata-rata BJB-01 turun dari {fl(r_asli['rata2_bjb01'])} menjadi {fl(r_m['rata2_bjb01'])} µg/m³, {r_asli['stasiun_tertinggi']} yang semula stasiun tertinggi digantikan {r_m['stasiun_tertinggi']}, episode kabut asap berkurang dari {r_asli['n_episode']} menjadi {r_m['n_episode']} ({r_asli['n_hari_episode']} → {r_m['n_hari_episode']} hari), dan hari Berbahaya dari {r_asli['hari_berbahaya']} menjadi {r_m['hari_berbahaya']}.",
       "Keputusan: angka notebook 02–04 tetap sebagai acuan 'apa adanya'; seluruh analisis lanjutan di notebook ini memakai data bertopeng. Perbaikan permanen sebaiknya dimasukkan ke aturan pembersihan `src/clean.py` (rekomendasi, belum dijalankan).")

**Interpretasi.** Ditemukan 2 deret nilai identik ≥ 24 jam, seluruhnya pada BJB-01: 4.886 jam (204 hari) bernilai 336,70 µg/m³ (07-09-2024…29-03-2025); 2.443 jam (102 hari) bernilai 3,50 µg/m³ (29-03-2025…09-07-2025). Stasiun lain paling banyak 4 jam identik berturut-turut. Dengan flatline dibuang, rata-rata BJB-01 turun dari 102,1 menjadi 49,2 µg/m³, BJB-01 yang semula stasiun tertinggi digantikan PLB-01, episode kabut asap berkurang dari 21 menjadi 20 (396 → 192 hari), dan hari Berbahaya dari 301 menjadi 98. Keputusan: angka notebook 02–04 tetap sebagai acuan 'apa adanya'; seluruh analisis lanjutan di notebook ini memakai data bertopeng. Perbaikan permanen sebaiknya dimasukkan ke aturan pembersihan `src/clean.py` (rekomendasi, belum dijalankan).

## 1. Faktor paling berpengaruh pada PM2.5 harian
**Target:** log(1 + PM2.5 harian), stasiun dengan ≥ 100 hari sah (data bertopeng). **Fitur:** cuaca (suhu, kelembapan, angin), waktu (bulan, hari-dalam-tahun, hari-minggu), tipe lokasi, karakteristik lokasi (lalu lintas, jarak ke jalan). `baseline_pm25` **sengaja tidak dipakai** (hampir sama dengan target), dan koordinat hanya pada uji ablasi karena dapat menjadi pengenal stasiun.
**Validasi time-split dua lipatan:** (1) latih 2024 → uji 2025 (rentang nilai serupa) dan (2) latih 2024–2025 → uji 2026, yang memuat musim asap jauh lebih parah (di luar rentang latih).
**Pembanding:** regresi berganda (OLS, SE klaster per stasiun, bulan sebagai dummy).

In [6]:
n_h = daily_m.groupby("kode_stasiun").pm25_ug_m3.count(); stas_ml = n_h[n_h >= 100].index.tolist()
F = M.bangun_fitur(daily_m, sites, stas_ml)
FOLD = {"L1: latih 2024 → uji 2025": ((F.tanggal < "2025-01-01"), (F.tanggal >= "2025-01-01") & (F.tanggal < "2026-01-01")),
        "L2: latih 2024–25 → uji 2026": ((F.tanggal < "2026-01-01"), (F.tanggal >= "2026-01-01"))}
grup_utama = [g for g in M.GRUP_FITUR if "Koordinat" not in g]; kol_utama = M.kolom_fitur(grup_utama)
print(f"{len(stas_ml)} stasiun, {rp(len(F))} stasiun-hari; fitur utama ({len(kol_utama)}): {kol_utama}")
for nm, (tr, te) in FOLD.items():
    print(f"{nm}: latih {rp(tr.sum())} | uji {rp(te.sum())} | maks PM2.5 latih {F.loc[tr, 'y'].max():.0f} · uji {F.loc[te, 'y'].max():.0f} µg/m³")

hasil_ml, pred = [], {}
for nm, (tr, te) in FOLD.items():
    Xtr, Xte, ytr, yte = F.loc[tr, kol_utama], F.loc[te, kol_utama], F.loc[tr, "y_log"].to_numpy(), F.loc[te, "y_log"].to_numpy()
    ne = ~F.loc[te, "episode"].to_numpy()
    for jenis, nama in (("rf", "Random Forest"), ("gb", "Gradient Boosting")):
        mdl = M.buat_model(jenis).fit(Xtr, ytr); p = mdl.predict(Xte); pred[(nm, nama)] = (mdl, p)
        hasil_ml.append(dict(lipatan=nm, model=nama, **M.skor(yte, p, ne)))
    ols, p_ols, koef = M.regresi_berganda(F.loc[tr], F.loc[te])
    pred[(nm, "Regresi berganda")] = (ols, p_ols); hasil_ml.append(dict(lipatan=nm, model="Regresi berganda", **M.skor(yte, p_ols, ne)))
    pred[(nm, "koef")] = koef
    hasil_ml.append(dict(lipatan=nm, model="Acuan: rata-rata latih", **M.skor(yte, np.full_like(yte, ytr.mean()), ne)))
ML = pd.DataFrame(hasil_ml); display(ML.round(3))

25 stasiun, 23.929 stasiun-hari; fitur utama (13): ['temperature_c', 'humidity_pct', 'wind_speed_ms', 'bulan', 'doy', 'hari_ke', 'tipe_Perkotaan', 'tipe_Jalan Raya', 'tipe_Industri', 'tipe_Gambut', 'tipe_Latar Bersih', 'lalu_lintas', 'jarak_jalan']
L1: latih 2024 → uji 2025: latih 8.611 | uji 8.506 | maks PM2.5 latih 292 · uji 242 µg/m³
L2: latih 2024–25 → uji 2026: latih 17.117 | uji 6.812 | maks PM2.5 latih 292 · uji 1128 µg/m³


,lipatan,model,r2_log,r2,mae,rmse,r2_non_episode,mae_non_episode
0,L1: latih 2024 → uji 2025,Random Forest,0.935,0.780,4.391,13.044,0.779,4.177
1,L1: latih 2024 → uji 2025,Gradient Boosting,0.920,0.737,5.410,14.240,0.736,5.182
2,L1: latih 2024 → uji 2025,Regresi berganda,0.749,0.472,10.367,20.189,0.478,10.019
3,L1: latih 2024 → uji 2025,Acuan: rata-rata latih,-0.007,-0.105,19.227,29.213,-0.102,18.756
4,L2: latih 2024–25 → uji 2026,Random Forest,0.862,0.257,11.525,62.396,0.795,4.621
5,L2: latih 2024–25 → uji 2026,Gradient Boosting,0.839,0.234,13.070,63.378,0.726,6.076
6,L2: latih 2024–25 → uji 2026,Regresi berganda,0.644,0.119,19.285,67.982,0.450,11.817
7,L2: latih 2024–25 → uji 2026,Acuan: rata-rata latih,-0.012,-0.056,28.126,74.410,-0.091,20.143


In [7]:
# uji statistik: galat absolut RF vs regresi berganda (berpasangan, Wilcoxon) per lipatan
uji = []
for nm, (tr, te) in FOLD.items():
    yte = F.loc[te, "y"].to_numpy()
    e_rf = np.abs(yte - np.expm1(pred[(nm, "Random Forest")][1])); e_ols = np.abs(yte - np.clip(np.expm1(pred[(nm, "Regresi berganda")][1]), 0, None))
    w = stats.wilcoxon(e_rf, e_ols)
    uji.append(dict(lipatan=nm, mae_rf=e_rf.mean(), mae_ols=e_ols.mean(), median_selisih=float(np.median(e_rf - e_ols)), p_wilcoxon=w.pvalue, persen_hari_rf_lebih_baik=float(100 * (e_rf < e_ols).mean())))
uji = pd.DataFrame(uji); display(uji.round(3))

fig = make_subplots(rows=1, cols=2, subplot_titles=("R² pada skala log (utama)", "MAE skala asli (µg/m³)"), horizontal_spacing=0.1)
model_urut = ["Random Forest", "Gradient Boosting", "Regresi berganda"]; warna_m = {"Random Forest": style.WARNA_UTAMA, "Gradient Boosting": "#6C5CE7", "Regresi berganda": "#E17055"}
for m in model_urut:
    x = ML[ML.model == m]
    fig.add_bar(x=x.lipatan, y=x.r2_log, name=m, marker_color=warna_m[m], text=[fl(v, 2) for v in x.r2_log], textposition="outside", legendgroup=m, row=1, col=1)
    fig.add_bar(x=x.lipatan, y=x.mae, name=m, marker_color=warna_m[m], text=[fl(v, 1) for v in x.mae], textposition="outside", showlegend=False, row=1, col=2)
fig.update_yaxes(range=[0, 1.05], row=1, col=1)
style.terapkan(fig, "Kinerja model PM2.5 harian pada data uji (time-split)", 520)
simpan(fig, "05_1_kinerja_model", 520); fig.show()
KN["1_ml"] = {"n_stasiun": len(stas_ml), "n_stasiun_hari": int(len(F)), "kinerja": ML.round(4).to_dict("records"), "uji_wilcoxon_rf_vs_ols": uji.round(4).to_dict("records"), "fitur_utama": kol_utama,
              "maks_pm25_latih_uji": {nm: [float(F.loc[tr, 'y'].max()), float(F.loc[te, 'y'].max())] for nm, (tr, te) in FOLD.items()}}

,lipatan,mae_rf,mae_ols,median_selisih,p_wilcoxon,persen_hari_rf_lebih_baik
0,L1: latih 2024 → uji 2025,4.391,10.367,-3.186,0.0,88.302
1,L2: latih 2024–25 → uji 2026,11.525,19.285,-4.751,0.0,91.133


### 1.1 Ablasi kelompok fitur dan permutation importance
Ablasi menambah kelompok fitur secara bertahap (Random Forest). Permutation importance dihitung **per kelompok** pada data uji (kolom satu kelompok diacak bersama; 10 ulangan; penurunan R² skala log).

In [8]:
urut_abl = [("Cuaca + waktu", ["Cuaca (suhu, kelembapan, angin)", "Waktu (bulan, hari-tahun, hari-minggu)"]),
            ("+ tipe lokasi", ["Cuaca (suhu, kelembapan, angin)", "Waktu (bulan, hari-tahun, hari-minggu)", "Tipe lokasi"]),
            ("+ lalu lintas & jarak jalan", grup_utama), ("+ koordinat (pengenal stasiun)", list(M.GRUP_FITUR))]
abl = []
for nm, (tr, te) in FOLD.items():
    for lab, g in urut_abl:
        kol = M.kolom_fitur(g); mdl = M.buat_model("rf").fit(F.loc[tr, kol], F.loc[tr, "y_log"])
        abl.append(dict(lipatan=nm, set_fitur=lab, **{k: v for k, v in M.skor(F.loc[te, "y_log"].to_numpy(), mdl.predict(F.loc[te, kol])).items() if k in ("r2_log", "mae")}))
abl = pd.DataFrame(abl); display(abl.pivot(index="set_fitur", columns="lipatan", values="r2_log").reindex([a for a, _ in urut_abl]).round(3))

perm = {}
for nm, (tr, te) in FOLD.items():
    for jenis, nama in (("Random Forest", "Random Forest"), ("Gradient Boosting", "Gradient Boosting")):
        mdl = pred[(nm, nama)][0]
        perm[(nm, nama)] = M.permutation_berkelompok(mdl, F.loc[te, kol_utama], F.loc[te, "y_log"].to_numpy(), M.GRUP_FITUR, n_ulang=10, seed=3)
        perm[(nm, nama)]["lipatan"], perm[(nm, nama)]["model"] = nm, nama
PG = pd.concat(perm.values()); display(PG.pivot_table(index="kelompok", columns=["lipatan", "model"], values="penurunan_r2").round(3))

lipatan,L1: latih 2024 → uji 2025,L2: latih 2024–25 → uji 2026
set_fitur,,
Cuaca + waktu,0.666,0.638
+ tipe lokasi,0.897,0.825
+ lalu lintas & jarak jalan,0.935,0.862
+ koordinat (pengenal stasiun),0.935,0.863


lipatan                                L1: latih 2024 → uji 2025               L2: latih 2024–25 → uji 2026              
model                                          Gradient Boosting Random Forest            Gradient Boosting Random Forest
kelompok                                                                                                                 
Cuaca (suhu, kelembapan, angin)                            0.167         0.174                        0.141         0.177
Lalu lintas & jarak ke jalan                               0.773         0.583                        0.439         0.522
Tipe lokasi                                                0.087         0.170                        0.120         0.157
Waktu (bulan, hari-tahun, hari-minggu)                     0.511         0.560                        0.588         0.647

In [9]:
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.04, subplot_titles=list(FOLD))
urut_g = [g for g in M.GRUP_FITUR if "Koordinat" not in g]
for i, nm in enumerate(FOLD, 1):
    for nama, wrn in (("Random Forest", style.WARNA_UTAMA), ("Gradient Boosting", "#6C5CE7")):
        x = perm[(nm, nama)].set_index("kelompok").reindex(urut_g)
        fig.add_bar(y=x.index, x=x.penurunan_r2, orientation="h", name=nama, marker_color=wrn, error_x=dict(type="data", array=x.sd, color=style.WARNA_TEKS_REDUP), legendgroup=nama, showlegend=(i == 1),
                    text=[fl(v, 2) for v in x.penurunan_r2], textposition="outside", row=1, col=i)
fig.update_yaxes(autorange="reversed"); fig.update_xaxes(title_text="Penurunan R² (skala log) saat kelompok diacak")
style.terapkan(fig, "Permutation importance per kelompok fitur (data uji)", 480)
simpan(fig, "05_1_permutation_importance", 480); fig.show()

ukuran = {nm: perm[(nm, "Random Forest")].set_index("kelompok").penurunan_r2 for nm in FOLD}
top = {nm: s.idxmax() for nm, s in ukuran.items()}
KN["1_ml"]["permutation_rf"] = {nm: s.round(4).to_dict() for nm, s in ukuran.items()}; KN["1_ml"]["permutation_gb"] = {nm: perm[(nm, "Gradient Boosting")].set_index("kelompok").penurunan_r2.round(4).to_dict() for nm in FOLD}
KN["1_ml"]["ablasi_r2_log"] = abl.pivot(index="set_fitur", columns="lipatan", values="r2_log").round(4).to_dict("index")

# --- pemisahan perbedaan ANTAR-stasiun vs variasi DALAM-stasiun: acak seluruh data vs acak hanya di dalam stasiun, dan baseline stasiun saja
dalam, base = [], {}
G_CUACA, G_WAKTU = "Cuaca (suhu, kelembapan, angin)", "Waktu (bulan, hari-tahun, hari-minggu)"
for nm, (tr, te) in FOLD.items():
    mdl = pred[(nm, "Random Forest")][0]; Xte = F.loc[te, kol_utama]; yte = F.loc[te, "y_log"].to_numpy(); strata = F.loc[te, "kode_stasiun"].to_numpy()
    for g in (G_CUACA, G_WAKTU):
        m_, sd_ = M.permutation_dalam_strata(mdl, Xte, yte, M.GRUP_FITUR[g], strata, n_ulang=10, seed=4)
        dalam.append(dict(lipatan=nm, kelompok=g, acak_seluruh_data=float(ukuran[nm][g]), acak_dalam_stasiun=m_, sd=sd_))
    mu = F.loc[tr].groupby("kode_stasiun").y_log.mean(); p_st = F.loc[te, "kode_stasiun"].map(mu).to_numpy()
    mm = F.loc[tr].groupby(["kode_stasiun", "bulan"]).y_log.mean()
    p_sm = mm.reindex(pd.MultiIndex.from_arrays([F.loc[te, "kode_stasiun"], F.loc[te, "bulan"]])).to_numpy(); p_sm = np.where(np.isnan(p_sm), p_st, p_sm)
    base[nm] = dict(r2_log_stasiun_saja=float(r2_score(yte, p_st)), r2_log_stasiun_x_bulan=float(r2_score(yte, p_sm)), r2_log_rf_penuh=float(ML[(ML.lipatan == nm) & (ML.model == "Random Forest")].r2_log.iloc[0]))
dalam = pd.DataFrame(dalam); base = pd.DataFrame(base).T
print("Penurunan R² log: acak seluruh data vs acak di dalam stasiun:"); display(dalam.round(3))
print("Baseline tanpa model ML (rata-rata latih per stasiun, dan per stasiun × bulan):"); display(base.round(3))
KN["1_ml"]["permutation_dalam_vs_seluruh"] = dalam.round(4).to_dict("records"); KN["1_ml"]["baseline_stasiun"] = base.round(4).to_dict("index")

Penurunan R² log: acak seluruh data vs acak di dalam stasiun:


,lipatan,kelompok,acak_seluruh_data,acak_dalam_stasiun,sd
0,L1: latih 2024 → uji 2025,"Cuaca (suhu, kelembapan, angin)",0.174,-0.000,0.000
1,L1: latih 2024 → uji 2025,"Waktu (bulan, hari-tahun, hari-minggu)",0.560,0.542,0.007
2,L2: latih 2024–25 → uji 2026,"Cuaca (suhu, kelembapan, angin)",0.177,0.001,0.001
3,L2: latih 2024–25 → uji 2026,"Waktu (bulan, hari-tahun, hari-minggu)",0.647,0.639,0.010


Baseline tanpa model ML (rata-rata latih per stasiun, dan per stasiun × bulan):


,r2_log_stasiun_saja,r2_log_stasiun_x_bulan,r2_log_rf_penuh
L1: latih 2024 → uji 2025,0.600,0.935,0.935
L2: latih 2024–25 → uji 2026,0.447,0.864,0.862


### 1.2 SHAP (Gradient Boosting, lipatan 2)
SHAP (TreeExplainer) pada 1.500 hari uji acak: arah dan besar kontribusi tiap fitur. Warna = nilai fitur (rendah → tinggi).

In [10]:
import shap
nm2 = list(FOLD)[1]; tr2, te2 = FOLD[nm2]
gb = pred[(nm2, "Gradient Boosting")][0]
samp = F.loc[te2, kol_utama].sample(1500, random_state=1)
sv = shap.TreeExplainer(gb).shap_values(samp)
mabs = pd.Series(np.abs(sv).mean(axis=0), index=kol_utama).sort_values(ascending=False)
grup_of = {c: g for g, cs in M.GRUP_FITUR.items() for c in cs}
mabs_g = mabs.groupby(mabs.index.map(grup_of)).sum().sort_values(ascending=False)
print("Rata-rata |SHAP| per fitur:"); display(mabs.round(4).to_frame("mean_abs_shap").T)
print("Per kelompok:"); display(mabs_g.round(4).to_frame("mean_abs_shap").T)
rng = np.random.default_rng(5); top8 = list(mabs.index[:8])
fig = make_subplots(rows=1, cols=2, column_widths=[0.38, 0.62], horizontal_spacing=0.12, subplot_titles=("Rata-rata |SHAP| per kelompok", "Sebaran SHAP fitur teratas (warna = nilai fitur)"))
fig.add_bar(y=mabs_g.index, x=mabs_g.values, orientation="h", marker_color=style.WARNA_UTAMA, text=[fl(v, 2) for v in mabs_g.values], textposition="outside", showlegend=False, row=1, col=1)
for j, c in enumerate(top8):
    v = samp[c].to_numpy(float); z = (v - v.min()) / (np.ptp(v) or 1)
    fig.add_trace(go.Scatter(x=sv[:, kol_utama.index(c)], y=j + rng.uniform(-0.3, 0.3, len(v)), mode="markers", marker=dict(size=4, color=z, colorscale=style.SKALA_KELENGKAPAN, opacity=0.7, showscale=(j == 0), colorbar=dict(title="nilai fitur", thickness=10, len=0.5, x=1.02)),
                             showlegend=False, customdata=v, hovertemplate=c + "<br>SHAP %{x:.3f}<br>nilai %{customdata:.2f}<extra></extra>"), row=1, col=2)
fig.update_yaxes(autorange="reversed", row=1, col=1)
fig.update_yaxes(tickmode="array", tickvals=list(range(8)), ticktext=top8, autorange="reversed", row=1, col=2); fig.update_xaxes(title_text="Kontribusi ke log(1+PM2.5)", row=1, col=2)
fig.update_layout(margin=dict(l=290))
style.terapkan(fig, "SHAP: faktor yang menggerakkan prediksi PM2.5 (lipatan 2)", 560)
simpan(fig, "05_1_shap", 560); fig.show()
KN["1_ml"]["shap_gb_lipatan2"] = {"per_fitur": mabs.round(4).to_dict(), "per_kelompok": mabs_g.round(4).to_dict()}

Rata-rata |SHAP| per fitur:


,bulan,jarak_jalan,tipe_Latar Bersih,lalu_lintas,doy,humidity_pct,temperature_c,wind_speed_ms,tipe_Jalan Raya,tipe_Gambut,tipe_Perkotaan,tipe_Industri,hari_ke
mean_abs_shap,0.2067,0.1921,0.101,0.0934,0.0562,0.0544,0.0529,0.0266,0.0186,0.0174,0.0046,0.0046,0.0002


Per kelompok:


,Lalu lintas & jarak ke jalan,"Waktu (bulan, hari-tahun, hari-minggu)",Tipe lokasi,"Cuaca (suhu, kelembapan, angin)"
mean_abs_shap,0.2855,0.263,0.1463,0.1339


In [11]:
koef = pred[(nm2, "koef")].copy(); koef["abs"] = koef.koef_terstandar.abs(); display(koef.sort_values("abs", ascending=False).drop(columns="abs").head(10).round(4))
fig = go.Figure()
te_m = FOLD[nm2][1]; yo = F.loc[te_m, "y"].to_numpy(); ep_m = F.loc[te_m, "episode"].to_numpy()
for lab, p_, wrn in (("Random Forest", np.expm1(pred[(nm2, "Random Forest")][1]), style.WARNA_UTAMA), ("Regresi berganda", np.clip(np.expm1(pred[(nm2, "Regresi berganda")][1]), 0, None), "#E17055")):
    fig.add_trace(go.Scatter(x=yo[~ep_m], y=p_[~ep_m], mode="markers", marker=dict(size=4, color=wrn, opacity=0.35), name=f"{lab} (bukan episode)"))
    fig.add_trace(go.Scatter(x=yo[ep_m], y=p_[ep_m], mode="markers", marker=dict(size=7, color=wrn, symbol="diamond", line=dict(width=1, color="black")), name=f"{lab} (episode asap)"))
fig.add_trace(go.Scatter(x=[1, 1500], y=[1, 1500], mode="lines", line=dict(dash="dash", color=style.WARNA_TEKS_REDUP), name="garis 1:1"))
fig.update_xaxes(type="log", title_text="PM2.5 teramati (µg/m³, log)"); fig.update_yaxes(type="log", title_text="PM2.5 prediksi (µg/m³, log)")
style.terapkan(fig, "Prediksi vs teramati pada uji 2026: model tidak mampu mengekstrapolasi episode asap di luar rentang latih", 620)
simpan(fig, "05_1_prediksi_vs_teramati", 620); fig.show()
KN["1_ml"]["lipatan2_maks_teramati_vs_prediksi_rf"] = [float(yo.max()), float(np.expm1(pred[(nm2, 'Random Forest')][1]).max())]

,koef_terstandar,se,p
lalu_lintas,0.3094,0.0487,0.0000
bulan_9,0.1878,0.0275,0.0000
tipe_Jalan Raya,-0.1837,0.0540,0.0007
bulan_10,0.1801,0.0282,0.0000
bulan_8,0.1767,0.0246,0.0000
tipe_Latar Bersih,-0.1179,0.0516,0.0222
bulan_7,0.1008,0.0111,0.0000
temperature_c,0.0857,0.0526,0.1036
bulan_6,0.0555,0.0090,0.0000
jarak_jalan,-0.0547,0.0402,0.1732


In [12]:
r1, r2 = ML[ML.lipatan == list(FOLD)[0]].set_index("model"), ML[ML.lipatan == nm2].set_index("model")
pg_top = {nm: (ukuran[nm].idxmax(), ukuran[nm].max()) for nm in FOLD}
dl_ = dalam.set_index(["lipatan", "kelompok"]); L1, L2 = list(FOLD)
tafsir(f"Pada lipatan 1 (uji 2025, rentang nilai serupa) Random Forest mencapai R² log = {fl(r1.loc['Random Forest', 'r2_log'], 2)} dan Gradient Boosting {fl(r1.loc['Gradient Boosting', 'r2_log'], 2)}, dibanding {fl(r1.loc['Regresi berganda', 'r2_log'], 2)} untuk regresi berganda (galat absolut RF lebih kecil pada {fl(uji.iloc[0].persen_hari_rf_lebih_baik, 0)}% hari, Wilcoxon {pv(uji.iloc[0].p_wilcoxon)}). Pada lipatan 2 (uji 2026, puncak asap {fl(F.loc[FOLD[nm2][1], 'y'].max(), 0)} µg/m³ vs maksimum latih {fl(F.loc[FOLD[nm2][0], 'y'].max(), 0)}) R² log {fl(r2.loc['Random Forest', 'r2_log'], 2)} tetapi pada skala asli hanya {fl(r2.loc['Random Forest', 'r2'], 2)}: model berbasis pohon tidak dapat memprediksi di luar rentang latih (prediksi maksimum {fl(KN['1_ml']['lipatan2_maks_teramati_vs_prediksi_rf'][1], 0)} µg/m³).",
       f"Sebagian besar kemampuan prediksi berasal dari perbedaan level antar-stasiun: rata-rata latih per stasiun saja sudah memberi R² log {fl(base.loc[L1, 'r2_log_stasiun_saja'], 2)} (lipatan 1) dan {fl(base.loc[L2, 'r2_log_stasiun_saja'], 2)} (lipatan 2); menambah bulan menjadi {fl(base.loc[L1, 'r2_log_stasiun_x_bulan'], 2)} dan {fl(base.loc[L2, 'r2_log_stasiun_x_bulan'], 2)}, mendekati model penuh. Karena tipe lokasi serta lalu lintas/jarak ke jalan konstan per stasiun, kedua kelompok itu bertindak sebagai pengenal stasiun: pentingnya (permutation {fl(ukuran[L1]['Lalu lintas & jarak ke jalan'], 2)} dan {fl(ukuran[L2]['Lalu lintas & jarak ke jalan'], 2)}) mencerminkan perbedaan antar-stasiun, bukan bukti efek langsung lalu lintas.",
       f"Cuaca tampak penting bila diacak di seluruh data (penurunan R² {fl(dl_.loc[(L1, G_CUACA), 'acak_seluruh_data'], 2)} / {fl(dl_.loc[(L2, G_CUACA), 'acak_seluruh_data'], 2)}), tetapi bila diacak hanya di dalam stasiun penurunannya {fl(dl_.loc[(L1, G_CUACA), 'acak_dalam_stasiun'], 2)} / {fl(dl_.loc[(L2, G_CUACA), 'acak_dalam_stasiun'], 2)}: " + ("cuaca berperan terutama lewat iklim rata-rata tiap stasiun (pembeda antar-stasiun), sedangkan variasi cuaca harian dalam stasiun berpengaruh jauh lebih kecil — selaras dengan notebook 04 (|ρ| harian ≤ 0,23)." if dl_.loc[(L1, G_CUACA), 'acak_dalam_stasiun'] < 0.5 * dl_.loc[(L1, G_CUACA), 'acak_seluruh_data'] else "variasi cuaca harian dalam stasiun juga bermakna bagi model, berbeda dari kesan korelasi harian yang lemah pada notebook 04.") + f" Pengaruh waktu/musim di dalam stasiun (diacak dalam stasiun: {fl(dl_.loc[(L1, G_WAKTU), 'acak_dalam_stasiun'], 2)} / {fl(dl_.loc[(L2, G_WAKTU), 'acak_dalam_stasiun'], 2)}) {'jauh lebih besar daripada cuaca' if dl_.loc[(L1, G_WAKTU), 'acak_dalam_stasiun'] > 2 * dl_.loc[(L1, G_CUACA), 'acak_dalam_stasiun'] else 'sebanding dengan cuaca'}. Dua fitur SHAP teratas pada lipatan 2: {mabs.index[0]} dan {mabs.index[1]}.")

**Interpretasi.** Pada lipatan 1 (uji 2025, rentang nilai serupa) Random Forest mencapai R² log = 0,94 dan Gradient Boosting 0,92, dibanding 0,75 untuk regresi berganda (galat absolut RF lebih kecil pada 88% hari, Wilcoxon p < 0,001). Pada lipatan 2 (uji 2026, puncak asap 1.128 µg/m³ vs maksimum latih 292) R² log 0,86 tetapi pada skala asli hanya 0,26: model berbasis pohon tidak dapat memprediksi di luar rentang latih (prediksi maksimum 195 µg/m³). Sebagian besar kemampuan prediksi berasal dari perbedaan level antar-stasiun: rata-rata latih per stasiun saja sudah memberi R² log 0,60 (lipatan 1) dan 0,45 (lipatan 2); menambah bulan menjadi 0,93 dan 0,86, mendekati model penuh. Karena tipe lokasi serta lalu lintas/jarak ke jalan konstan per stasiun, kedua kelompok itu bertindak sebagai pengenal stasiun: pentingnya (permutation 0,58 dan 0,52) mencerminkan perbedaan antar-stasiun, bukan bukti efek langsung lalu lintas. Cuaca tampak penting bila diacak di seluruh data (penurunan R² 0,17 / 0,18), tetapi bila diacak hanya di dalam stasiun penurunannya 0,00 / 0,00: cuaca berperan terutama lewat iklim rata-rata tiap stasiun (pembeda antar-stasiun), sedangkan variasi cuaca harian dalam stasiun berpengaruh jauh lebih kecil — selaras dengan notebook 04 (|ρ| harian ≤ 0,23). Pengaruh waktu/musim di dalam stasiun (diacak dalam stasiun: 0,54 / 0,64) jauh lebih besar daripada cuaca. Dua fitur SHAP teratas pada lipatan 2: bulan dan jarak_jalan.

## 2. Dampak kesehatan: RR per +10 µg/m³
**Model:** GLM quasi-Poisson (Poisson dengan skala Pearson) dengan **efek tetap RS**, hari dalam minggu, spline alami suhu dan kelembapan (df 3), spline alami waktu (df 20 ≈ 7/tahun), dan CI **robust klaster per RS**. Sampel: RS-hari dengan PM2.5 lag 0–7 lengkap (data bertopeng). **Eksposur dipotong pada 250,4 µg/m³** (batas ISPU Sangat Tidak Sehat) agar RR log-linear tidak diekstrapolasi ke > 1.000 µg/m³; tanpa pemotongan = analisis sensitivitas.
**Distributed lag:** Almon derajat 2 (lag 0–7). Selisih balita vs lansia diuji pada model gabungan (kovarians klaster menangkap korelasi antar-model). Kontrol musiman melalui spline waktu berarti RR menggambarkan variasi **jangka pendek** di sekitar tren/musim — lebih konservatif daripada korelasi mentah di notebook 04 (varian A).

In [13]:
Hs = M.siapkan_kesehatan(health_m, cap=M.CAP_PM25)
Hs["episode_m"] = Hs["episode_m"].astype(bool)
grp = Hs.hospital_id.astype("category").cat.codes.to_numpy()
Xd = M.desain_dasar(Hs, 20)
print(f"RS-hari analisis: {rp(len(Hs))} | RS: {Hs.hospital_id.nunique()} | kolom desain dasar: {Xd.shape[1]} | tanggal {Hs.date.min():%Y-%m-%d}…{Hs.date.max():%Y-%m-%d}")
print("RS per tipe lokasi:", Hs.drop_duplicates("hospital_id").kelompok_lokasi.value_counts().reindex(TIPE).to_dict())
RRL = M.tabel_rr_lag(Hs, Xd, grp, M.KASUS)
tabel_rr = RRL.pivot(index="kasus", columns="lag", values="rr").reindex(M.KASUS)
tabel_rr.index = [M.NAMA_KASUS[k] for k in tabel_rr.index]
print("RR per +10 µg/m³ (lag tunggal, quasi-Poisson, efek tetap RS):"); display(tabel_rr.round(3))
print("Dispersi (skala Pearson) berkisar", round(float(RRL.dispersi.min()), 2), "–", round(float(RRL.dispersi.max()), 2))

RS-hari analisis: 22.742 | RS: 24 | kolom desain dasar: 56 | tanggal 2024-01-08…2026-10-14
RS per tipe lokasi: {'Perkotaan': 6, 'Jalan Raya': 2, 'Industri': 4, 'Gambut': 7, 'Latar Bersih': 5}


RR per +10 µg/m³ (lag tunggal, quasi-Poisson, efek tetap RS):


lag,0,1,2,3,4,5,6,7
ISPA balita (<5 th),1.088,1.086,1.084,1.081,1.078,1.076,1.074,1.074
ISPA lansia (>60 th),1.087,1.086,1.084,1.081,1.079,1.076,1.075,1.075
Asma (IGD),1.088,1.087,1.085,1.082,1.079,1.076,1.075,1.075
PPOK,1.092,1.091,1.088,1.085,1.082,1.078,1.076,1.076
Pernapasan umum,1.076,1.075,1.073,1.071,1.069,1.067,1.066,1.066
Kardiovaskular,1.087,1.087,1.087,1.086,1.085,1.084,1.083,1.082
Kematian,1.094,1.094,1.093,1.092,1.090,1.089,1.088,1.088


Dispersi (skala Pearson) berkisar 0.54 – 1.56


In [14]:
DL = {}
for c in M.KASUS:
    DL[c] = M.dlm_almon(Hs, Xd, Hs[c].astype(float).to_numpy(), grp)
cum = pd.DataFrame({M.NAMA_KASUS[c]: dict(rr=d["rr_kumulatif"], bawah=d["rr_kum_bawah"], atas=d["rr_kum_atas"], p=2 * (1 - stats.norm.cdf(abs(d["beta_kum"] / d["se_kum"])))) for c, d in DL.items()}).T
cum["persen_per10"] = 100 * (cum.rr - 1)
print("RR kumulatif lag 0–7 (DLM Almon; per +10 µg/m³ yang berlangsung pada setiap lag):"); display(cum.round(4))
bl_a, bl_b = "resp_admissions_under5", "resp_admissions_over60"
sd_lag0 = M.selisih_rr(Hs, Xd, grp, bl_a, bl_b, "e0"); sd_cum = M.selisih_rr(Hs, Xd, grp, bl_a, bl_b, [f"e{k}" for k in M.LAG])
print("Selisih balita vs lansia (rasio RR; >1 = balita lebih responsif):"); display(pd.DataFrame({"lag 0": sd_lag0, "kumulatif lag 0–7": sd_cum}).T.round(4))
KN["2_kesehatan"] = {"n_rs_hari": int(len(Hs)), "n_rs": int(Hs.hospital_id.nunique()), "cap_pm25": M.CAP_PM25, "df_waktu": 20,
                     "rr_lag_tunggal": {M.NAMA_KASUS[c]: {int(r.lag): [round(r.rr, 4), round(r.rr_bawah, 4), round(r.rr_atas, 4)] for r in RRL[RRL.kasus == c].itertuples()} for c in M.KASUS},
                     "rr_kumulatif": {k: {kk: float(vv) for kk, vv in v.items()} for k, v in cum.round(5).to_dict("index").items()},
                     "balita_vs_lansia": {"lag0": sd_lag0, "kumulatif": sd_cum}, "dispersi_rentang": [float(RRL.dispersi.min()), float(RRL.dispersi.max())]}

RR kumulatif lag 0–7 (DLM Almon; per +10 µg/m³ yang berlangsung pada setiap lag):


,rr,bawah,atas,p,persen_per10
ISPA balita (<5 th),1.0858,1.0514,1.1214,0.0,8.5840
ISPA lansia (>60 th),1.0856,1.0532,1.1190,0.0,8.5603
Asma (IGD),1.0863,1.0527,1.1209,0.0,8.6260
PPOK,1.0899,1.0551,1.1258,0.0,8.9869
Pernapasan umum,1.0748,1.0521,1.0979,0.0,7.4759
Kardiovaskular,1.0910,1.0702,1.1121,0.0,9.0953
Kematian,1.0965,1.0855,1.1076,0.0,9.6470


Selisih balita vs lansia (rasio RR; >1 = balita lebih responsif):


,beta_a,beta_b,selisih,se,z,p,rasio_rr,rasio_bawah,rasio_atas
lag 0,0.0840,0.0834,0.0006,0.0013,0.4413,0.6590,1.0006,0.9981,1.0031
kumulatif lag 0–7,0.0103,0.0103,-0.0000,0.0001,-0.0007,0.9994,1.0000,0.9997,1.0003


In [15]:
fig = make_subplots(rows=1, cols=3, column_widths=[0.36, 0.30, 0.34], horizontal_spacing=0.07,
                    subplot_titles=("RR per lag tunggal (+10 µg/m³)", "RR per lag: DLM Almon (bentuk U = artefak polinomial kuadrat + kolinearitas lag)", "RR kumulatif lag 0–7 per jenis kasus"))
for c, wrn in ((bl_a, "#E17055"), (bl_b, "#3B82C4")):
    x = RRL[RRL.kasus == c].sort_values("lag")
    fig.add_trace(go.Scatter(x=list(x.lag) + list(x.lag[::-1]), y=list(x.rr_atas) + list(x.rr_bawah[::-1]), fill="toself", fillcolor=wrn, opacity=0.15, line=dict(width=0), hoverinfo="skip", showlegend=False), row=1, col=1)
    fig.add_trace(go.Scatter(x=x.lag, y=x.rr, mode="lines+markers", line=dict(color=wrn, width=3), name=M.NAMA_KASUS[c], legendgroup=c), row=1, col=1)
    t = DL[c]["tabel_lag"]
    fig.add_trace(go.Scatter(x=list(t.lag) + list(t.lag[::-1]), y=list(t.rr_atas) + list(t.rr_bawah[::-1]), fill="toself", fillcolor=wrn, opacity=0.15, line=dict(width=0), hoverinfo="skip", showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=t.lag, y=t.rr, mode="lines+markers", line=dict(color=wrn, width=3), showlegend=False, legendgroup=c), row=1, col=2)
cum_u = cum.reindex([M.NAMA_KASUS[k] for k in M.KASUS]).iloc[::-1]
fig.add_trace(go.Scatter(x=cum_u.rr, y=cum_u.index, mode="markers", marker=dict(size=10, color=[("#E17055" if "balita" in k else "#3B82C4" if "lansia" in k else style.WARNA_UTAMA) for k in cum_u.index]),
                         error_x=dict(type="data", symmetric=False, array=(cum_u.atas - cum_u.rr).values, arrayminus=(cum_u.rr - cum_u.bawah).values), showlegend=False,
                         hovertemplate="%{y}<br>RR %{x:.3f}<extra></extra>"), row=1, col=3)
for c in (1, 2, 3):
    if c < 3: fig.add_hline(y=1, line=dict(color=style.WARNA_TEKS_REDUP, dash="dash", width=1), row=1, col=c)
fig.add_vline(x=1, line=dict(color=style.WARNA_TEKS_REDUP, dash="dash", width=1), row=1, col=3)
fig.update_xaxes(title_text="Lag (hari)", dtick=1, row=1, col=1); fig.update_xaxes(title_text="Lag (hari)", dtick=1, row=1, col=2); fig.update_xaxes(title_text="RR per +10 µg/m³ (CI 95%)", row=1, col=3)
fig.update_yaxes(title_text="Risiko relatif", col=1); fig.update_layout(margin=dict(l=70, r=30))
style.terapkan(fig, "Risiko relatif rawat/kasus per +10 µg/m³ PM2.5: balita vs lansia", 520)
simpan(fig, "05_2_rr_lag_dan_kumulatif", 520); fig.show()

Resorting to unclean kill browser.


Resorting to unclean kill browser.


In [16]:
# --- sensitivitas RR kumulatif (balita & lansia) terhadap pilihan analitik
def rr_kum(Hx, df_waktu=20, kasus=(bl_a, bl_b)):
    gx = Hx.hospital_id.astype("category").cat.codes.to_numpy(); Xx = M.desain_dasar(Hx, df_waktu)
    out = {}
    for c in kasus:
        d_ = M.dlm_almon(Hx, Xx, Hx[c].astype(float).to_numpy(), gx); out[c] = (d_["rr_kumulatif"], d_["rr_kum_bawah"], d_["rr_kum_atas"])
    return out
varian = {"Utama (potong 250,4; df 20)": Hs}
Hnc = M.siapkan_kesehatan(health_m, cap=None); varian["Tanpa pemotongan eksposur"] = Hnc
varian["Hanya stasiun Layak"] = Hs[Hs.status_analisis_stasiun == "Layak"]
varian["Tanpa RS stasiun gambut"] = Hs[Hs.kelompok_lokasi != "Gambut"]
varian["Tanpa BJB-01 & SM2-01"] = Hs[~Hs.kode_stasiun.isin(["BJB-01", "SM2-01"])]
sens = []
for nm, Hx in varian.items():
    r = rr_kum(Hx); sens.append(dict(varian=nm, n_rs=Hx.hospital_id.nunique(), n_rs_hari=len(Hx), **{f"{k}_{s}": v for k, vv in r.items() for s, v in zip(("rr", "bawah", "atas"), vv)}))
for dfw in (10, 30):
    r = rr_kum(Hs, dfw); sens.append(dict(varian=f"Spline waktu df {dfw}", n_rs=Hs.hospital_id.nunique(), n_rs_hari=len(Hs), **{f"{k}_{s}": v for k, vv in r.items() for s, v in zip(("rr", "bawah", "atas"), vv)}))
sens = pd.DataFrame(sens); sens.columns = [c.replace(bl_a, "balita").replace(bl_b, "lansia") for c in sens.columns]; display(sens.round(3))
# negative binomial (alpha dari momen Poisson) pada lag 0
nb = []
for c in (bl_a, bl_b):
    y = Hs[c].astype(float).to_numpy(); X0 = Xd.copy(); X0["e0"] = Hs["e0"].to_numpy()
    rp_ = M.fit_poisson(y, X0, grp); a_ = M.alpha_nb(rp_, y); rn_ = M.fit_nb(y, X0, grp, a_)
    nb.append(dict(kasus=M.NAMA_KASUS[c], alpha=a_, rr_quasiPoisson=M.rr(rp_, "e0")["rr"], rr_NB=M.rr(rn_, "e0")["rr"], bawah_NB=M.rr(rn_, "e0")["rr_bawah"], atas_NB=M.rr(rn_, "e0")["rr_atas"]))
nb = pd.DataFrame(nb); print("Negative binomial vs quasi-Poisson (lag 0):"); display(nb.round(4))
KN["2_kesehatan"]["sensitivitas_kumulatif"] = sens.round(4).to_dict("records"); KN["2_kesehatan"]["nb_vs_qp_lag0"] = nb.round(4).to_dict("records")

,varian,n_rs,n_rs_hari,balita_rr,balita_bawah,balita_atas,lansia_rr,lansia_bawah,lansia_atas
0,"Utama (potong 250,4; df 20)",24,22742,1.086,1.051,1.121,1.086,1.053,1.119
1,Tanpa pemotongan eksposur,24,22742,1.021,1.014,1.028,1.022,1.015,1.029
2,Hanya stasiun Layak,22,21930,1.086,1.051,1.122,1.086,1.053,1.120
3,Tanpa RS stasiun gambut,17,16822,1.073,1.047,1.101,1.074,1.048,1.100
4,Tanpa BJB-01 & SM2-01,22,21397,1.086,1.051,1.122,1.086,1.053,1.119
5,Spline waktu df 10,24,22742,1.088,1.053,1.124,1.088,1.055,1.122
6,Spline waktu df 30,24,22742,1.087,1.053,1.121,1.086,1.055,1.119


Negative binomial vs quasi-Poisson (lag 0):


,kasus,alpha,rr_quasiPoisson,rr_NB,bawah_NB,atas_NB
0,ISPA balita (<5 th),0.0091,1.0876,1.0914,1.0535,1.1306
1,ISPA lansia (>60 th),0.0075,1.0870,1.0901,1.0551,1.1261


In [17]:
# --- bentuk respons-eksposur: RR per kelas rata-rata PM2.5 7 hari (tanpa pemotongan) vs ekstrapolasi linear dari model utama
bins = [-np.inf, 15.5, 35, 55.4, 150.4, 250.4, np.inf]; blab = ["≤15,5", "15,5–35", "35–55,4", "55,4–150,4", "150,4–250,4", ">250,4"]
Hnc["p7"] = Hnc[[f"e{k}" for k in M.LAG]].mean(axis=1) * 10; Hnc["kelas"] = pd.cut(Hnc.p7, bins, labels=blab)
Xnc = M.desain_dasar(Hnc, 20); gnc = Hnc.hospital_id.astype("category").cat.codes.to_numpy()
er = []
for c in (bl_a, bl_b):
    X = Xnc.copy()
    for lb in blab[1:]: X["k_" + lb] = (Hnc.kelas == lb).astype(float).to_numpy()
    r = M.fit_poisson(Hnc[c].astype(float).to_numpy(), X, gnc)
    er.append(dict(kasus=M.NAMA_KASUS[c], kelas=blab[0], n_hari=int((Hnc.kelas == blab[0]).sum()), rr=1.0, rr_bawah=1.0, rr_atas=1.0, p7_rata2=float(Hnc.loc[Hnc.kelas == blab[0], "p7"].mean())))
    for lb in blab[1:]:
        er.append(dict(kasus=M.NAMA_KASUS[c], kelas=lb, n_hari=int((Hnc.kelas == lb).sum()), p7_rata2=float(Hnc.loc[Hnc.kelas == lb, "p7"].mean()), **{k: v for k, v in M.rr(r, "k_" + lb).items() if k in ("rr", "rr_bawah", "rr_atas")}))
ER = pd.DataFrame(er)
ref_p7 = ER.groupby("kasus").p7_rata2.transform("first")
for c in (bl_a, bl_b):
    m_ = ER.kasus == M.NAMA_KASUS[c]
    ER.loc[m_, "rr_linier_model_utama"] = np.exp(DL[c]["beta_kum"] * (ER.loc[m_, "p7_rata2"].clip(upper=M.CAP_PM25) - ref_p7[m_]) / M.SATUAN)
print("RR per kelas rata-rata PM2.5 7 hari (acuan: ≤15,5), eksposur TANPA pemotongan, vs ekstrapolasi linear model utama (dipotong 250,4):"); display(ER.round(3))
fig = go.Figure()
for c, wrn in (("ISPA balita (<5 th)", "#E17055"), ("ISPA lansia (>60 th)", "#3B82C4")):
    x = ER[ER.kasus == c]
    fig.add_trace(go.Scatter(x=x.kelas, y=x.rr, mode="lines+markers", line=dict(color=wrn, width=3), marker=dict(size=10), name=c, error_y=dict(type="data", symmetric=False, array=(x.rr_atas - x.rr).values, arrayminus=(x.rr - x.rr_bawah).values, color=wrn)))
x = ER[ER.kasus == "ISPA balita (<5 th)"]
fig.add_trace(go.Scatter(x=x.kelas, y=x.rr_linier_model_utama, mode="lines", line=dict(color=style.WARNA_TEKS_REDUP, dash="dash"), name="ekstrapolasi linear (RR 1,086 per +10 µg/m³, dipotong 250,4)"))
fig.add_hline(y=1, line=dict(color=style.WARNA_TEKS_REDUP, width=1))
fig.update_xaxes(title_text="Kelas rata-rata PM2.5 7 hari (µg/m³)"); fig.update_yaxes(title_text="Risiko relatif vs kelas ≤15,5", type="log")
style.terapkan(fig, "Bentuk respons-eksposur: RR per kelas PM2.5 (garis putus = ekstrapolasi linear)", 520)
simpan(fig, "05_2_respons_eksposur_kelas", 520); fig.show()
KN["2_kesehatan"]["respons_eksposur_kelas"] = ER.round(4).to_dict("records")

# --- heterogenitas: RR per kelompok tipe lokasi RS (lag 0) dan RR per RS (rata-rata 7 hari)
Hs["e7"] = Hs[[f"e{k}" for k in M.LAG]].mean(axis=1)
het = []
for t in TIPE:
    Hx = Hs[Hs.kelompok_lokasi == t]
    if Hx.hospital_id.nunique() < 2: continue
    gx = Hx.hospital_id.astype("category").cat.codes.to_numpy(); Xx = M.desain_dasar(Hx, 20)
    for c in (bl_a, bl_b):
        X0 = Xx.copy(); X0["e7"] = Hx["e7"].to_numpy(); r = M.fit_poisson(Hx[c].astype(float).to_numpy(), X0, gx)
        het.append(dict(tipe=t, n_rs=Hx.hospital_id.nunique(), kasus=M.NAMA_KASUS[c], **M.rr(r, "e7")))
het = pd.DataFrame(het); print("RR per +10 µg/m³ (rata-rata PM2.5 7 hari) menurut tipe lokasi RS:"); display(het.round(3))
def rr_per_rs(outcome="balita_lansia", kol="e7", df_t=14):
    rows = []
    for hid, x in Hs.groupby("hospital_id"):
        if len(x) < 200: continue
        Xr = dmatrix(f"dow + cr(temp_c, df=3) + cr(rhum_pct, df=3) + cr(t, df={df_t})", x, return_type="dataframe"); Xr["e"] = x[kol].to_numpy()
        r = sm.GLM(x[outcome].to_numpy(float), Xr, family=sm.families.Poisson()).fit(scale="X2", cov_type="HC1")
        rows.append(dict(hospital_id=hid, kode_stasiun=x.kode_stasiun.iloc[0], tipe=x.kelompok_lokasi.iloc[0], n=len(x), **M.rr(r, "e")))
    return pd.DataFrame(rows).set_index("hospital_id")
rr_rs = rr_per_rs(); print("RR per RS (balita+lansia, 7 hari):"); display(rr_rs.sort_values("rr").round(3))
KN["2_kesehatan"]["heterogenitas_tipe"] = het.round(4).to_dict("records"); KN["2_kesehatan"]["rr_per_rs_rentang"] = [float(rr_rs.rr.min()), float(rr_rs.rr.max())]

RR per kelas rata-rata PM2.5 7 hari (acuan: ≤15,5), eksposur TANPA pemotongan, vs ekstrapolasi linear model utama (dipotong 250,4):


,kasus,kelas,n_hari,rr,rr_bawah,rr_atas,p7_rata2,rr_linier_model_utama
0,ISPA balita (<5 th),"≤15,5",4126,1.000,1.000,1.000,11.456,1.000
1,ISPA balita (<5 th),"15,5–35",9164,1.117,0.957,1.304,24.624,1.115
2,ISPA balita (<5 th),"35–55,4",5758,1.720,1.360,2.177,44.883,1.317
3,ISPA balita (<5 th),"55,4–150,4",3550,2.735,2.108,3.547,77.868,1.728
4,ISPA balita (<5 th),"150,4–250,4",60,4.044,2.477,6.601,172.059,3.753
5,ISPA balita (<5 th),">250,4",84,4.536,3.458,5.951,516.357,7.155
6,ISPA lansia (>60 th),"≤15,5",4126,1.000,1.000,1.000,11.456,1.000
7,ISPA lansia (>60 th),"15,5–35",9164,1.074,0.943,1.223,24.624,1.114
8,ISPA lansia (>60 th),"35–55,4",5758,1.614,1.305,1.997,44.883,1.316
9,ISPA lansia (>60 th),"55,4–150,4",3550,2.499,1.978,3.158,77.868,1.725


RR per +10 µg/m³ (rata-rata PM2.5 7 hari) menurut tipe lokasi RS:


,tipe,n_rs,kasus,rr,rr_bawah,rr_atas,beta,se,p
0,Perkotaan,6,ISPA balita (<5 th),1.062,1.050,1.074,0.060,0.006,0.000
1,Perkotaan,6,ISPA lansia (>60 th),1.063,1.049,1.076,0.061,0.007,0.000
2,Jalan Raya,2,ISPA balita (<5 th),1.047,0.938,1.168,0.046,0.056,0.416
3,Jalan Raya,2,ISPA lansia (>60 th),1.021,0.917,1.137,0.021,0.055,0.701
4,Industri,4,ISPA balita (<5 th),1.206,1.168,1.244,0.187,0.016,0.000
5,Industri,4,ISPA lansia (>60 th),1.189,1.158,1.221,0.173,0.014,0.000
6,Gambut,7,ISPA balita (<5 th),1.052,1.006,1.102,0.051,0.023,0.028
7,Gambut,7,ISPA lansia (>60 th),1.053,1.009,1.100,0.052,0.022,0.017
8,Latar Bersih,5,ISPA balita (<5 th),1.251,1.224,1.278,0.224,0.011,0.000
9,Latar Bersih,5,ISPA lansia (>60 th),1.209,1.178,1.242,0.190,0.013,0.000


RR per RS (balita+lansia, 7 hari):


,kode_stasiun,tipe,n,rr,rr_bawah,rr_atas,beta,se,p
hospital_id,,,,,,,,,
RS-XXX-06,BGR-01,Latar Bersih,1011,0.467,0.078,2.785,-0.760,0.911,0.404
RS-XXX-02,JKT-02,Industri,1011,0.905,0.807,1.015,-0.099,0.059,0.089
RS-XXX-18,PLU-01,Latar Bersih,1011,0.958,0.652,1.406,-0.043,0.196,0.826
RS-XXX-14,BGL-01,Perkotaan,1011,0.999,0.833,1.197,-0.001,0.093,0.990
RS-XXX-03,JKT-03,Jalan Raya,1011,1.015,0.903,1.140,0.015,0.059,0.804
RS-XXX-05,KTG-02,Gambut,1011,1.034,1.024,1.045,0.034,0.005,0.000
RS-XXX-04,KTG-01,Perkotaan,1011,1.036,1.025,1.046,0.035,0.005,0.000
RS-XXX-01,JKT-01,Perkotaan,1011,1.081,0.969,1.205,0.077,0.055,0.163
RS-XXX-22,PSW-01,Perkotaan,1011,1.082,1.070,1.093,0.078,0.005,0.000


In [18]:
# --- kasus berlebih saat episode kabut asap (acuan: median PM2.5 stasiun pada hari non-episode Jul–Okt)
dnm = daily_m[~daily_m.episode_m & daily_m.pm25_ug_m3.notna() & daily_m.bulan.isin([7, 8, 9, 10])]
x_ref_st = dnm.groupby("kode_stasiun").pm25_ug_m3.median()
Hs["x_ref"] = Hs.kode_stasiun.map(x_ref_st)
mask_ep = Hs.episode_m.to_numpy()
print(f"Hari episode (lag lengkap, tanpa BJB-01): {int(mask_ep.sum())} pada RS {sorted(Hs.loc[mask_ep, 'hospital_id'].unique())}")
Dnc = {c: M.dlm_almon(Hnc, M.desain_dasar(Hnc, 20), Hnc[c].astype(float).to_numpy(), Hnc.hospital_id.astype("category").cat.codes.to_numpy()) for c in (bl_a, bl_b, "resp")}
Hnc["x_ref"] = Hnc.kode_stasiun.map(x_ref_st); Hnc["episode_m"] = Hnc["episode_m"].astype(bool)
baris = []
for c in (bl_a, bl_b, "resp"):
    for hid in sorted(Hs.loc[mask_ep, "hospital_id"].unique()):
        mk = mask_ep & (Hs.hospital_id.to_numpy() == hid)
        e = M.kasus_berlebih(Hs, DL[c], c, mk, Hs.x_ref, n_sim=500, seed=11)
        epi = Hs[(Hs.hospital_id == hid) & Hs.episode_m]; non = Hs[(Hs.hospital_id == hid) & ~Hs.episode_m & Hs.bulan.isin([7, 8, 9, 10])]
        emp = (epi[c].astype(float).mean() - non[c].astype(float).mean()) * len(epi)
        mk2 = (Hnc.episode_m.to_numpy()) & (Hnc.hospital_id.to_numpy() == hid)
        e_nc = M.kasus_berlebih(Hnc, Dnc[c], c, mk2, Hnc.x_ref, n_sim=300, seed=11)["berlebih"]
        baris.append(dict(kasus=M.NAMA_KASUS[c], hospital_id=hid, kode_stasiun=Hs[Hs.hospital_id == hid].kode_stasiun.iloc[0], n_hari_episode=e["n_hari"], teramati=e["kasus_teramati"], berlebih_model=e["berlebih"], bawah=e["bawah"], atas=e["atas"],
                          persen_dari_kasus=e["persen_dari_kasus"], berlebih_empiris=emp, berlebih_model_tanpa_potong=e_nc))
EX = pd.DataFrame(baris)
tot = EX.groupby("kasus").agg(n_hari_episode=("n_hari_episode", "sum"), teramati=("teramati", "sum"), berlebih_model=("berlebih_model", "sum"), bawah=("bawah", "sum"), atas=("atas", "sum"), berlebih_empiris=("berlebih_empiris", "sum"), berlebih_tanpa_potong=("berlebih_model_tanpa_potong", "sum"))
tot["persen_dari_kasus"] = 100 * tot.berlebih_model / tot.teramati
print("Kasus berlebih saat episode (jumlah seluruh RS; CI dari simulasi θ; jumlah batas CI = batas konservatif):"); display(tot.round(1))
print("Per RS (ISPA balita & lansia):"); display(EX[EX.kasus.isin([M.NAMA_KASUS[bl_a], M.NAMA_KASUS[bl_b]])].round(1))
KN["2_kesehatan"]["kasus_berlebih"] = {"total": tot.round(2).to_dict("index"), "per_rs": EX.round(2).to_dict("records"), "n_hari_episode": int(mask_ep.sum()), "acuan": "median PM2.5 stasiun pada hari non-episode Jul–Okt"}

Hari episode (lag lengkap, tanpa BJB-01): 148 pada RS ['RS-XXX-04', 'RS-XXX-05', 'RS-XXX-19', 'RS-XXX-20', 'RS-XXX-21']


Kasus berlebih saat episode (jumlah seluruh RS; CI dari simulasi θ; jumlah batas CI = batas konservatif):


,n_hari_episode,teramati,berlebih_model,bawah,atas,berlebih_empiris,berlebih_tanpa_potong,persen_dari_kasus
kasus,,,,,,,,
ISPA balita (<5 th),148,4957.0,3402.9,2639.6,3857.0,2864.1,2401.6,68.6
ISPA lansia (>60 th),148,5588.0,3821.2,3027.3,4314.3,3163.8,2775.7,68.4
Pernapasan umum,148,17740.0,11740.8,9757.9,13098.8,9933.9,8754.7,66.2


Per RS (ISPA balita & lansia):


,kasus,hospital_id,kode_stasiun,n_hari_episode,teramati,berlebih_model,bawah,atas,persen_dari_kasus,berlebih_empiris,berlebih_model_tanpa_potong
0,ISPA balita (<5 th),RS-XXX-04,KTG-01,44,1397.0,1113.4,877.2,1237.4,79.7,897.3,918.1
1,ISPA balita (<5 th),RS-XXX-05,KTG-02,44,1406.0,1118.5,880.3,1243.8,79.6,929.3,928.1
2,ISPA balita (<5 th),RS-XXX-19,STG-01,5,218.0,160.8,132.5,177.9,73.7,156.2,78.0
3,ISPA balita (<5 th),RS-XXX-20,PKN-01,6,454.0,349.8,281.2,389.5,77.1,364.6,263.5
4,ISPA balita (<5 th),RS-XXX-21,PLB-01,49,1482.0,660.3,468.5,808.5,44.6,516.7,214.0
5,ISPA lansia (>60 th),RS-XXX-04,KTG-01,44,1588.0,1263.5,1016.7,1400.0,79.6,1041.1,1078.0
6,ISPA lansia (>60 th),RS-XXX-05,KTG-02,44,1578.0,1254.9,1009.7,1390.6,79.5,1016.4,1066.4
7,ISPA lansia (>60 th),RS-XXX-19,STG-01,5,238.0,174.3,144.8,192.8,73.2,166.2,86.0
8,ISPA lansia (>60 th),RS-XXX-20,PKN-01,6,500.0,383.8,313.7,425.9,76.8,397.8,294.9
9,ISPA lansia (>60 th),RS-XXX-21,PLB-01,49,1684.0,744.7,542.4,904.9,44.2,542.3,250.4


In [19]:
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.05, subplot_titles=("ISPA balita (<5 th)", "ISPA lansia (>60 th)"))
for i, c in enumerate((bl_a, bl_b), 1):
    x = EX[EX.kasus == M.NAMA_KASUS[c]].sort_values("berlebih_model")
    wrn = "#E17055" if i == 1 else "#3B82C4"
    fig.add_bar(y=x.hospital_id + " (" + x.kode_stasiun + ")", x=x.berlebih_model, orientation="h", marker_color=wrn, name="model (potong 250)", showlegend=(i == 1),
                error_x=dict(type="data", symmetric=False, array=(x.atas - x.berlebih_model).values, arrayminus=(x.berlebih_model - x.bawah).values),
                text=[f"{fl(v, 0)} ({fl(p, 0)}%)" for v, p in zip(x.berlebih_model, x.persen_dari_kasus)], textposition="outside", row=1, col=i)
    fig.add_trace(go.Scatter(x=x.berlebih_empiris, y=x.hospital_id + " (" + x.kode_stasiun + ")", mode="markers", marker=dict(symbol="diamond", size=10, color=style.WARNA_TEKS), name="selisih empiris episode vs non-episode", showlegend=(i == 1)), row=1, col=i)
fig.update_xaxes(title_text="Kasus berlebih selama episode (jumlah hari episode)")
style.terapkan(fig, "Estimasi kasus berlebih saat episode kabut asap (batang = model + CI 95%; ◆ = selisih empiris)", 560)
simpan(fig, "05_2_kasus_berlebih_episode", 560); fig.show()

In [20]:
ub, ul = cum.loc[M.NAMA_KASUS[bl_a]], cum.loc[M.NAMA_KASUS[bl_b]]
tb = tot.loc[M.NAMA_KASUS[bl_a]]; tl = tot.loc[M.NAMA_KASUS[bl_b]]; tr_ = tot.loc[M.NAMA_KASUS["resp"]]
sig = cum[cum.p < 0.05]
sj = sens.set_index("varian"); ek = ER[ER.kasus == "ISPA balita (<5 th)"].set_index("kelas")
tinggi = ek.iloc[-1]; kls_t = ek.index[-1]
tafsir(f"RR kumulatif lag 0–7 per +10 µg/m³ (eksposur dipotong 250,4): ISPA balita {fl(ub.rr, 3)} (CI {fl(ub.bawah, 3)}–{fl(ub.atas, 3)}) dan ISPA lansia {fl(ul.rr, 3)} ({fl(ul.bawah, 3)}–{fl(ul.atas, 3)}), yaitu +{fl(ub.persen_per10, 1)}% dan +{fl(ul.persen_per10, 1)}% kasus per +10 µg/m³; {len(sig)} dari {len(cum)} jenis kasus signifikan. Hasil tahan terhadap spline waktu (df 10–30: {fl(sj.loc['Spline waktu df 10', 'balita_rr'], 3)}–{fl(sj.loc['Spline waktu df 30', 'balita_rr'], 3)}), negative binomial (lag 0: {fl(nb.iloc[0].rr_NB, 3)} vs {fl(nb.iloc[0].rr_quasiPoisson, 3)}), serta pembatasan ke stasiun Layak ({fl(sj.loc['Hanya stasiun Layak', 'balita_rr'], 3)}).",
       f"Namun RR bergantung kuat pada rentang eksposur: tanpa pemotongan menjadi {fl(sj.loc['Tanpa pemotongan eksposur', 'balita_rr'], 3)} (balita), dan RR per kelas PM2.5 menunjukkan respons yang jenuh — pada kelas {kls_t} RR teramati {fl(tinggi.rr, 2)} (CI {fl(tinggi.rr_bawah, 2)}–{fl(tinggi.rr_atas, 2)}) padahal ekstrapolasi linear memperkirakan {fl(tinggi.rr_linier_model_utama, 2)}; sebaliknya pada kelas 55,4–150,4 teramati {fl(ek.loc['55,4–150,4', 'rr'], 2)} vs linear {fl(ek.loc['55,4–150,4', 'rr_linier_model_utama'], 2)} — model satu-kemiringan {'meremehkan kelas menengah dan melebih-lebihkan kelas ekstrem' if (ek.loc['55,4–150,4', 'rr'] > ek.loc['55,4–150,4', 'rr_linier_model_utama'] and tinggi.rr < tinggi.rr_linier_model_utama) else 'tidak cocok dengan bentuk respons'}. RR per tipe lokasi RS juga tersebar ({fl(het.rr.min(), 2)}–{fl(het.rr.max(), 2)} per +10 µg/m³, terendah pada {het.loc[het.rr.idxmin(), 'tipe']}, tertinggi pada {het.loc[het.rr.idxmax(), 'tipe']}), sehingga satu angka RR nasional tidak boleh dipakai untuk memprediksi kasus pada PM2.5 ekstrem.",
       f"Selisih balita vs lansia pada efek kumulatif: rasio RR {fl(sd_cum['rasio_rr'], 4)} (CI {fl(sd_cum['rasio_bawah'], 4)}–{fl(sd_cum['rasio_atas'], 4)}, {pv(sd_cum['p'])}) dan pada lag 0 {fl(sd_lag0['rasio_rr'], 4)} ({pv(sd_lag0['p'])}) → tidak ada bukti perbedaan kerentanan; kedua kelompok identik secara praktis pada data ini. Profil lag per-lag DLM tidak datar (RR per lag balita {fl(DL[bl_a]['tabel_lag'].rr.min(), 3)}–{fl(DL[bl_a]['tabel_lag'].rr.max(), 3)}, tertinggi pada lag {int(DL[bl_a]['tabel_lag'].rr.idxmax())}) tetapi lag-lag sangat berkolinear sehingga bentuk jeda tidak dapat dipastikan; RR lag tunggal menurun landai dari {fl(tabel_rr.iloc[0, 0], 3)} (lag 0) ke {fl(tabel_rr.iloc[0, -1], 3)} (lag 7).",
       f"Selama episode asap (tanpa BJB-01; {int(mask_ep.sum())} RS-hari pada 5 RS) kasus ISPA balita berlebih diperkirakan {fl(tb.berlebih_tanpa_potong, 0)} (tanpa pemotongan), {fl(tb.berlebih_empiris, 0)} (selisih empiris episode vs non-episode Jul–Okt) hingga {fl(tb.berlebih_model, 0)} (model dipotong; jumlah batas CI per RS {fl(tb.bawah, 0)}–{fl(tb.atas, 0)}) atau {fl(100 * tb.berlebih_empiris / tb.teramati, 0)}–{fl(tb.persen_dari_kasus, 0)}% dari kasus teramati; ISPA lansia {fl(tl.berlebih_tanpa_potong, 0)}–{fl(tl.berlebih_model, 0)} (empiris {fl(tl.berlebih_empiris, 0)}). Rentang inilah yang dilaporkan, bukan satu angka.")

**Interpretasi.** RR kumulatif lag 0–7 per +10 µg/m³ (eksposur dipotong 250,4): ISPA balita 1,086 (CI 1,051–1,121) dan ISPA lansia 1,086 (1,053–1,119), yaitu +8,6% dan +8,6% kasus per +10 µg/m³; 7 dari 7 jenis kasus signifikan. Hasil tahan terhadap spline waktu (df 10–30: 1,088–1,087), negative binomial (lag 0: 1,091 vs 1,088), serta pembatasan ke stasiun Layak (1,086). Namun RR bergantung kuat pada rentang eksposur: tanpa pemotongan menjadi 1,021 (balita), dan RR per kelas PM2.5 menunjukkan respons yang jenuh — pada kelas >250,4 RR teramati 4,54 (CI 3,46–5,95) padahal ekstrapolasi linear memperkirakan 7,15; sebaliknya pada kelas 55,4–150,4 teramati 2,73 vs linear 1,73 — model satu-kemiringan meremehkan kelas menengah dan melebih-lebihkan kelas ekstrem. RR per tipe lokasi RS juga tersebar (1,02–1,25 per +10 µg/m³, terendah pada Jalan Raya, tertinggi pada Latar Bersih), sehingga satu angka RR nasional tidak boleh dipakai untuk memprediksi kasus pada PM2.5 ekstrem. Selisih balita vs lansia pada efek kumulatif: rasio RR 1,0000 (CI 0,9997–1,0003, p = 0,999) dan pada lag 0 1,0006 (p = 0,659) → tidak ada bukti perbedaan kerentanan; kedua kelompok identik secara praktis pada data ini. Profil lag per-lag DLM tidak datar (RR per lag balita 0,988–1,063, tertinggi pada lag 0) tetapi lag-lag sangat berkolinear sehingga bentuk jeda tidak dapat dipastikan; RR lag tunggal menurun landai dari 1,088 (lag 0) ke 1,074 (lag 7). Selama episode asap (tanpa BJB-01; 148 RS-hari pada 5 RS) kasus ISPA balita berlebih diperkirakan 2.402 (tanpa pemotongan), 2.864 (selisih empiris episode vs non-episode Jul–Okt) hingga 3.403 (model dipotong; jumlah batas CI per RS 2.640–3.857) atau 58–69% dari kasus teramati; ISPA lansia 2.776–3.821 (empiris 3.164). Rentang inilah yang dilaporkan, bukan satu angka.

## 3. Klaster stasiun
Fitur: profil **bulanan** (12) dan profil **per jam lokal** (24) PM2.5 yang dinormalkan dengan rata-ratanya sendiri (bentuk, bukan tingkat), data bertopeng; tiap blok dibagi √(jumlah fitur) agar bobot blok seimbang. Pemilihan k: silhouette untuk Ward dan k-means; stabilitas: ARI antar-metode. Klaster diberi nama dari karakteristik terukur.

In [21]:
stas_k = sorted(hourly_m.kode_stasiun.unique())
Fp = M.profil_stasiun(hourly_m, stas_k)
Fb = Fp.copy(); Fb[[c for c in Fp if c.startswith("bln")]] /= np.sqrt(12); Fb[[c for c in Fp if c.startswith("jam")]] /= np.sqrt(24)
Lw = sch.linkage(Fb, "ward")
sil = []
for k in range(2, 7):
    lab = sch.fcluster(Lw, k, "maxclust"); km = KMeans(k, n_init=30, random_state=1).fit(Fb)
    sil.append(dict(k=k, silhouette_ward=silhouette_score(Fb, lab), silhouette_kmeans=silhouette_score(Fb, km.labels_), ARI_ward_vs_kmeans=adjusted_rand_score(lab, km.labels_), ukuran_klaster_ward=sorted(pd.Series(lab).value_counts().tolist(), reverse=True)))
sil = pd.DataFrame(sil); display(sil.round(3))

,k,silhouette_ward,silhouette_kmeans,ARI_ward_vs_kmeans,ukuran_klaster_ward
0,2,0.470,0.491,0.576,"[14, 12]"
1,3,0.535,0.535,1.000,"[14, 10, 2]"
2,4,0.416,0.416,1.000,"[14, 5, 5, 2]"
3,5,0.361,0.361,1.000,"[8, 6, 5, 5, 2]"
4,6,0.385,0.385,1.000,"[7, 6, 5, 5, 2, 1]"


In [22]:
# k: silhouette Ward tertinggi dengan ARI ≥ 0,9 dan tidak ada klaster beranggota 1; kandidat dipilih secara otomatis
kand = sil[(sil.ARI_ward_vs_kmeans >= 0.9) & (sil.ukuran_klaster_ward.map(min) >= 2)]
k_pilih = int(kand.sort_values("silhouette_ward", ascending=False).iloc[0].k) if len(kand) else int(sil.sort_values("silhouette_ward", ascending=False).iloc[0].k)
lab = sch.fcluster(Lw, k_pilih, "maxclust")
km = KMeans(k_pilih, n_init=30, random_state=1).fit(Fb)
print(f"k terpilih = {k_pilih} (silhouette Ward {sil.set_index('k').loc[k_pilih, 'silhouette_ward']:.3f}; ARI Ward vs k-means {adjusted_rand_score(lab, km.labels_):.2f})")
kl = pd.DataFrame({"klaster": lab}, index=Fp.index)
bln = Fp[[c for c in Fp if c.startswith("bln")]]; jam = Fp[[c for c in Fp if c.startswith("jam")]]
kl["rasio_musiman"] = bln.max(axis=1) / bln.min(axis=1); kl["bulan_puncak"] = bln.idxmax(axis=1).str.replace("bln_", "").astype(int)
kl["amplitudo_diurnal_persen"] = 100 * (jam.max(axis=1) - jam.min(axis=1)); kl["jam_puncak"] = jam.idxmax(axis=1).str.replace("jam_", "").astype(int)
lv = hourly_m.groupby("kode_stasiun").pm25_ug_m3.mean(); kl["pm25_rata2"] = lv.reindex(kl.index)
kl["tipe"] = st.kelompok_lokasi.reindex(kl.index); kl["status"] = status.reindex(kl.index)
ring_k = kl.groupby("klaster").agg(n=("tipe", "size"), rasio_musiman=("rasio_musiman", "median"), bulan_puncak=("bulan_puncak", lambda s: int(s.mode().iloc[0])), amplitudo_diurnal_persen=("amplitudo_diurnal_persen", "median"),
                                    pm25_rata2=("pm25_rata2", "median"), tipe_dominan=("tipe", lambda s: s.value_counts().index[0]), anggota=("tipe", lambda s: ", ".join(s.index)))
# penamaan berbasis aturan dari karakteristik terukur
def beri_nama(r):
    musim = "kemarau tegas" if r.rasio_musiman >= 2.0 else "musiman lemah" if r.rasio_musiman >= 1.3 else "relatif datar sepanjang tahun"
    diu = "dua puncak harian kuat" if r.amplitudo_diurnal_persen >= 40 else "variasi harian sedang" if r.amplitudo_diurnal_persen >= 15 else "variasi harian kecil"
    tingkat = "tingkat tinggi" if r.pm25_rata2 >= 45 else "tingkat sedang" if r.pm25_rata2 >= 25 else "tingkat rendah"
    return f"{musim.capitalize()} · {diu} · {tingkat}"
ring_k["nama"] = ring_k.apply(beri_nama, axis=1)
ring_k["pm25_min"] = kl.groupby("klaster").pm25_rata2.min(); ring_k["pm25_maks"] = kl.groupby("klaster").pm25_rata2.max()
ring_k["rasio_musiman_maks"] = kl.groupby("klaster").rasio_musiman.max()
kl["nama_klaster"] = kl.klaster.map(ring_k.nama); display(ring_k.drop(columns="anggota").round(2)); display(ring_k[["nama", "anggota"]])
KN["3_klaster"] = {"k": k_pilih, "silhouette": sil.round(4).drop(columns="ukuran_klaster_ward").to_dict("records"), "klaster": {int(k): {"nama": r.nama, "n": int(r.n), "anggota": r.anggota.split(", "), "rasio_musiman_median": float(r.rasio_musiman), "pm25_rata2_median": float(r.pm25_rata2), "pm25_rentang": [float(r.pm25_min), float(r.pm25_maks)]} for k, r in ring_k.iterrows()}}

k terpilih = 3 (silhouette Ward 0.535; ARI Ward vs k-means 1.00)


,n,rasio_musiman,bulan_puncak,amplitudo_diurnal_persen,pm25_rata2,tipe_dominan,nama,pm25_min,pm25_maks,rasio_musiman_maks
klaster,,,,,,,,,,
1,14,1.42,7,68.47,28.13,Perkotaan,Musiman lemah · dua puncak harian kuat · tingkat sedang,9.34,60.07,2.08
2,2,10.73,10,3.58,61.03,Perkotaan,Kemarau tegas · variasi harian kecil · tingkat tinggi,60.84,61.23,10.84
3,10,4.14,9,66.68,47.34,Gambut,Kemarau tegas · dua puncak harian kuat · tingkat tinggi,18.65,78.84,6.87


,nama,anggota
klaster,,
1,Musiman lemah · dua puncak harian kuat · tingkat sedang,"BGL-01, BGR-01, BGR-02, IDP-01, JKT-01, JKT-02, JKT-03, MDN-01, MER-01, MLG-01, MRS-01, PLU-01, SRG-01, YK..."
2,Kemarau tegas · variasi harian kecil · tingkat tinggi,"KTG-01, KTG-02"
3,Kemarau tegas · dua puncak harian kuat · tingkat tinggi,"BJB-01, BTM-01, JMB-01, KTB-01, PKN-01, PKU-01, PLB-01, PSW-01, SM2-01, STG-01"


In [23]:
warna_k = ["#0B6E6E", "#E17055", "#6C5CE7", "#F2B01E", "#8D6E63", "#3B82C4"]
fig = make_subplots(rows=2, cols=2, specs=[[{"colspan": 2}, None], [{}, {}]], row_heights=[0.45, 0.55], vertical_spacing=0.14,
                    subplot_titles=("Dendrogram (Ward) profil bulanan + per jam", "Profil bulanan (indeks = bulan ÷ rata-rata)", "Profil per jam lokal (indeks)"))
dd = sch.dendrogram(Lw, labels=list(Fb.index), no_plot=True)
for xs, ys in zip(dd["icoord"], dd["dcoord"]):
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color=style.WARNA_TEKS_REDUP, width=1.5), hoverinfo="skip", showlegend=False), row=1, col=1)
pos = [5 + 10 * i for i in range(len(dd["ivl"]))]
fig.update_xaxes(tickmode="array", tickvals=pos, ticktext=[f"{k}" for k in dd["ivl"]], tickangle=-60, row=1, col=1)
for i, (k, r) in enumerate(ring_k.iterrows()):
    ks = kl.index[kl.klaster == k]; wrn = warna_k[i % len(warna_k)]
    for s_ in ks:
        fig.add_trace(go.Scatter(x=list(range(1, 13)), y=bln.loc[s_], mode="lines", line=dict(color=wrn, width=1), opacity=0.35, showlegend=False, hoverinfo="skip"), row=2, col=1)
        fig.add_trace(go.Scatter(x=list(range(24)), y=jam.loc[s_], mode="lines", line=dict(color=wrn, width=1), opacity=0.35, showlegend=False, hoverinfo="skip"), row=2, col=2)
    fig.add_trace(go.Scatter(x=list(range(1, 13)), y=bln.loc[ks].median(), mode="lines+markers", line=dict(color=wrn, width=3), name=f"K{k}: {r.nama} (n={int(r.n)})", legendgroup=str(k)), row=2, col=1)
    fig.add_trace(go.Scatter(x=list(range(24)), y=jam.loc[ks].median(), mode="lines", line=dict(color=wrn, width=3), showlegend=False, legendgroup=str(k)), row=2, col=2)
fig.update_xaxes(title_text="Bulan", dtick=1, row=2, col=1); fig.update_xaxes(title_text="Jam lokal", dtick=3, row=2, col=2); fig.update_yaxes(title_text="Jarak (Ward)", row=1, col=1)
fig.update_layout(legend=dict(orientation="h", y=-0.22), margin=dict(b=150))
style.terapkan(fig, "Klaster stasiun menurut bentuk profil musiman dan harian PM2.5", 840)
simpan(fig, "05_3_klaster_dendrogram_profil", 820); fig.show()

In [24]:
pca = PCA(2).fit(Fb); Zp = pca.transform(Fb)
fig = go.Figure()
for i, (k, r) in enumerate(ring_k.iterrows()):
    ks = list(kl.index[kl.klaster == k]); ix = [list(Fb.index).index(s_) for s_ in ks]
    fig.add_trace(go.Scatter(x=Zp[ix, 0], y=Zp[ix, 1], mode="markers+text", text=ks, textposition="top center", textfont=dict(size=9), marker=dict(size=12, color=warna_k[i % 6], symbol=["circle" if status[s_] == "Layak" else "diamond-open" for s_ in ks], line=dict(width=2, color=warna_k[i % 6])),
                             name=f"K{k}: {r.nama}"))
fig.update_xaxes(title_text=f"PC1 ({100 * pca.explained_variance_ratio_[0]:.0f}% varians)"); fig.update_yaxes(title_text=f"PC2 ({100 * pca.explained_variance_ratio_[1]:.0f}% varians)")
style.terapkan(fig, "Proyeksi PCA stasiun menurut profil (lingkaran = Layak, berlian = status Terbatas)", 560)
simpan(fig, "05_3_klaster_pca", 560); fig.show()
tab_k = pd.crosstab(kl.nama_klaster, kl.tipe).reindex(columns=TIPE, fill_value=0); display(tab_k)
kwk = eda.kruskal({k: kl.loc[kl.klaster == k, "rasio_musiman"].to_numpy() for k in sorted(kl.klaster.unique())})
print(f"Uji beda rasio musiman antar-klaster: KW H={kwk['H']:.1f}, {pv(kwk['p'])}, ε²={kwk['epsilon2']:.2f}")
KN["3_klaster"]["kruskal_rasio_musiman"] = kwk
tafsir(f"Terbentuk {k_pilih} klaster (silhouette Ward {fl(sil.set_index('k').loc[k_pilih, 'silhouette_ward'], 2)}, ARI Ward vs k-means {fl(adjusted_rand_score(lab, km.labels_), 2)}): " + "; ".join(f"K{k} '{r.nama}' ({int(r.n)} stasiun)" for k, r in ring_k.iterrows()) + ".",
       f"Rasio musiman berbeda antar-klaster (KW {pv(kwk['p'])}, ε² = {fl(kwk['epsilon2'], 2)}); klaster mana yang tegas musiman: " + ", ".join(f"K{k} (median rasio bulan puncak/terendah {fl(r.rasio_musiman, 1)}×)" for k, r in ring_k.sort_values('rasio_musiman', ascending=False).iterrows()) + ".",
       "Silhouette berada pada kisaran sedang dan jumlah stasiun kecil (26), sehingga batas klaster bersifat indikatif; stasiun status Terbatas (MDN-01, MER-01, PLB-01, SM2-01) memiliki profil yang lebih tidak pasti.")

tipe,Perkotaan,Jalan Raya,Industri,Gambut,Latar Bersih
nama_klaster,,,,,
Kemarau tegas · dua puncak harian kuat · tingkat tinggi,1,0,2,6,1
Kemarau tegas · variasi harian kecil · tingkat tinggi,1,0,0,1,0
Musiman lemah · dua puncak harian kuat · tingkat sedang,5,2,2,0,5


Uji beda rasio musiman antar-klaster: KW H=19.7, p < 0,001, ε²=0.79


**Interpretasi.** Terbentuk 3 klaster (silhouette Ward 0,53, ARI Ward vs k-means 1,00): K1 'Musiman lemah · dua puncak harian kuat · tingkat sedang' (14 stasiun); K2 'Kemarau tegas · variasi harian kecil · tingkat tinggi' (2 stasiun); K3 'Kemarau tegas · dua puncak harian kuat · tingkat tinggi' (10 stasiun). Rasio musiman berbeda antar-klaster (KW p < 0,001, ε² = 0,79); klaster mana yang tegas musiman: K2 (median rasio bulan puncak/terendah 10,7×), K3 (median rasio bulan puncak/terendah 4,1×), K1 (median rasio bulan puncak/terendah 1,4×). Silhouette berada pada kisaran sedang dan jumlah stasiun kecil (26), sehingga batas klaster bersifat indikatif; stasiun status Terbatas (MDN-01, MER-01, PLB-01, SM2-01) memiliki profil yang lebih tidak pasti.

## 4. Anomali
### 4.1 BJB-01: uji hipotesis berbasis data
Pada notebook 04 BJB-01 tampak berpola musiman "terbalik" (r = 0,26 vs stasiun gambut lain) dan memiliki episode 204 hari. Empat hipotesis diuji **dengan data**, bukan ditebak:
| Hipotesis | Prediksi bila benar | Uji |
|---|---|---|
| H1 Timestamp bergeser | pola PM2.5 sama dengan stasiun lain tetapi digeser; geseran waktu besar | besar geseran terukur (notebook 02); korelasi silang harian & geser bulan |
| H2 Satuan/skala keliru | rasio PM2.5/PM10 atau tingkat dasar menyimpang; level berubah serentak | rasio PM2.5/PM10, tingkat dasar vs `baseline_pm25`, uji perubahan Pettitt per polutan |
| H3 Pembalikan musim nyata | pola bulanan tidak berulang konsisten antar tahun, variasi harian tetap ada | rata-rata bulanan per tahun; variasi (nilai unik, sd) |
| H4 Sensor macet (flatline) | nilai identik berhari-hari; sd = 0; pola pulih setelahnya | deret identik, sd, korelasi sebelum/sesudah |

In [25]:
P = daily.pivot(index="tanggal", columns="kode_stasiun", values="pm25_ug_m3")
gam = [k for k in ["PKU-01", "JMB-01", "STG-01", "PKN-01", "KTG-02"] if status[k] == "Layak"]; peer = P[gam].mean(axis=1); b = P["BJB-01"]
prof02 = pd.read_parquet(PR / "profiling" / "profil_stasiun.parquet").set_index("kode")
# H1
geser_menit = float(prof02.loc["BJB-01", "geser_median_menit"])
mb, mp = daily[daily.kode_stasiun == "BJB-01"].groupby("bulan").pm25_ug_m3.mean(), daily[daily.kode_stasiun.isin(gam)].groupby("bulan").pm25_ug_m3.mean()
gs = M.geser_melingkar(mb.values, mp.values).sort_values("r", ascending=False)
cc = M.ccf(np.log(b), np.log(peer), range(-240, 241, 10)); cc_best = cc.loc[cc.r.idxmax()]
print(f"H1  geseran timestamp BJB-01 terukur: {geser_menit:+.2f} menit (notebook 02) | korelasi harian dengan stasiun gambut lain: lag 0 r={cc[cc.lag == 0].r.iloc[0]:.2f}; terbaik lag {int(cc_best.lag)} hari r={cc_best.r:.2f}; geser bulan terbaik {int(gs.iloc[0].geser_bulan)} (r={gs.iloc[0].r:.2f})")
# H2
bj = daily[daily.kode_stasiun == "BJB-01"]
rasio = (bj.pm25_ug_m3 / bj.pm10_ug_m3); rasio_lain = (daily[daily.kode_stasiun.isin(gam)].pm25_ug_m3 / daily[daily.kode_stasiun.isin(gam)].pm10_ug_m3)
cp_tab = []
for v in ["pm25_ug_m3", "pm10_ug_m3", "co_ug_m3", "no2_ug_m3", "so2_ug_m3", "o3_ug_m3"]:
    x = bj.set_index("tanggal")[v].dropna(); pt = M.pettitt(np.log(x.values)); i = pt["indeks"]
    cp_tab.append(dict(polutan=v.replace("_ug_m3", ""), tanggal_perubahan=x.index[i].date(), p=pt["p"], median_sebelum=x.iloc[:i + 1].median(), median_sesudah=x.iloc[i + 1:].median(), rasio_sebelum_sesudah=x.iloc[:i + 1].median() / x.iloc[i + 1:].median()))
cp_tab = pd.DataFrame(cp_tab)
print(f"H2  rasio PM2.5/PM10 BJB-01: median {rasio.median():.2f} (sd antar-bulan {bj.groupby('bulan').apply(lambda x: (x.pm25_ug_m3 / x.pm10_ug_m3).median()).std():.3f}) vs stasiun gambut lain {rasio_lain.median():.2f}; baseline_pm25 BJB-01 = {st.loc['BJB-01', 'baseline_pm25']:.1f} µg/m³")
display(cp_tab.round(3))

H1  geseran timestamp BJB-01 terukur: +1.26 menit (notebook 02) | korelasi harian dengan stasiun gambut lain: lag 0 r=0.23; terbaik lag 70 hari r=0.38; geser bulan terbaik 2 (r=0.75)
H2  rasio PM2.5/PM10 BJB-01: median 0.69 (sd antar-bulan 0.001) vs stasiun gambut lain 0.67; baseline_pm25 BJB-01 = 35.0 µg/m³


,polutan,tanggal_perubahan,p,median_sebelum,median_sesudah,rasio_sebelum_sesudah
0,pm25,2025-03-29,0.0,81.035,36.713,2.207
1,pm10,2025-03-29,0.0,117.290,53.188,2.205
2,co,2025-03-29,0.0,582.062,559.532,1.040
3,no2,2025-03-29,0.0,33.053,30.858,1.071
4,so2,2025-03-29,0.0,8.391,7.785,1.078
5,o3,2025-07-09,0.0,15.706,27.836,0.564


In [26]:
# H3 & H4
fl_b = flat[flat.kode_stasiun == "BJB-01"].reset_index(drop=True)
bm_ = daily[daily.kode_stasiun == "BJB-01"].set_index("tanggal").pm25_ug_m3
tab = []
for nama, (a, z) in {"normal awal (2024-01…07)": ("2024-01-01", "2024-07-31"), "normal akhir (2025-11…2026-07)": ("2025-11-01", "2026-07-31")}.items():
    x = hourly[(hourly.kode_stasiun == "BJB-01") & (hourly.jam >= a) & (hourly.jam <= z + " 23:00")].pm25_ug_m3
    tab.append(dict(segmen=nama, n_jam=len(x), nilai_unik=x.nunique(), median=x.median(), sd=x.std()))
for i, r in fl_b.iterrows():
    tab.append(dict(segmen=f"flatline {i + 1}: {r.awal:%Y-%m-%d}…{r.akhir:%Y-%m-%d}", n_jam=r.n_jam, nilai_unik=1, median=r.nilai, sd=0.0))
tab = pd.DataFrame(tab); display(tab.round(2))
pt25 = cp_tab.set_index("polutan").loc["pm25"]; cp_date = pd.Timestamp(pt25.tanggal_perubahan)
sebelum = pd.concat([b.loc[:cp_date], peer.loc[:cp_date]], axis=1).dropna(); sesudah = pd.concat([b.loc[cp_date + pd.Timedelta(days=1):], peer.loc[cp_date + pd.Timedelta(days=1):]], axis=1).dropna()
r_sb, r_ss = sebelum.corr().iloc[0, 1], sesudah.corr().iloc[0, 1]
bm_m = daily_m[daily_m.kode_stasiun == "BJB-01"].set_index("tanggal").pm25_ug_m3; pm_m = pd.concat([bm_m, peer], axis=1).dropna().corr().iloc[0, 1]
print(f"H4  korelasi harian BJB-01 vs rata-rata stasiun gambut lain: sebelum {cp_date:%Y-%m-%d} r={r_sb:.2f} (n={len(sebelum)}) | sesudah r={r_ss:.2f} (n={len(sesudah)}) | setelah flatline dibuang seluruh periode r={pm_m:.2f}")
kg = flat[flat.kode_stasiun == "BJB-01"]; rasio_tinggi = float(kg.nilai.max() / tab.iloc[1]["median"]); rasio_rendah = float(tab.iloc[1]["median"] / kg.nilai.min())
print(f"Dataran tinggi = {rasio_tinggi:.1f}× tingkat normal; dataran rendah = tingkat normal ÷ {rasio_rendah:.1f}")

,segmen,n_jam,nilai_unik,median,sd
0,normal awal (2024-01…07),5112,437,37.0,9.36
1,normal akhir (2025-11…2026-07),6552,456,37.1,9.35
2,flatline 1: 2024-09-07…2025-03-29,4886,1,336.7,0.00
3,flatline 2: 2025-03-29…2025-07-09,2443,1,3.5,0.00


H4  korelasi harian BJB-01 vs rata-rata stasiun gambut lain: sebelum 2025-03-29 r=0.17 (n=454) | sesudah r=0.73 (n=564) | setelah flatline dibuang seluruh periode r=0.77
Dataran tinggi = 9.1× tingkat normal; dataran rendah = tingkat normal ÷ 10.6


In [27]:
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.14, row_heights=[0.58, 0.42], subplot_titles=("PM2.5 harian BJB-01 (data pembersihan 03) vs rata-rata stasiun gambut lain", "Rata-rata bulanan BJB-01 per tahun (µg/m³)"))
fig.add_trace(go.Scatter(x=b.index, y=b, mode="lines", line=dict(color=WARNA["Gambut"], width=1.2), name="BJB-01"), row=1, col=1)
fig.add_trace(go.Scatter(x=peer.index, y=peer, mode="lines", line=dict(color=style.WARNA_UTAMA, width=1.2, dash="dot"), name="rata-rata PKU, JMB-01, STG, PKN, KTG-02 (Layak)"), row=1, col=1)
for r in fl_b.itertuples():
    fig.add_vrect(x0=r.awal, x1=r.akhir, fillcolor=style.KEPARAHAN_WARNA["Kritis"], opacity=0.12, line_width=0, row=1, col=1, annotation_text=f"flatline {fl(r.nilai, 1)}", annotation_position="top left")
mm = bm_.resample("MS").mean()
for thn, col_ in zip((2024, 2025, 2026), ("#6C5CE7", "#E17055", style.WARNA_UTAMA)):
    x = mm[mm.index.year == thn]; fig.add_trace(go.Scatter(x=x.index.month, y=x.values, mode="lines+markers", name=str(thn), line=dict(color=col_, width=2.5)), row=2, col=1)
fig.update_xaxes(title_text="Bulan", dtick=1, row=2, col=1); fig.update_yaxes(title_text="µg/m³", row=1, col=1); fig.update_yaxes(title_text="µg/m³", type="log", row=2, col=1)
style.terapkan(fig, "BJB-01: dua dataran konstan (sensor macet) — bukan pembalikan musim", 760)
simpan(fig, "05_4_bjb01_flatline", 760); fig.show()
KN["4_anomali"] = {"bjb01": {"geser_timestamp_menit": geser_menit, "ccf_lag0": float(cc[cc.lag == 0].r.iloc[0]), "ccf_terbaik": {"lag_hari": int(cc_best.lag), "r": float(cc_best.r)}, "geser_bulan_terbaik": {"bulan": int(gs.iloc[0].geser_bulan), "r": float(gs.iloc[0].r)},
                            "rasio_pm25_pm10_median": float(rasio.median()), "rasio_pm25_pm10_gambut_lain": float(rasio_lain.median()), "baseline_pm25": float(st.loc["BJB-01", "baseline_pm25"]),
                            "pettitt": cp_tab.assign(tanggal_perubahan=cp_tab.tanggal_perubahan.astype(str)).round(4).to_dict("records"), "segmen": tab.round(3).to_dict("records"),
                            "korelasi_peer_sebelum": float(r_sb), "korelasi_peer_sesudah": float(r_ss), "korelasi_peer_tanpa_flatline": float(pm_m), "rasio_dataran_tinggi_vs_normal": rasio_tinggi, "rasio_normal_vs_dataran_rendah": rasio_rendah}}

In [28]:
H1_ok = abs(geser_menit) > 60; H2_ok = (abs(rasio.median() - rasio_lain.median()) > 0.1) and pt25.rasio_sebelum_sesudah > 1 and False
r0m = float(gs[gs.geser_bulan == 0].r.iloc[0])
tafsir(f"H1 (timestamp bergeser) tidak didukung sebagai penyebab: geseran timestamp BJB-01 hanya {fl(abs(geser_menit), 2)} menit. Memang profil bulanannya paling mirip stasiun gambut lain bila digeser {int(gs.iloc[0].geser_bulan)} bulan (r = {fl(gs.iloc[0].r, 2)} vs {fl(r0m, 2)} tanpa geseran), tetapi pada tingkat harian korelasi hanya naik dari {fl(cc[cc.lag == 0].r.iloc[0], 2)} (lag 0) ke {fl(cc_best.r, 2)} (lag {int(cc_best.lag)} hari); 'kemiripan setelah digeser' ini dihasilkan oleh dataran konstan yang melebar (H4), bukan pergeseran waktu data.",
       f"H2 (satuan/skala keliru) tidak didukung sebagai penjelasan tunggal: rasio PM2.5/PM10 BJB-01 konstan {fl(rasio.median(), 2)} (stasiun gambut lain {fl(rasio_lain.median(), 2)}) dan tingkat normalnya ({fl(tab.iloc[1]['median'], 1)} µg/m³) sesuai `baseline_pm25` ({fl(st.loc['BJB-01', 'baseline_pm25'], 1)}). Uji Pettitt mendeteksi perubahan tingkat pada {cp_date:%d-%m-%Y} untuk PM2.5 dan PM10 (rasio {fl(pt25.rasio_sebelum_sesudah, 2)}×), sedangkan CO, NO₂, dan SO₂ pada tanggal yang sama hanya berubah {fl(cp_tab.set_index('polutan').loc[['co', 'no2', 'so2'], 'rasio_sebelum_sesudah'].min(), 2)}–{fl(cp_tab.set_index('polutan').loc[['co', 'no2', 'so2'], 'rasio_sebelum_sesudah'].max(), 2)}× (O₃ berubah pada tanggal lain) — perubahan besar terkonsentrasi pada kanal partikulat.",
       f"H3 (pembalikan musim nyata) ditolak dan H4 (sensor macet) didukung: ada {len(fl_b)} deret identik ({'; '.join(f'{rp(r.n_jam)} jam = {fl(r.n_hari, 0)} hari bernilai {fl(r.nilai, 1)} µg/m³' for r in fl_b.itertuples())}) dengan sd = 0, padahal stasiun normal bersd ±{fl(tab.iloc[0]['sd'], 1)} dan stasiun lain tidak pernah > {int(terpanjang.drop(flat.kode_stasiun.unique(), errors='ignore').max())} jam identik. Di luar dataran, tingkat normal awal dan akhir sama ({fl(tab.iloc[0]['median'], 1)} vs {fl(tab.iloc[1]['median'], 1)} µg/m³) dan setelah kedua dataran dibuang korelasi seluruh periode dengan stasiun gambut lain {fl(pm_m, 2)} (vs {fl(r_sb, 2)} sebelum {cp_date:%d-%m-%Y} pada data apa adanya). Dataran tinggi = {fl(rasio_tinggi, 1)}× dan dataran rendah = 1/{fl(rasio_rendah, 1)}× tingkat normal. Sebab pasti (kegagalan alat vs nilai tertahan di logger) tidak dapat ditentukan dari data.")

**Interpretasi.** H1 (timestamp bergeser) tidak didukung sebagai penyebab: geseran timestamp BJB-01 hanya 1,26 menit. Memang profil bulanannya paling mirip stasiun gambut lain bila digeser 2 bulan (r = 0,75 vs 0,29 tanpa geseran), tetapi pada tingkat harian korelasi hanya naik dari 0,23 (lag 0) ke 0,38 (lag 70 hari); 'kemiripan setelah digeser' ini dihasilkan oleh dataran konstan yang melebar (H4), bukan pergeseran waktu data. H2 (satuan/skala keliru) tidak didukung sebagai penjelasan tunggal: rasio PM2.5/PM10 BJB-01 konstan 0,69 (stasiun gambut lain 0,67) dan tingkat normalnya (37,1 µg/m³) sesuai `baseline_pm25` (35,0). Uji Pettitt mendeteksi perubahan tingkat pada 29-03-2025 untuk PM2.5 dan PM10 (rasio 2,21×), sedangkan CO, NO₂, dan SO₂ pada tanggal yang sama hanya berubah 1,04–1,08× (O₃ berubah pada tanggal lain) — perubahan besar terkonsentrasi pada kanal partikulat. H3 (pembalikan musim nyata) ditolak dan H4 (sensor macet) didukung: ada 2 deret identik (4.886 jam = 204 hari bernilai 336,7 µg/m³; 2.443 jam = 102 hari bernilai 3,5 µg/m³) dengan sd = 0, padahal stasiun normal bersd ±9,4 dan stasiun lain tidak pernah > 4 jam identik. Di luar dataran, tingkat normal awal dan akhir sama (37,0 vs 37,1 µg/m³) dan setelah kedua dataran dibuang korelasi seluruh periode dengan stasiun gambut lain 0,77 (vs 0,17 sebelum 29-03-2025 pada data apa adanya). Dataran tinggi = 9,1× dan dataran rendah = 1/10,6× tingkat normal. Sebab pasti (kegagalan alat vs nilai tertahan di logger) tidak dapat ditentukan dari data.

### 4.2 "Polusi tinggi, kasus rendah": BJB-01 dan SM2-01
Dua hipotesis: **(Ha)** ketidakselarasan hanya perbedaan tingkat dasar kasus antar-RS (ekologis); **(Hb)** respons dalam-RS terhadap PM2.5 berbeda (kesalahan pengukuran paparan). Uji: residual studentized dari regresi log-log antar-RS (n = jumlah RS) dengan PM2.5 *apa adanya* dan *bertopeng*; dan RR per RS vs RS lain.

In [29]:
Hh = health_m.copy(); Hh["bl"] = Hh.resp_admissions_under5.astype(float) + Hh.resp_admissions_over60.astype(float)
hs = Hh.groupby("hospital_id").agg(kasus=("bl", "mean"), kapasitas=("kapasitas_tempat_tidur", "first"), kode=("kode_stasiun", "first"))
hs["pm_asli"] = hs.kode.map(hourly.groupby("kode_stasiun").pm25_ug_m3.mean()); hs["pm_topeng"] = hs.kode.map(hourly_m.groupby("kode_stasiun").pm25_ug_m3.mean())      # rata-rata per jam (MER-01 hanya punya 2 hari harian sah)
hs = hs.dropna(subset=["pm_asli", "pm_topeng"])
def resid(pm):
    r = stats.linregress(np.log(hs[pm]), np.log(hs.kasus)); e = np.log(hs.kasus) - (r.intercept + r.slope * np.log(hs[pm]))
    # residual terstudentisasi leave-one-out (mengurangi pengaruh titik itu sendiri)
    z = []
    for i in range(len(hs)):
        o = np.delete(np.arange(len(hs)), i); rr_ = stats.linregress(np.log(hs[pm].to_numpy()[o]), np.log(hs.kasus.to_numpy()[o])); s_ = np.std(np.log(hs.kasus.to_numpy()[o]) - (rr_.intercept + rr_.slope * np.log(hs[pm].to_numpy()[o])), ddof=2)
        z.append((np.log(hs.kasus.to_numpy()[i]) - (rr_.intercept + rr_.slope * np.log(hs[pm].to_numpy()[i]))) / s_)
    return r, pd.Series(z, index=hs.index)
r_a, z_a = resid("pm_asli"); r_t, z_t = resid("pm_topeng")
hs["z_asli"], hs["z_topeng"] = z_a, z_t
print(f"Antar-RS (n={len(hs)}): Spearman kasus~PM2.5 apa adanya ρ={stats.spearmanr(hs.kasus, hs.pm_asli)[0]:.2f}; bertopeng ρ={stats.spearmanr(hs.kasus, hs.pm_topeng)[0]:.2f}; kasus~kapasitas ρ={stats.spearmanr(hs.kasus, hs.kapasitas)[0]:.2f}")
display(hs.sort_values("z_asli")[["kode", "kasus", "kapasitas", "pm_asli", "pm_topeng", "z_asli", "z_topeng"]].head(6).round(2))
rr_cmp = rr_rs.copy(); lain_ = rr_cmp.drop(index=[i for i in rr_cmp.index if rr_cmp.loc[i, "kode_stasiun"] in ("BJB-01", "SM2-01")])
zrr = {k: float((rr_cmp.loc[rr_cmp.kode_stasiun == k, "rr"].iloc[0] - lain_.rr.mean()) / lain_.rr.std()) for k in ("BJB-01", "SM2-01") if (rr_cmp.kode_stasiun == k).any()}
print("RR per RS (balita+lansia, +10 µg/m³ rata-rata 7 hari): rata-rata RS lain", round(lain_.rr.mean(), 3), "sd", round(lain_.rr.std(), 3)); display(rr_cmp[rr_cmp.kode_stasiun.isin(["BJB-01", "SM2-01"])].round(3))
print("z-skor RR terhadap RS lain:", {k: round(v, 2) for k, v in zrr.items()})

Antar-RS (n=26): Spearman kasus~PM2.5 apa adanya ρ=0.65; bertopeng ρ=0.73; kasus~kapasitas ρ=0.18


,kode,kasus,kapasitas,pm_asli,pm_topeng,z_asli,z_topeng
hospital_id,,,,,,,
RS-XXX-26,BJB-01,7.33,300.0,102.14,49.21,-4.47,-2.79
RS-XXX-25,SM2-01,9.75,600.0,78.84,78.84,-2.25,-3.25
RS-XXX-09,IDP-01,4.27,120.0,9.34,9.34,-1.10,-0.78
RS-XXX-24,MDN-01,10.20,720.0,45.57,45.57,-1.05,-1.37
RS-XXX-23,MER-01,5.86,150.0,16.08,16.08,-0.96,-0.87
RS-XXX-07,BGR-02,6.76,260.0,19.24,19.24,-0.84,-0.80


RR per RS (balita+lansia, +10 µg/m³ rata-rata 7 hari): rata-rata RS lain 1.142 sd 0.314


,kode_stasiun,tipe,n,rr,rr_bawah,rr_atas,beta,se,p
hospital_id,,,,,,,,,
RS-XXX-25,SM2-01,Industri,646,1.098,0.845,1.427,0.093,0.134,0.486
RS-XXX-26,BJB-01,Gambut,699,1.116,1.075,1.158,0.109,0.019,0.000


z-skor RR terhadap RS lain: {'BJB-01': -0.08, 'SM2-01': -0.14}


In [30]:
# SM2-01: seberapa besar pemangkasan data (aturan 18 jam) memengaruhi eksposur dan hubungan dengan kasus?
s2 = daily_m[daily_m.kode_stasiun == "SM2-01"].set_index("tanggal")
rs25 = health_m[health_m.hospital_id == "RS-XXX-25"].set_index("date")
y25 = (rs25.resp_admissions_under5.astype(float) + rs25.resp_admissions_over60.astype(float))
cmp = {}
for nama, ser in (("ketat (≥18 jam)", s2.pm25_ug_m3), ("longgar (≥12 jam)", s2.pm25_ug_m3_longgar)):
    z = pd.concat([ser, y25], axis=1).dropna(); cmp[nama] = dict(n_hari=len(z), spearman=stats.spearmanr(z.iloc[:, 0], z.iloc[:, 1])[0], pm25_rata2=z.iloc[:, 0].mean())
kem = s2[s2.bulan.isin([7, 8, 9, 10])]
cmp = pd.DataFrame(cmp).T; display(cmp.round(3))
print(f"SM2-01 Jul–Okt: hari sah ketat {100 * kem.pm25_ug_m3.notna().mean():.0f}% vs longgar {100 * kem.pm25_ug_m3_longgar.notna().mean():.0f}% (notebook 03 mencatat ±30% dan 97%)")
KN["4_anomali"]["asimetri_kasus_pm"] = {"spearman_antar_rs_asli": float(stats.spearmanr(hs.kasus, hs.pm_asli)[0]), "spearman_antar_rs_topeng": float(stats.spearmanr(hs.kasus, hs.pm_topeng)[0]), "spearman_kasus_kapasitas": float(stats.spearmanr(hs.kasus, hs.kapasitas)[0]),
                                       "z_residual": hs.loc[hs.kode.isin(["BJB-01", "SM2-01", "MDN-01"]), ["kode", "z_asli", "z_topeng"]].round(3).to_dict("records"), "z_rr_dalam_rs": zrr, "rr_per_rs": rr_cmp[rr_cmp.kode_stasiun.isin(["BJB-01", "SM2-01"])].round(4).to_dict("records"),
                                       "sm2_korelasi": cmp.round(4).to_dict("index")}
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1, subplot_titles=("Antar-RS: kasus balita+lansia vs PM2.5 (log–log)", "RR per RS (+10 µg/m³, rata-rata 7 hari)"))
for jenis, pm, sym, wrn in (("apa adanya", "pm_asli", "circle-open", "#B3261E"), ("bertopeng", "pm_topeng", "circle", style.WARNA_UTAMA)):
    fig.add_trace(go.Scatter(x=hs[pm], y=hs.kasus, mode="markers+text", text=hs.kode, textposition="top center", textfont=dict(size=9), marker=dict(size=10, symbol=sym, color=wrn, line=dict(width=2, color=wrn)), name=f"PM2.5 {jenis}"), row=1, col=1)
fig.update_xaxes(type="log", title_text="PM2.5 rata-rata (µg/m³)", row=1, col=1); fig.update_yaxes(type="log", title_text="Kasus balita+lansia per hari", row=1, col=1)
x = rr_rs.sort_values("rr"); fig.add_trace(go.Scatter(x=x.rr, y=x.kode_stasiun, mode="markers", marker=dict(size=9, color=[("#B3261E" if k in ("BJB-01", "SM2-01") else style.WARNA_UTAMA) for k in x.kode_stasiun]), error_x=dict(type="data", symmetric=False, array=(x.rr_atas - x.rr).values, arrayminus=(x.rr - x.rr_bawah).values), showlegend=False), row=1, col=2)
fig.add_vline(x=1, line=dict(dash="dash", color=style.WARNA_TEKS_REDUP), row=1, col=2)
style.terapkan(fig, "BJB-01 dan SM2-01: apakah 'polusi tinggi, kasus rendah' anomali nyata?", 620)
simpan(fig, "05_4_polusi_tinggi_kasus_rendah", 620); fig.show()
b_, s_ = hs.loc[hs.kode == "BJB-01"].iloc[0], (hs.loc[hs.kode == "SM2-01"].iloc[0] if (hs.kode == "SM2-01").any() else None)
out_a = hs.index[hs.z_asli.abs() > 2]; out_t = hs.index[hs.z_topeng.abs() > 2]
f_ = lambda ix, col: ", ".join(f"{hs.loc[i, 'kode']} ({fl(hs.loc[i, col], 2)})" for i in ix) or "tidak ada"
bjb_, sm2_ = hs.loc[hs.kode == "BJB-01"].iloc[0], hs.loc[hs.kode == "SM2-01"].iloc[0]
rr_b, rr_s = rr_cmp[rr_cmp.kode_stasiun == "BJB-01"].iloc[0], rr_cmp[rr_cmp.kode_stasiun == "SM2-01"].iloc[0]
tafsir(f"Antar-RS (n = {len(hs)}), kasus balita+lansia berkorelasi dengan PM2.5 (ρ = {fl(stats.spearmanr(hs.kasus, hs.pm_asli)[0], 2)} apa adanya; {fl(stats.spearmanr(hs.kasus, hs.pm_topeng)[0], 2)} bertopeng) tetapi tidak dengan kapasitas tempat tidur (ρ = {fl(stats.spearmanr(hs.kasus, hs.kapasitas)[0], 2)}). Residual terstudentisasi (leave-one-out) di luar ±2: apa adanya {f_(out_a, 'z_asli')}; bertopeng {f_(out_t, 'z_topeng')}.",
       f"Untuk BJB-01 kesenjangan mengecil dari {fl(bjb_.z_asli, 2)} menjadi {fl(bjb_.z_topeng, 2)} setelah dataran flatline dibuang (rata-rata PM2.5 {fl(bjb_.pm_asli, 0)} → {fl(bjb_.pm_topeng, 0)} µg/m³): sebagian kesenjangan adalah artefak sensor, sebagian tersisa. SM2-01 {fl(sm2_.z_asli, 2)} → {fl(sm2_.z_topeng, 2)} (menjadi lebih negatif karena regresi tanpa artefak BJB-01 lebih ketat), jadi kasus balita+lansia RS-XXX-25 tetap lebih rendah daripada yang diharapkan dari PM2.5-nya.",
       f"Hipotesis salah-ukur eksposur (respons dalam-RS berbeda) tidak didukung: RR dalam-RS BJB-01 {fl(rr_b.rr, 3)} (CI {fl(rr_b.rr_bawah, 2)}–{fl(rr_b.rr_atas, 2)}) dan SM2-01 {fl(rr_s.rr, 3)} ({fl(rr_s.rr_bawah, 2)}–{fl(rr_s.rr_atas, 2)}; sangat lebar) tidak berbeda dari RS lain (z = {fl(zrr.get('BJB-01', np.nan), 2)} dan {fl(zrr.get('SM2-01', np.nan), 2)}). Pada SM2-01 korelasi harian dengan kasus {fl(cmp.iloc[0].spearman, 2)} (aturan 18 jam, n = {int(cmp.iloc[0].n_hari)}) naik menjadi {fl(cmp.iloc[1].spearman, 2)} (12 jam, n = {int(cmp.iloc[1].n_hari)}): pemangkasan musim asap melemahkan hubungan terukur.",
       "Penjelasan yang tersisa paling mungkin berupa tingkat dasar kasus yang rendah pada kedua RS (kapasitas tidak memprediksi jumlah kasus), tetapi data ini tidak dapat membedakannya dari perbedaan definisi pencatatan atau cakupan layanan; kesimpulan tentang 'kasus rendah' sebaiknya tidak dipakai untuk menilai kerentanan wilayah tersebut.")

,n_hari,spearman,pm25_rata2
ketat (≥18 jam),771.0,0.236,59.778
longgar (≥12 jam),1008.0,0.448,88.195


SM2-01 Jul–Okt: hari sah ketat 30% vs longgar 97% (notebook 03 mencatat ±30% dan 97%)


**Interpretasi.** Antar-RS (n = 26), kasus balita+lansia berkorelasi dengan PM2.5 (ρ = 0,65 apa adanya; 0,73 bertopeng) tetapi tidak dengan kapasitas tempat tidur (ρ = 0,18). Residual terstudentisasi (leave-one-out) di luar ±2: apa adanya SM2-01 (-2,25), BJB-01 (-4,47); bertopeng SM2-01 (-3,25), BJB-01 (-2,79). Untuk BJB-01 kesenjangan mengecil dari -4,47 menjadi -2,79 setelah dataran flatline dibuang (rata-rata PM2.5 102 → 49 µg/m³): sebagian kesenjangan adalah artefak sensor, sebagian tersisa. SM2-01 -2,25 → -3,25 (menjadi lebih negatif karena regresi tanpa artefak BJB-01 lebih ketat), jadi kasus balita+lansia RS-XXX-25 tetap lebih rendah daripada yang diharapkan dari PM2.5-nya. Hipotesis salah-ukur eksposur (respons dalam-RS berbeda) tidak didukung: RR dalam-RS BJB-01 1,116 (CI 1,07–1,16) dan SM2-01 1,098 (0,84–1,43; sangat lebar) tidak berbeda dari RS lain (z = -0,08 dan -0,14). Pada SM2-01 korelasi harian dengan kasus 0,24 (aturan 18 jam, n = 771) naik menjadi 0,45 (12 jam, n = 1008): pemangkasan musim asap melemahkan hubungan terukur. Penjelasan yang tersisa paling mungkin berupa tingkat dasar kasus yang rendah pada kedua RS (kapasitas tidak memprediksi jumlah kasus), tetapi data ini tidak dapat membedakannya dari perbedaan definisi pencatatan atau cakupan layanan; kesimpulan tentang 'kasus rendah' sebaiknya tidak dipakai untuk menilai kerentanan wilayah tersebut.

## 5. Skor risiko wilayah
**Komponen** (peringkat persentil 0–1 antar stasiun; data bertopeng): (1) **beban polusi** = rata-rata persentil dari rata-rata PM2.5 jam, % jam > 55 µg/m³, dan P95 jam; (2) **tren** = kemiringan Sen musiman (% per tahun; hanya stasiun Layak, lainnya netral 0,5); (3) **beban kesehatan** = kasus ISPA balita+lansia+asma+PPOK per 100 tempat tidur per hari; (4) **kerentanan** = rata-rata persentil dari 1/kapasitas RS dan RR per RS (balita+lansia, 7 hari).
**Bobot dasar:** polusi 0,35 · kesehatan 0,25 · kerentanan 0,25 · tren 0,15. Alasan: polusi adalah penyebab langsung dan terukur paling andal; kesehatan dan kerentanan sama penting untuk keputusan layanan; tren diberi bobot terkecil karena tidak ada tren signifikan pada 2,8 tahun data. **JMB-02** tidak dinilai (tidak punya data PM2.5 sah).

In [31]:
stas_r = [k for k in sorted(hourly_m.kode_stasiun.unique())]
hm = hourly_m[hourly_m.pm25_ug_m3.notna()]
pol = pd.DataFrame({"pm25_rata2": hm.groupby("kode_stasiun").pm25_ug_m3.mean(), "persen_jam_gt55": hm.groupby("kode_stasiun").pm25_ug_m3.apply(lambda s: 100 * (s > 55).mean()), "p95": hm.groupby("kode_stasiun").pm25_ug_m3.quantile(.95)}).reindex(stas_r)
# tren Sen musiman (stasiun Layak, bulan sah >= 20 hari) pada data bertopeng
dmk = daily_m[daily_m.pm25_ug_m3.notna()].copy(); dmk["ym"] = dmk.tanggal.dt.to_period("M").dt.to_timestamp()
g_ = dmk.groupby(["kode_stasiun", "ym"]).pm25_ug_m3.agg(["mean", "count"]); g_["mean"] = g_["mean"].where(g_["count"] >= 20); bulm = g_["mean"].unstack(0)
tren_r = {}
for k in stas_r:
    if status[k] == "Layak" and k in bulm and bulm[k].notna().sum() >= 24:
        r = eda.mk_musiman(bulm[k]); tren_r[k] = dict(slope_persen=100 * r["slope_tahun"] / bulm[k].median(), p=r["p"], n_bulan=r["n_bulan"])
tren_r = pd.DataFrame(tren_r).T.reindex(stas_r)
hb = health_m.copy(); hb["vul"] = hb.resp_admissions_under5.astype(float) + hb.resp_admissions_over60.astype(float) + hb.asthma_emergency_cases.astype(float) + hb.copd_admissions.astype(float)
beban = hb.groupby("kode_stasiun").agg(kasus_vul=("vul", "mean"), kapasitas=("kapasitas_tempat_tidur", "first")); beban["per100tt"] = 100 * beban.kasus_vul / beban.kapasitas
beban = beban.reindex(stas_r)
kerent = pd.DataFrame({"inv_kapasitas": 1 / beban.kapasitas, "rr_rs": rr_rs.set_index("kode_stasiun").rr}).reindex(stas_r)
komp = pd.DataFrame(index=stas_r)
komp["polusi"] = pd.concat([M.persentil(pol[c]) for c in pol], axis=1).mean(axis=1)
komp["tren"] = M.persentil(tren_r.slope_persen)
komp["kesehatan"] = M.persentil(beban.per100tt)
komp["kerentanan"] = pd.concat([M.persentil(kerent.inv_kapasitas), M.persentil(kerent.rr_rs)], axis=1).mean(axis=1)
imputasi = komp.isna()
komp_f = komp.fillna(0.5)
BOBOT = {"polusi": 0.35, "kesehatan": 0.25, "kerentanan": 0.25, "tren": 0.15}
komp_f["skor"] = M.skor_berbobot(komp_f, BOBOT) * 100
komp_f["peringkat"] = komp_f.skor.rank(ascending=False, method="min").astype(int)
for c in ("tipe", "pulau", "provinsi"): komp_f[c] = st[{"tipe": "kelompok_lokasi", "pulau": "pulau", "provinsi": "provinsi"}[c]].reindex(komp_f.index)
komp_f["status"] = status.reindex(komp_f.index); komp_f["komponen_netral"] = imputasi.sum(axis=1)
risiko = komp_f.sort_values("peringkat"); display(risiko[["peringkat", "skor", "polusi", "kesehatan", "kerentanan", "tren", "tipe", "pulau", "status", "komponen_netral"]].round(2))

,peringkat,skor,polusi,kesehatan,kerentanan,tren,tipe,pulau,status,komponen_netral
KTG-02,1,81.00,0.77,0.96,0.60,1.00,Gambut,Kalimantan,Layak,0
PSW-01,2,76.71,0.76,1.00,0.64,0.62,Perkotaan,Sumatera,Layak,0
PKN-01,3,73.98,0.69,0.81,0.70,0.81,Gambut,Kalimantan,Layak,0
PLB-01,4,72.47,0.94,0.92,0.37,0.50,Gambut,Sumatera,Terbatas,1
PKU-01,5,71.14,0.69,0.73,0.63,0.86,Gambut,Sumatera,Layak,0
BTM-01,6,70.24,0.71,0.77,0.65,0.67,Industri,Sumatera,Layak,0
KTG-01,7,65.41,0.78,0.62,0.33,0.95,Perkotaan,Kalimantan,Layak,0
JMB-01,8,65.25,0.63,0.69,0.61,0.71,Gambut,Sumatera,Layak,0
STG-01,9,64.00,0.58,0.58,0.72,0.76,Gambut,Kalimantan,Layak,0
JKT-02,10,59.36,0.81,0.88,0.19,0.29,Industri,Jawa,Layak,0


In [32]:
SB = M.sensitivitas_bobot(komp_f[list(BOBOT)], BOBOT, n=5000, seed=7, top_k=5)
rg = SB["ringkasan"].join(komp_f[["skor", "tipe", "status"]]); print("Sensitivitas bobot (Dirichlet 5.000 undian): rentang peringkat 90%"); display(rg.round(2).head(12))
print(f"Korelasi peringkat (Spearman) dasar vs bobot acak: median {SB['rho_dirichlet_median']:.2f}, persentil-5 {SB['rho_dirichlet_p05']:.2f}"); print("Satu-per-satu ±50%:"); display(SB["oat"].round(3))
# agregasi wilayah
wil_p = komp_f.groupby("pulau").agg(n_stasiun=("skor", "size"), skor_rata2=("skor", "mean"), skor_maks=("skor", "max")).reindex(PULAU).round(1)
wil_pr = komp_f.groupby("provinsi").agg(n_stasiun=("skor", "size"), skor_rata2=("skor", "mean"), pulau=("pulau", "first")).sort_values("skor_rata2", ascending=False).round(1)
print("Per pulau:"); display(wil_p); print("Per provinsi:"); display(wil_pr)
KN["5_risiko"] = {"bobot": BOBOT, "peringkat": risiko[["peringkat", "skor", "tipe", "pulau", "status"]].round(2).reset_index().rename(columns={"index": "kode_stasiun"}).to_dict("records"), "n_dinilai": int(len(risiko)),
                  "sensitivitas": {"rho_dirichlet_median": SB["rho_dirichlet_median"], "rho_dirichlet_p05": SB["rho_dirichlet_p05"], "top5_dasar": risiko.index[:5].tolist(), "peluang_top5": SB["ringkasan"].peluang_top5.round(3).to_dict(),
                                   "rentang_peringkat_90": {k: [float(r.p05), float(r.p95)] for k, r in SB["ringkasan"].iterrows()}},
                  "per_pulau": wil_p.to_dict("index"), "per_provinsi": wil_pr.to_dict("index")}
fig = make_subplots(rows=1, cols=2, column_widths=[0.55, 0.45], horizontal_spacing=0.1, subplot_titles=("Skor risiko stasiun (bobot dasar) dan rentang peringkat 90% pada bobot acak", "Skor risiko menurut komponen (bobot × persentil)"))
rr_ = rg.sort_values("peringkat_dasar")
fig.add_bar(y=rr_.index, x=rr_.skor, orientation="h", marker_color=[WARNA[t] for t in rr_.tipe], showlegend=False, text=[f"#{int(p)} ({int(a)}–{int(b)})" for p, a, b in zip(rr_.peringkat_dasar, rr_.p05, rr_.p95)], textposition="outside", row=1, col=1)
for c, wrn in (("polusi", "#E17055"), ("kesehatan", style.WARNA_UTAMA), ("kerentanan", "#6C5CE7"), ("tren", "#F2B01E")):
    fig.add_bar(y=rr_.index, x=(komp_f.loc[rr_.index, c] * BOBOT[c] * 100), orientation="h", name=f"{c} (×{BOBOT[c]})", marker_color=wrn, row=1, col=2)
fig.update_layout(barmode="stack"); fig.update_yaxes(autorange="reversed"); fig.update_xaxes(title_text="Skor (0–100)", range=[0, 100], row=1, col=1); fig.update_xaxes(title_text="Kontribusi komponen", row=1, col=2)
legenda_tipe(fig, 1, 1)
style.terapkan(fig, "Peringkat risiko stasiun (warna batang kiri = tipe lokasi)", 800)
simpan(fig, "05_5_skor_risiko", 800); fig.show()

Sensitivitas bobot (Dirichlet 5.000 undian): rentang peringkat 90%


,peringkat_dasar,median,p05,p95,peluang_top5,skor,tipe,status
KTG-02,1.0,1.0,1.0,3.0,0.98,81.00,Gambut,Layak
PSW-01,2.0,3.0,1.0,9.0,0.80,76.71,Perkotaan,Layak
PKN-01,3.0,3.0,2.0,6.0,0.93,73.98,Gambut,Layak
PLB-01,4.0,6.0,1.0,15.0,0.42,72.47,Gambut,Terbatas
PKU-01,5.0,5.0,3.0,8.0,0.67,71.14,Gambut,Layak
BTM-01,6.0,6.0,4.0,8.0,0.32,70.24,Industri,Layak
KTG-01,7.0,8.0,2.0,15.0,0.34,65.41,Perkotaan,Layak
JMB-01,8.0,8.0,6.0,11.0,0.00,65.25,Gambut,Layak
STG-01,9.0,8.0,4.0,12.0,0.22,64.00,Gambut,Layak
JKT-02,10.0,11.0,4.0,23.0,0.09,59.36,Industri,Layak


Korelasi peringkat (Spearman) dasar vs bobot acak: median 0.89, persentil-5 0.57
Satu-per-satu ±50%:


,komponen,faktor,spearman_vs_dasar,top_k_sama
0,polusi,0.5,0.947,4
1,polusi,1.5,0.986,5
2,kesehatan,0.5,0.988,5
3,kesehatan,1.5,0.989,5
4,kerentanan,0.5,0.981,5
5,kerentanan,1.5,0.973,4
6,tren,0.5,0.994,4
7,tren,1.5,0.994,5


Per pulau:


,n_stasiun,skor_rata2,skor_maks
pulau,,,
Sumatera,9,55.4,76.7
Jawa,7,45.0,59.4
Kalimantan,6,63.6,81.0
Sulawesi,2,35.3,47.9
Papua,2,41.0,42.5


Per provinsi:


,n_stasiun,skor_rata2,pulau
provinsi,,,
Lampung,1,76.7,Sumatera
Kalimantan Tengah,3,73.5,Kalimantan
Sumatera Selatan,1,72.5,Sumatera
Riau,1,71.1,Sumatera
Kepulauan Riau,1,70.2,Sumatera
Jambi,1,65.2,Sumatera
Kalimantan Barat,1,64.0,Kalimantan
DKI Jakarta,3,54.1,Jawa
Kalimantan Timur,1,50.9,Kalimantan


In [33]:
top5 = risiko.index[:5].tolist(); pt5 = SB["ringkasan"].peluang_top5
stabil = [k for k in top5 if pt5[k] >= 0.5]
tafsir(f"Lima stasiun berisiko tertinggi pada bobot dasar: {', '.join(f'{k} ({fl(risiko.loc[k, 'skor'], 0)})' for k in top5)}; {len(stabil)} di antaranya tetap masuk lima besar pada ≥ 50% undian bobot acak ({', '.join(f'{k} {fl(100 * pt5[k], 0)}%' for k in top5)}).",
       f"Peringkat tahan terhadap bobot: korelasi Spearman dengan peringkat dasar median {fl(SB['rho_dirichlet_median'], 2)} (persentil-5 {fl(SB['rho_dirichlet_p05'], 2)}); perubahan satu bobot ±50% mempertahankan {int(SB['oat'].top_k_sama.min())}–{int(SB['oat'].top_k_sama.max())} dari 5 stasiun teratas.",
       f"Menurut pulau skor rata-rata tertinggi {wil_p.skor_rata2.idxmax()} ({fl(wil_p.skor_rata2.max(), 1)}) dan provinsi {wil_pr.index[0]} ({fl(wil_pr.skor_rata2.iloc[0], 1)}); {int((komp_f.komponen_netral > 0).sum())} stasiun memakai nilai netral pada ≥ 1 komponen (tren tidak dihitung untuk stasiun non-Layak), sehingga skornya kurang pasti. Skor ini peringkat relatif antar-stasiun di jaringan, bukan ukuran risiko absolut.")

**Interpretasi.** Lima stasiun berisiko tertinggi pada bobot dasar: KTG-02 (81), PSW-01 (77), PKN-01 (74), PLB-01 (72), PKU-01 (71); 4 di antaranya tetap masuk lima besar pada ≥ 50% undian bobot acak (KTG-02 98%, PSW-01 80%, PKN-01 93%, PLB-01 42%, PKU-01 67%). Peringkat tahan terhadap bobot: korelasi Spearman dengan peringkat dasar median 0,89 (persentil-5 0,57); perubahan satu bobot ±50% mempertahankan 4–5 dari 5 stasiun teratas. Menurut pulau skor rata-rata tertinggi Kalimantan (63,6) dan provinsi Lampung (76,7); 5 stasiun memakai nilai netral pada ≥ 1 komponen (tren tidak dihitung untuk stasiun non-Layak), sehingga skornya kurang pasti. Skor ini peringkat relatif antar-stasiun di jaringan, bukan ukuran risiko absolut.

## 6. Kecukupan jaringan pemantauan
### 6.1 Cakupan, kelengkapan, dan status sensor
Dasar: `qc_summary` dan `sites_clean` (notebook 03). Provinsi tanpa stasiun dihitung terhadap **daftar 38 provinsi** (acuan eksternal, pengetahuan umum; data proyek hanya memuat provinsi yang bertanda stasiun). Sentroid provinsi adalah **koordinat kasar** (± ratusan km) dan hanya untuk analisis kesenjangan, bukan penentuan lokasi.

In [34]:
prov_stasiun = set(sites.provinsi); prov_tanpa = [p for p in M.PROVINSI_ACUAN if p not in prov_stasiun]
assert prov_stasiun <= set(M.PROVINSI_ACUAN), prov_stasiun - set(M.PROVINSI_ACUAN)
print(f"{len(prov_stasiun)} dari 38 provinsi memiliki stasiun; {len(prov_tanpa)} tanpa stasiun: {', '.join(prov_tanpa)}")
ps = sites.groupby("pulau").agg(stasiun=("kode_stasiun", "size"), provinsi=("provinsi", "nunique")).reindex(PULAU)
ps["layak"] = qc.merge(sites[["kode_stasiun", "pulau"]], on="kode_stasiun").query("status_analisis == 'Layak'").groupby("pulau").size().reindex(PULAU).fillna(0).astype(int)
ps["terbatas_dikecualikan"] = ps.stasiun - ps.layak; display(ps)
D = M.matriks_jarak(sites.set_index("kode_stasiun")); D = D.where(~np.eye(len(D), dtype=bool), np.inf)
nn = pd.DataFrame({"tetangga_terdekat": D.idxmin(axis=1), "jarak_km": D.min(axis=1)}); nn["pulau"] = st.pulau; nn["status"] = status
print("Jarak ke stasiun terdekat (km):", nn.jarak_km.describe().round(0).to_dict()); display(nn.sort_values("jarak_km", ascending=False).head(6).round(0))
print("Pasangan < 100 km:", int((M.matriks_jarak(sites.set_index('kode_stasiun')).where(lambda x: x > 0).stack() < 100).sum() // 2))
qs = qc.set_index("kode_stasiun")[["status_sensor", "persen_jam_pm25_valid", "persen_hari_pm25_valid", "persen_hari_pm25_valid_kemarau", "cakupan_terendah_persen", "status_analisis"]]
display(qs.groupby("status_analisis").agg(n=("status_sensor", "size"), cakupan_median=("cakupan_terendah_persen", "median")).round(1))

21 dari 38 provinsi memiliki stasiun; 17 tanpa stasiun: Kepulauan Bangka Belitung, Banten, Jawa Tengah, Bali, Nusa Tenggara Barat, Nusa Tenggara Timur, Kalimantan Utara, Sulawesi Utara, Gorontalo, Sulawesi Barat, Sulawesi Tenggara, Maluku, Maluku Utara, Papua Barat, Papua, Papua Tengah, Papua Pegunungan


,stasiun,provinsi,layak,terbatas_dikecualikan
pulau,,,,
Sumatera,10,9,7,3
Jawa,7,4,7,0
Kalimantan,6,4,5,1
Sulawesi,2,2,2,0
Papua,2,2,1,1


Jarak ke stasiun terdekat (km): {'count': 27.0, 'mean': 267.0, 'std': 323.0, 'min': 1.0, '25%': 65.0, '50%': 186.0, '75%': 335.0, 'max': 1319.0}


,tetangga_terdekat,jarak_km,pulau,status
kode_stasiun,,,,
MER-01,SRG-01,1319.0,Papua,Terbatas
SRG-01,PLU-01,1239.0,Papua,Layak
MDN-01,IDP-01,425.0,Sumatera,Terbatas
IDP-01,MDN-01,425.0,Sumatera,Layak
MRS-01,PLU-01,378.0,Sulawesi,Layak
SM2-01,PLU-01,361.0,Kalimantan,Terbatas


Pasangan < 100 km: 11


,n,cakupan_median
status_analisis,,
Dikecualikan,1,0.5
Layak,22,99.4
Terbatas,4,45.1


In [35]:
# peluruhan korelasi PM2.5 antar-stasiun terhadap jarak (stasiun Layak, data bertopeng): apakah ada 'radius representatif'?
lay = [k for k in daily_m.kode_stasiun.unique() if status[k] == "Layak"]
Pm = daily_m.pivot(index="tanggal", columns="kode_stasiun", values="pm25_ug_m3")[lay]
an = Pm.sub(Pm.groupby(Pm.index.month).transform("mean"))
Dl = M.matriks_jarak(sites.set_index("kode_stasiun").loc[lay]); iu = np.triu_indices(len(lay), 1)
dc = pd.DataFrame({"jarak": Dl.to_numpy()[iu], "r_mentah": Pm.corr().to_numpy()[iu], "r_anomali": an.corr().to_numpy()[iu]})
sp_raw, sp_an = stats.spearmanr(dc.jarak, dc.r_mentah), stats.spearmanr(dc.jarak, dc.r_anomali)
bins = pd.cut(dc.jarak, [0, 100, 300, 1000, 2000, 7000]); tb_dc = dc.groupby(bins).agg(n_pasang=("jarak", "size"), r_mentah_median=("r_mentah", "median"), r_anomali_median=("r_anomali", "median")); display(tb_dc.round(2))
print(f"Spearman jarak vs korelasi: mentah ρ={sp_raw[0]:.2f} ({pv(sp_raw[1])}); anomali tanpa musim ρ={sp_an[0]:.2f} ({pv(sp_an[1])}); n pasangan = {len(dc)}")
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08, subplot_titles=("Korelasi harian PM2.5 antar-stasiun Layak vs jarak", "Kelengkapan data per stasiun (cakupan PM2.5 sah)"))
fig.add_trace(go.Scatter(x=dc.jarak, y=dc.r_mentah, mode="markers", marker=dict(size=5, color=style.WARNA_UTAMA, opacity=0.5), name="mentah (termasuk musim)"), row=1, col=1)
fig.add_trace(go.Scatter(x=dc.jarak, y=dc.r_anomali, mode="markers", marker=dict(size=5, color="#E17055", opacity=0.5), name="anomali (musim dibuang)"), row=1, col=1)
q2 = qs.sort_values("cakupan_terendah_persen")
for c, wrn, nmc in (("persen_jam_pm25_valid", "#8CCBC3", "jam sah"), ("persen_hari_pm25_valid", "#2E9E94", "hari sah"), ("persen_hari_pm25_valid_kemarau", "#0B4F55", "hari sah musim kemarau")):
    fig.add_bar(y=q2.index, x=q2[c], orientation="h", name=nmc, marker_color=wrn, row=1, col=2)
fig.update_xaxes(type="log", title_text="Jarak antar-stasiun (km)", row=1, col=1); fig.update_yaxes(title_text="Korelasi Pearson", row=1, col=1); fig.update_xaxes(title_text="% dari periode", row=1, col=2)
fig.update_layout(barmode="group")
style.terapkan(fig, "Redundansi dan kelengkapan jaringan", 700)
simpan(fig, "05_6_redundansi_kelengkapan", 700); fig.show()
KN["6_jaringan"] = {"n_provinsi_dengan_stasiun": len(prov_stasiun), "n_provinsi_tanpa_stasiun": len(prov_tanpa), "provinsi_tanpa_stasiun": prov_tanpa, "stasiun_per_pulau": ps.to_dict("index"), "jarak_terdekat_median_km": float(nn.jarak_km.median()), "jarak_terdekat_maks": {"kode": nn.jarak_km.idxmax(), "km": float(nn.jarak_km.max())},
                    "status": qc.status_analisis.value_counts().to_dict(), "spearman_jarak_korelasi_mentah": [float(sp_raw[0]), float(sp_raw[1])], "spearman_jarak_korelasi_anomali": [float(sp_an[0]), float(sp_an[1])], "n_pasangan": int(len(dc))}
tafsir(f"Hanya {len(prov_stasiun)} dari 38 provinsi memiliki stasiun ({len(prov_tanpa)} tanpa stasiun, antara lain {', '.join(prov_tanpa[:6])}); pulau besar tercakup (Sumatera {ps.loc['Sumatera', 'stasiun']}, Jawa {ps.loc['Jawa', 'stasiun']}, Kalimantan {ps.loc['Kalimantan', 'stasiun']}, Sulawesi {ps.loc['Sulawesi', 'stasiun']}, Papua {ps.loc['Papua', 'stasiun']}) tetapi Bali–Nusa Tenggara, Maluku, dan sebagian Papua tidak. Dari {len(qc)} stasiun, {int((qc.status_analisis == 'Layak').sum())} berstatus Layak dan {int((qc.status_analisis != 'Layak').sum())} Terbatas/Dikecualikan ({', '.join(qc[qc.status_analisis != 'Layak'].kode_stasiun)}).",
       f"Korelasi harian antar-stasiun tidak meluruh dengan jarak (ρ = {fl(sp_raw[0], 2)}, {pv(sp_raw[1])}); setelah musim dibuang korelasi ≈ {fl(dc.r_anomali.median(), 2)} pada semua jarak, termasuk pasangan < 100 km (median {fl(tb_dc.r_anomali_median.iloc[0], 2)}). Artinya variasi harian tiap stasiun bersifat lokal: tidak ada 'radius representatif' yang dapat diturunkan, tetapi juga tidak ada bukti stasiun saling mubazir (redundan).",
       "Karena radius representatif tidak dapat diestimasi dari data, kesenjangan cakupan pada 6.2 dinilai dengan jarak ke stasiun Layak terdekat dan kondisi stasiun sekitar, bukan dengan ambang radius tertentu.")

,n_pasang,r_mentah_median,r_anomali_median
jarak,,,
"(0, 100]",10,0.04,-0.00
"(100, 300]",13,0.77,0.04
"(300, 1000]",108,0.70,0.03
"(1000, 2000]",71,0.72,0.03
"(2000, 7000]",29,0.66,0.01


Spearman jarak vs korelasi: mentah ρ=-0.04 (p = 0,580); anomali tanpa musim ρ=-0.10 (p = 0,137); n pasangan = 231


**Interpretasi.** Hanya 21 dari 38 provinsi memiliki stasiun (17 tanpa stasiun, antara lain Kepulauan Bangka Belitung, Banten, Jawa Tengah, Bali, Nusa Tenggara Barat, Nusa Tenggara Timur); pulau besar tercakup (Sumatera 10, Jawa 7, Kalimantan 6, Sulawesi 2, Papua 2) tetapi Bali–Nusa Tenggara, Maluku, dan sebagian Papua tidak. Dari 27 stasiun, 22 berstatus Layak dan 5 Terbatas/Dikecualikan (JMB-02, MDN-01, MER-01, PLB-01, SM2-01). Korelasi harian antar-stasiun tidak meluruh dengan jarak (ρ = -0,04, p = 0,580); setelah musim dibuang korelasi ≈ 0,03 pada semua jarak, termasuk pasangan < 100 km (median 0,00). Artinya variasi harian tiap stasiun bersifat lokal: tidak ada 'radius representatif' yang dapat diturunkan, tetapi juga tidak ada bukti stasiun saling mubazir (redundan). Karena radius representatif tidak dapat diestimasi dari data, kesenjangan cakupan pada 6.2 dinilai dengan jarak ke stasiun Layak terdekat dan kondisi stasiun sekitar, bukan dengan ambang radius tertentu.

### 6.2 Skor prioritas: penambahan stasiun baru dan perbaikan stasiun
**(A) Penambahan stasiun** — kandidat: provinsi tanpa stasiun. **(B) Perbaikan** — stasiun non-Layak atau dengan flatline. Seluruh kriteria dinormalkan min–max (0–1) dalam kelompok kandidat.
Kriteria A (bobot): **kesenjangan jarak** ke stasiun Layak terdekat (0,35); **proksi polusi** = PM2.5 rata-rata IDW tiga stasiun Layak terdekat (0,20); **proksi asap** = % hari Tidak Sehat+ IDW tiga stasiun terdekat (0,25); **ketidakandalan sekitar** = porsi non-Layak di tiga stasiun terdekat (0,20). Proksi bergantung pada anggapan bahwa wilayah mirip tetangga terdekat dan sentroid kasar — **bukan** pengukuran.
Kriteria B (bobot): **kesenjangan data** (100 − cakupan terendah; 0,30), **isolasi** = jarak ke stasiun Layak lain (0,20), **beban polusi** (0,20), **risiko asap** (0,15), **beban kesehatan** RS pasangan (0,15).

In [36]:
layak_k = [k for k in qc.kode_stasiun if status[k] == "Layak" and k != "BJB-01"]          # BJB-01 punya flatline: dikeluarkan sebagai acuan proksi
ref = st.loc[layak_k]
pm_st = hourly_m.groupby("kode_stasiun").pm25_ug_m3.mean()
dl_ok = daily_m[daily_m.pm25_ug_m3.notna()]
asap_st = dl_ok.assign(tsp=dl_ok.pm25_ug_m3 > 55.4).groupby("kode_stasiun").tsp.mean() * 100            # % hari > 55,4 (Tidak Sehat+)
rows = []
for p in prov_tanpa:
    la, lo = M.PROVINSI_ACUAN[p]
    v_pm, nb_, dist = M.idw(la, lo, ref, pm_st); v_as, _, _ = M.idw(la, lo, ref, asap_st)
    semua_d = M.haversine_km(la, lo, sites.latitude_dd.to_numpy(), sites.longitude_dd.to_numpy()); idx3 = np.argsort(semua_d)[:3]
    rows.append(dict(provinsi=p, jarak_layak_terdekat_km=float(dist[0]), tetangga=", ".join(nb_), proksi_pm25=v_pm, proksi_asap_persen=v_as, tidak_andal=float(np.mean([status[sites.kode_stasiun.iloc[i]] != "Layak" for i in idx3]))))
A_ = pd.DataFrame(rows).set_index("provinsi")
def minmax(s): return (s - s.min()) / (s.max() - s.min()) if s.max() > s.min() else s * 0
kA = pd.DataFrame({"jarak": minmax(A_.jarak_layak_terdekat_km), "polusi": minmax(A_.proksi_pm25), "asap": minmax(A_.proksi_asap_persen), "tidak_andal": minmax(A_.tidak_andal)})
BA = {"jarak": 0.35, "polusi": 0.20, "asap": 0.25, "tidak_andal": 0.20}
A_["skor"] = M.skor_berbobot(kA, BA) * 100; A_["peringkat"] = A_.skor.rank(ascending=False, method="min").astype(int)
SA = M.sensitivitas_bobot(kA, BA, n=5000, seed=17, top_k=5)
A_ = A_.join(SA["ringkasan"][["median", "p05", "p95", "peluang_top5"]]).sort_values("peringkat")
display(A_.round(2))
print(f"Sensitivitas A: Spearman median {SA['rho_dirichlet_median']:.2f}, p5 {SA['rho_dirichlet_p05']:.2f}"); display(SA["oat"].round(3))

,jarak_layak_terdekat_km,tetangga,proksi_pm25,proksi_asap_persen,tidak_andal,skor,peringkat,median,p05,p95,peluang_top5
provinsi,,,,,,,,,,,
Kepulauan Bangka Belitung,306.28,"PSW-01, JMB-01, JKT-02",50.91,44.29,0.67,74.57,1,1.0,1.0,4.0,0.97
Kalimantan Utara,646.61,"STG-01, KTG-01, PLU-01",37.22,15.82,0.33,53.65,2,2.0,2.0,4.0,1.00
Papua,956.61,"SRG-01, PLU-01, MRS-01",19.85,0.00,0.33,46.49,3,3.0,1.0,9.0,0.77
Papua Pegunungan,917.04,"SRG-01, PLU-01, MRS-01",19.84,0.00,0.33,44.93,4,4.0,2.0,10.0,0.68
Papua Tengah,617.50,"SRG-01, PLU-01, MRS-01",19.36,0.00,0.33,32.94,5,6.0,5.0,12.0,0.41
Sulawesi Utara,591.90,"PLU-01, SRG-01, MRS-01",18.93,0.00,0.33,31.68,6,8.0,6.0,13.0,0.03
Sulawesi Tenggara,308.11,"MRS-01, PLU-01, KTG-01",27.78,1.39,0.33,26.64,7,8.0,3.0,11.0,0.23
Nusa Tenggara Barat,466.79,"MRS-01, MLG-01, YKY-01",35.11,0.08,0.00,26.48,8,8.0,4.0,13.0,0.26
Nusa Tenggara Timur,535.57,"MRS-01, PLU-01, MLG-01",30.15,0.00,0.00,26.16,9,9.0,6.0,15.0,0.02


Sensitivitas A: Spearman median 0.87, p5 0.45


,komponen,faktor,spearman_vs_dasar,top_k_sama
0,jarak,0.5,0.946,5
1,jarak,1.5,0.973,5
2,polusi,0.5,0.958,5
3,polusi,1.5,0.975,5
4,asap,0.5,0.995,5
5,asap,1.5,0.998,5
6,tidak_andal,0.5,0.961,5
7,tidak_andal,1.5,0.980,5


In [37]:
rep = [k for k in qc.kode_stasiun if status[k] != "Layak"] + sorted(set(flat.kode_stasiun))
rep = list(dict.fromkeys(rep))
rows = []
for k in rep:
    dk = daily_m[daily_m.kode_stasiun == k]
    cov = 100 * dk.pm25_ug_m3.notna().mean() if k in set(dk.kode_stasiun) and len(dk) else 0.0
    cov_use = min(qc.set_index("kode_stasiun").loc[k, "cakupan_terendah_persen"], cov) if len(dk) else qc.set_index("kode_stasiun").loc[k, "cakupan_terendah_persen"]
    refk = ref.drop(index=k, errors="ignore"); la, lo = st.loc[k, "latitude_dd"], st.loc[k, "longitude_dd"]
    d_ = M.haversine_km(la, lo, refk.latitude_dd.to_numpy(), refk.longitude_dd.to_numpy())
    pm_own = pm_st.get(k, np.nan) if k in pm_st.index and not pd.isna(pm_st.get(k, np.nan)) else M.idw(la, lo, refk, pm_st)[0]
    as_own = asap_st.get(k, np.nan); as_own = as_own if not pd.isna(as_own) else M.idw(la, lo, refk, asap_st)[0]
    rows.append(dict(kode_stasiun=k, status=status[k], flatline=k in set(flat.kode_stasiun), cakupan_terendah_persen=cov_use, kesenjangan_data=100 - cov_use, isolasi_km=float(d_.min()), pm25=pm_own, asap_persen=as_own, kesehatan_per100tt=beban.per100tt.get(k, np.nan),
                     sumber_polusi="stasiun sendiri" if k in pm_st.index and not pd.isna(pm_st.get(k, np.nan)) else "proksi tetangga"))
B_ = pd.DataFrame(rows).set_index("kode_stasiun")
B_["kesehatan_per100tt"] = B_.kesehatan_per100tt.fillna(B_.kesehatan_per100tt.median())
kB = pd.DataFrame({"kesenjangan_data": minmax(B_.kesenjangan_data), "isolasi": minmax(B_.isolasi_km), "polusi": minmax(B_.pm25), "asap": minmax(B_.asap_persen), "kesehatan": minmax(B_.kesehatan_per100tt)})
BB = {"kesenjangan_data": 0.30, "isolasi": 0.20, "polusi": 0.20, "asap": 0.15, "kesehatan": 0.15}
B_["skor"] = M.skor_berbobot(kB, BB) * 100; B_["peringkat"] = B_.skor.rank(ascending=False, method="min").astype(int)
SBr = M.sensitivitas_bobot(kB, BB, n=5000, seed=19, top_k=3)
B_ = B_.join(SBr["ringkasan"][["median", "p05", "p95", "peluang_top3"]]).sort_values("peringkat"); display(B_.round(2))
print(f"Sensitivitas B: Spearman median {SBr['rho_dirichlet_median']:.2f}, p5 {SBr['rho_dirichlet_p05']:.2f}"); display(SBr["oat"].round(3))
KN["6_jaringan"].update({"bobot_A": BA, "bobot_B": BB, "prioritas_penambahan": A_.round(3).reset_index().to_dict("records"), "prioritas_perbaikan": B_.round(3).reset_index().to_dict("records"),
                         "sensitivitas_A": {"rho_median": SA["rho_dirichlet_median"], "rho_p05": SA["rho_dirichlet_p05"], "oat_top5_sama_min": int(SA["oat"].top_k_sama.min())}, "sensitivitas_B": {"rho_median": SBr["rho_dirichlet_median"], "rho_p05": SBr["rho_dirichlet_p05"], "oat_top3_sama_min": int(SBr["oat"].top_k_sama.min())},
                         "catatan": "kandidat A memakai sentroid kasar provinsi (acuan eksternal) dan proksi dari 3 stasiun Layak terdekat; bukan pengukuran"})

,status,flatline,cakupan_terendah_persen,kesenjangan_data,isolasi_km,pm25,asap_persen,kesehatan_per100tt,sumber_polusi,skor,peringkat,median,p05,p95,peluang_top3
kode_stasiun,,,,,,,,,,,,,,,
MER-01,Terbatas,False,0.20,99.80,1318.66,16.08,0.00,6.58,stasiun sendiri,53.13,1,3.0,1.0,6.0,0.58
JMB-02,Dikecualikan,False,0.50,99.50,0.87,44.45,25.25,5.77,proksi tetangga,53.00,2,3.0,1.0,6.0,0.74
PLB-01,Terbatas,False,68.37,31.63,187.32,69.05,32.33,19.34,stasiun sendiri,50.42,3,1.0,1.0,5.0,0.89
SM2-01,Terbatas,False,53.83,46.17,360.66,78.84,13.62,3.88,stasiun sendiri,39.36,4,4.0,2.0,5.0,0.40
MDN-01,Terbatas,False,36.44,63.56,425.02,45.57,6.47,3.21,stasiun sendiri,33.27,5,5.0,3.0,6.0,0.09
BJB-01,Layak,True,70.04,29.96,175.19,49.21,28.61,5.77,stasiun sendiri,28.86,6,4.0,2.0,6.0,0.30


Sensitivitas B: Spearman median 0.54, p5 -0.31


,komponen,faktor,spearman_vs_dasar,top_k_sama
0,kesenjangan_data,0.5,0.771,3
1,kesenjangan_data,1.5,1.000,3
2,isolasi,0.5,0.829,3
3,isolasi,1.5,1.000,3
4,polusi,0.5,1.000,3
5,polusi,1.5,0.771,3
6,asap,0.5,1.000,3
7,asap,1.5,0.771,3
8,kesehatan,0.5,0.943,3
9,kesehatan,1.5,0.829,3


In [38]:
fig = go.Figure()
# cincin 100 km dan 300 km dari stasiun Layak
th = np.linspace(0, 2 * np.pi, 60)
for r_km, wrn in ((100, "rgba(11,110,110,0.35)"), (300, "rgba(11,110,110,0.15)")):
    for k in layak_k:
        la, lo = st.loc[k, "latitude_dd"], st.loc[k, "longitude_dd"]
        fig.add_trace(go.Scattergeo(lat=la + (r_km / 111.0) * np.sin(th), lon=lo + (r_km / (111.0 * np.cos(np.radians(la)))) * np.cos(th), mode="lines", line=dict(width=0.6, color=wrn), hoverinfo="skip", showlegend=False))
warna_s = {"Layak": style.WARNA_UTAMA, "Terbatas": style.KEPARAHAN_WARNA["Tinggi"], "Dikecualikan": style.KEPARAHAN_WARNA["Kritis"]}
for stt, wrn in warna_s.items():
    x = sites[sites.kode_stasiun.map(status) == stt]
    fig.add_trace(go.Scattergeo(lat=x.latitude_dd, lon=x.longitude_dd, mode="markers+text", text=x.kode_stasiun, textposition="top right", textfont=dict(size=8), marker=dict(size=9, color=wrn, line=dict(width=1, color="white")), name=f"Stasiun {stt} ({len(x)})",
                                hovertemplate="%{text}<extra></extra>"))
fig.add_trace(go.Scattergeo(lat=[M.PROVINSI_ACUAN[p][0] for p in A_.index], lon=[M.PROVINSI_ACUAN[p][1] for p in A_.index], mode="markers+text", text=[f"{p} (#{int(r)})" for p, r in zip(A_.index, A_.peringkat)], textposition="bottom center", textfont=dict(size=8),
                            marker=dict(size=[8 + 0.22 * s for s in A_.skor], symbol="diamond-open", color=style.WARNA_TEKS, line=dict(width=2)), name="Provinsi tanpa stasiun (ukuran = skor prioritas; sentroid kasar)"))
fig.update_geos(lataxis_range=[-11.5, 6.5], lonaxis_range=[94, 142], resolution=50, showcountries=True, countrycolor="#B0B8C0", showland=True, landcolor="#EEF2F5", showocean=True, oceancolor="#F7FAFC", coastlinecolor="#9AA5B1")
style.terapkan(fig, "Peta jaringan: stasiun menurut status, cincin 100/300 km dari stasiun Layak, dan provinsi tanpa stasiun", 760)
fig.update_layout(legend=dict(orientation="h", y=-0.02)); simpan(fig, "05_6_peta_jaringan", 760); fig.show()

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12, subplot_titles=("A. Prioritas penambahan stasiun (provinsi tanpa stasiun)", "B. Prioritas perbaikan stasiun"))
for i, (df_, lab_) in enumerate(((A_.sort_values("peringkat"), "peluang_top5"), (B_.sort_values("peringkat"), "peluang_top3")), 1):
    fig.add_bar(y=df_.index, x=df_.skor, orientation="h", marker_color=style.WARNA_UTAMA if i == 1 else "#E17055", showlegend=False,
                text=[f"#{int(p)} · rentang {int(a)}–{int(b)} · top{5 if i == 1 else 3} {fl(100 * q, 0)}%" for p, a, b, q in zip(df_.peringkat, df_.p05, df_.p95, df_[lab_])], textposition="outside", row=1, col=i)
fig.update_yaxes(autorange="reversed"); fig.update_xaxes(title_text="Skor prioritas (0–100)", range=[0, 135])
style.terapkan(fig, "Skor prioritas dan stabilitas peringkat terhadap bobot (5.000 undian Dirichlet)", 720)
simpan(fig, "05_6_skor_prioritas", 720); fig.show()

In [39]:
topA, topB = A_.index[:3].tolist(), B_.index[:3].tolist()
tafsir(f"Prioritas penambahan stasiun (A) teratas: {', '.join(f'{k} (skor {fl(A_.loc[k, 'skor'], 0)}, {fl(A_.loc[k, 'jarak_layak_terdekat_km'], 0)} km dari stasiun Layak terdekat)' for k in topA)}; peluang masuk lima besar pada bobot acak {', '.join(f'{k} {fl(100 * A_.loc[k, 'peluang_top5'], 0)}%' for k in topA)}, korelasi peringkat median {fl(SA['rho_dirichlet_median'], 2)} (persentil-5 {fl(SA['rho_dirichlet_p05'], 2)}) → tingkat atas relatif stabil, urutan di bawahnya tidak.",
       f"Prioritas perbaikan (B): {', '.join(f'{k} (skor {fl(B_.loc[k, 'skor'], 0)}, cakupan {fl(B_.loc[k, 'cakupan_terendah_persen'], 0)}%, peluang top-3 {fl(100 * B_.loc[k, 'peluang_top3'], 0)}%)' for k in B_.index[:4])}. Dengan hanya {len(B_)} kandidat, peringkat tidak stabil (Spearman median {fl(SBr['rho_dirichlet_median'], 2)}, persentil-5 {fl(SBr['rho_dirichlet_p05'], 2)}); yang dapat dipercaya hanya tingkat prioritas. JMB-02 berjarak {fl(B_.loc['JMB-02', 'isolasi_km'], 1)} km dari stasiun Layak terdekat (JMB-01), sehingga kerugian cakupan spasialnya kecil meski datanya kosong; MER-01 {fl(B_.loc['MER-01', 'isolasi_km'], 0)} km dari stasiun terdekat sehingga perbaikannya paling bernilai bagi cakupan Papua bagian selatan.",
       "Peringkat A sangat bergantung pada sentroid kasar dan anggapan 'mirip tetangga terdekat'; hasilnya harus diverifikasi dengan data penduduk, titik panas, dan sebaran gambut sebelum menjadi keputusan lokasi.")

**Interpretasi.** Prioritas penambahan stasiun (A) teratas: Kepulauan Bangka Belitung (skor 75, 306 km dari stasiun Layak terdekat), Kalimantan Utara (skor 54, 647 km dari stasiun Layak terdekat), Papua (skor 46, 957 km dari stasiun Layak terdekat); peluang masuk lima besar pada bobot acak Kepulauan Bangka Belitung 97%, Kalimantan Utara 100%, Papua 77%, korelasi peringkat median 0,87 (persentil-5 0,45) → tingkat atas relatif stabil, urutan di bawahnya tidak. Prioritas perbaikan (B): MER-01 (skor 53, cakupan 0%, peluang top-3 58%), JMB-02 (skor 53, cakupan 0%, peluang top-3 74%), PLB-01 (skor 50, cakupan 68%, peluang top-3 89%), SM2-01 (skor 39, cakupan 54%, peluang top-3 40%). Dengan hanya 6 kandidat, peringkat tidak stabil (Spearman median 0,54, persentil-5 -0,31); yang dapat dipercaya hanya tingkat prioritas. JMB-02 berjarak 0,9 km dari stasiun Layak terdekat (JMB-01), sehingga kerugian cakupan spasialnya kecil meski datanya kosong; MER-01 1.319 km dari stasiun terdekat sehingga perbaikannya paling bernilai bagi cakupan Papua bagian selatan. Peringkat A sangat bergantung pada sentroid kasar dan anggapan 'mirip tetangga terdekat'; hasilnya harus diverifikasi dengan data penduduk, titik panas, dan sebaran gambut sebelum menjadi keputusan lokasi.

## 7. Matriks bukti per tipe wilayah (dasar rekomendasi)
Ringkasan angka yang dipakai untuk menautkan rekomendasi kebijakan ke bukti. Seluruh angka berasal dari bagian 0–6 notebook ini (data bertopeng).

In [40]:
dk = daily_m[daily_m.pm25_ug_m3.notna() & daily_m.kode_stasiun.isin(stas_ml)].copy(); dk["tipe"] = dk.kode_stasiun.map(st.kelompok_lokasi)
bt = pd.DataFrame(index=TIPE)
bt["n_stasiun"] = st.groupby("kelompok_lokasi").size().reindex(TIPE)
bt["n_non_layak"] = pd.Series(status[status != "Layak"].index.map(st.kelompok_lokasi)).value_counts().reindex(TIPE).fillna(0).astype(int).values
bt["pm25_rata2_harian"] = dk.groupby("tipe").pm25_ug_m3.mean()
bt["persen_hari_gt55"] = dk.groupby("tipe").pm25_ug_m3.apply(lambda s: 100 * (s > 55).mean())
bt["persen_hari_gt150"] = dk.groupby("tipe").pm25_ug_m3.apply(lambda s: 100 * (s > 150).mean())
bt["hari_episode"] = daily_m[daily_m.episode_m].assign(t=lambda d: d.kode_stasiun.map(st.kelompok_lokasi)).groupby("t").size().reindex(TIPE).fillna(0).astype(int)
het_b = het[het.kasus == "ISPA balita (<5 th)"].set_index("tipe")
bt["rr_balita_7hari_per10"] = het_b.rr.reindex(TIPE)
bt["skor_risiko_rata2"] = komp_f.groupby("tipe").skor.mean().reindex(TIPE); bt["skor_risiko_maks"] = komp_f.groupby("tipe").skor.max().reindex(TIPE)
bt["stasiun_risiko_tertinggi"] = [komp_f[komp_f.tipe == t].skor.idxmax() for t in TIPE]
bt["klaster_dominan"] = [kl[kl.tipe == t].nama_klaster.value_counts().index[0] for t in TIPE]
display(bt.round(2))
KN["7_bukti_tipe"] = bt.round(3).to_dict("index")

,n_stasiun,n_non_layak,pm25_rata2_harian,persen_hari_gt55,persen_hari_gt150,hari_episode,rr_balita_7hari_per10,skor_risiko_rata2,skor_risiko_maks,stasiun_risiko_tertinggi,klaster_dominan
Perkotaan,7,1,42.81,8.61,0.68,44,1.06,50.10,76.71,PSW-01,Musiman lemah · dua puncak harian kuat · tingkat sedang
Jalan Raya,2,0,36.65,17.29,0.00,0,1.05,42.21,54.89,JKT-03,Musiman lemah · dua puncak harian kuat · tingkat sedang
Industri,4,1,50.50,36.16,2.01,20,1.21,57.10,70.24,BTM-01,Kemarau tegas · dua puncak harian kuat · tingkat tinggi
Gambut,8,2,49.99,26.13,2.26,128,1.05,67.70,81.00,KTG-02,Kemarau tegas · dua puncak harian kuat · tingkat tinggi
Latar Bersih,6,1,14.17,0.12,0.04,0,1.25,35.11,44.40,KTB-01,Musiman lemah · dua puncak harian kuat · tingkat sedang


## 8. Konsistensi dengan notebook 02–04 dan penyimpanan angka kunci
Pemeriksaan: angka yang dihitung ulang di notebook ini dengan definisi identik harus sama dengan `reports/key_numbers.json` (notebook 04) dan tabel profil (notebook 02–03). Angka yang berbeda hanya boleh pada versi berlabel *bertopeng*.

In [41]:
cek = []
def tambah(nama, baru, lama, tol=1e-6):
    ok = bool(np.isclose(baru, lama, atol=tol, rtol=1e-9)); cek.append(dict(angka=nama, notebook_05=baru, acuan=lama, sama=ok))
# (a) angka 04 yang dihitung ulang apa adanya (bagian 0 sudah meng-assert beberapa)
tambah("rata-rata PM2.5 harian seluruh stasiun-hari (04.A)", r_asli["rata2_semua_hari_stasiun"], KN04["A"]["rata2_semua_hari_stasiun"])
tambah("% hari > 55 µg/m³ (04.A)", r_asli["persen_hari_lewat_bm55"], KN04["A"]["persen_hari_lewat_bm55"])
tambah("jumlah episode asap (04.B)", r_asli["n_episode"], KN04["B"]["episode"]["n_episode"]); tambah("jumlah hari episode (04.B)", r_asli["n_hari_episode"], KN04["B"]["episode"]["n_hari"])
tambah("ε² Kruskal-Wallis tipe lokasi (04.A)", r_asli["kw_tipe_epsilon2"], KN04["A"]["kruskal_wallis"]["tipe"]["epsilon2"])
# (b) korelasi lalu lintas vs median PM2.5 antar stasiun (04.C)
slh = hourly.groupby("kode_stasiun").pm25_ug_m3.median().to_frame("pm25").join(st[["traffic_intensity_per_h"]])
tambah("Spearman lalu lintas vs median PM2.5 antar-stasiun, n=26 (04.C)", stats.spearmanr(slh.traffic_intensity_per_h, slh.pm25)[0], KN04["C"]["antar_stasiun"]["pm25_ug_m3_traffic_intensity_per_h_semua"]["rho"])
# (c) rata-rata rho per-RS ISPA balita lag 0, varian A (04.D)
rows = []
for hid, x in health[health.hospital_id.isin(health.groupby("hospital_id").ada_paparan_pm25.sum().loc[lambda s: s >= 100].index)].groupby("hospital_id"):
    z = x[["pm25_lag0", "resp_admissions_under5"]].dropna()
    if len(z) >= 100: rows.append(stats.spearmanr(z.pm25_lag0, z.resp_admissions_under5.astype(float))[0])
tambah("rata-rata ρ Spearman ISPA balita lag 0, varian A (04.D)", float(np.mean(rows)), KN04["D"]["rho_A_lag_terbaik"]["ISPA balita (<5 th)"], tol=5e-4)
# (d) tren Mann-Kendall: jumlah signifikan setelah BH dengan data apa adanya
d_ok0 = daily[daily.pm25_ug_m3.notna()].copy(); d_ok0["ym"] = d_ok0.tanggal.dt.to_period("M").dt.to_timestamp()
g0 = d_ok0.groupby(["kode_stasiun", "ym"]).pm25_ug_m3.agg(["mean", "count"]); g0["mean"] = g0["mean"].where(g0["count"] >= 20); b0 = g0["mean"].unstack(0)
n0 = daily.groupby("kode_stasiun").pm25_ug_m3.count(); lay0 = [k for k in n0[n0 >= 100].index if status[k] == "Layak"]
tr0 = pd.DataFrame([eda.mk_musiman(b0[k]) for k in lay0]); tr0["p_bh"] = bh(tr0.p.values)
tambah("stasiun Layak diuji tren (04.A)", len(lay0), KN04["A"]["tren"]["n_stasiun_diuji"]); tambah("tren signifikan setelah BH (04.A)", int((tr0.p_bh < 0.05).sum()), KN04["A"]["tren"]["n_signifikan_setelah_bh"])
# (e) angka 03: jumlah hari-stasiun harian & jam
tambah("jumlah hari-stasiun PM2.5 sah dengan >= 100 hari/stasiun (04.meta)", int(daily[daily.kode_stasiun.isin(n0[n0 >= 100].index)].pm25_ug_m3.notna().sum()), KN04["meta"]["n_hari_pm25_sah"])
tambah("jumlah jam PM2.5 (04.meta)", int(hourly.pm25_ug_m3.notna().sum()), KN04["meta"]["n_jam_pm25"])
tambah("jumlah baris health_aq_daily (03)", len(health), 27486)
K_ = pd.DataFrame(cek); display(K_)
assert K_.sama.all(), K_[~K_.sama]
KN["7_konsistensi"] = {"n_pemeriksaan": int(len(K_)), "semua_sama": True, "pemeriksaan": K_.assign(notebook_05=K_.notebook_05.astype(float), acuan=K_.acuan.astype(float)).round(6).to_dict("records")}
print(f"{len(K_)}/{len(K_)} angka identik dengan notebook 02–04")

,angka,notebook_05,acuan,sama
0,rata-rata PM2.5 harian seluruh stasiun-hari (04.A),41.720629,41.720629,True
1,% hari > 55 µg/m³ (04.A),17.335149,17.335149,True
2,jumlah episode asap (04.B),21.000000,21.000000,True
3,jumlah hari episode (04.B),396.000000,396.000000,True
4,ε² Kruskal-Wallis tipe lokasi (04.A),0.436392,0.436392,True
5,"Spearman lalu lintas vs median PM2.5 antar-stasiun, n=26 (04.C)",0.765361,0.765361,True
6,"rata-rata ρ Spearman ISPA balita lag 0, varian A (04.D)",0.399914,0.399914,True
7,stasiun Layak diuji tren (04.A),22.000000,22.000000,True
8,tren signifikan setelah BH (04.A),0.000000,0.000000,True
9,jumlah hari-stasiun PM2.5 sah dengan >= 100 hari/stasiun (04.meta),24234.000000,24234.000000,True


12/12 angka identik dengan notebook 02–04


In [42]:
# angka kunci disimpan sebagai bagian 'insight' pada key_numbers.json tanpa menghapus bagian notebook 04
def _json(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return None if np.isnan(o) else float(o)
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, (pd.Timestamp,)): return str(o.date())
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)
def _bersih(o):
    if isinstance(o, dict): return {str(k): _bersih(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_bersih(v) for v in o]
    if isinstance(o, float) and (np.isnan(o) or np.isinf(o)): return None
    return o
KN["meta"] = {"gambar": GAMBAR, "sumber": "data/processed/*.parquet; notebooks/05_insights.ipynb", "data_kerja": "bertopeng: jam flatline BJB-01 dibuang (bagian 0)", "catatan": "Jalankan 05 setelah 04: notebook 04 menulis ulang berkas ini tanpa bagian 'insight'"}
asli = json.loads((cfg.REPORTS_DIR / "key_numbers.json").read_text(encoding="utf-8"))
asli["insight"] = json.loads(json.dumps(_bersih(KN), default=_json))
(cfg.REPORTS_DIR / "key_numbers.json").write_text(json.dumps(_bersih(asli), ensure_ascii=False, indent=2, default=_json), encoding="utf-8")
chk = json.loads((cfg.REPORTS_DIR / "key_numbers.json").read_text(encoding="utf-8"))
print("key_numbers.json diperbarui; bagian:", list(chk), "| bagian insight:", list(chk["insight"]), "| bagian 04 tidak berubah:", all(chk[k] == KN04[k] for k in KN04))
import os
from PIL import Image
png = [f for f in os.listdir(cfg.FIG_DIR) if f.startswith("05_") and f.endswith(".png")]
assert len(png) >= len(GAMBAR) and {Image.open(cfg.FIG_DIR / f).size[0] for f in png} == {1600}
print(f"{len(png)} PNG (1600 px) + HTML tersimpan di reports/figures/05_*")

key_numbers.json diperbarui; bagian: ['meta', 'A', 'B', 'C', 'D', 'E', 'insight'] | bagian insight: ['0_flatline', '1_ml', '2_kesehatan', '3_klaster', '4_anomali', '5_risiko', '6_jaringan', '7_bukti_tipe', '7_konsistensi', 'meta'] | bagian 04 tidak berubah: True


16 PNG (1600 px) + HTML tersimpan di reports/figures/05_*


In [43]:
KT = KN04["B"]["diurnal"]["jam_puncak_no2_median_per_tipe"]
L1, L2 = list(FOLD)
ml1, ml2 = [ML[(ML.lipatan == l) & (ML.model == "Random Forest")].iloc[0] for l in (L1, L2)]
ols1, ols2 = [ML[(ML.lipatan == l) & (ML.model == "Regresi berganda")].iloc[0] for l in (L1, L2)]
ub, ul = cum.loc[M.NAMA_KASUS[bl_a]], cum.loc[M.NAMA_KASUS[bl_b]]
tb, tl = tot.loc[M.NAMA_KASUS[bl_a]], tot.loc[M.NAMA_KASUS[bl_b]]
ekb = ER[ER.kasus == "ISPA balita (<5 th)"].set_index("kelas"); sj = sens.set_index("varian")
ex_b = EX[EX.kasus == M.NAMA_KASUS[bl_a]].set_index("kode_stasiun")
pt5 = SB["ringkasan"].peluang_top5; top5 = risiko.index[:5].tolist()
kota_ex = ", ".join(sorted(set(st.loc[ex_b.index, "kota_kabupaten"].str.replace("Kota ", "").str.replace("Kabupaten ", ""))))
kali_lo, kali_hi = tb.teramati / (tb.teramati - tb.berlebih_empiris), tb.teramati / (tb.teramati - tb.berlebih_model)
jkt02 = 100 * (dk[dk.kode_stasiun == "JKT-02"].pm25_ug_m3 > 55).mean()
status_txt = ", ".join(f"{v} {k}" for k, v in qc.status_analisis.value_counts().items())
epi_n, epi_h = r_m["n_episode"], r_m["n_hari_episode"]
nm_ = lambda xs: ", ".join(xs)
bjb_, sm2_ = hs.loc[hs.kode == "BJB-01"].iloc[0], hs.loc[hs.kode == "SM2-01"].iloc[0]
rr_b, rr_s = rr_cmp[rr_cmp.kode_stasiun == "BJB-01"].iloc[0], rr_cmp[rr_cmp.kode_stasiun == "SM2-01"].iloc[0]
tbl_dampak = pd.DataFrame({"angka": ["Rata-rata PM2.5 BJB-01 (µg/m³)", "Stasiun dengan rata-rata tertinggi", "Rata-rata PM2.5 tipe Gambut (µg/m³)", "Episode kabut asap", "Hari episode", "Hari berkategori Berbahaya", "ε² KW tipe lokasi"],
                           "04 (apa adanya)": [fl(r_asli["rata2_bjb01"]), r_asli["stasiun_tertinggi"], fl(r_asli["rata2_tipe_gambut"]), rp(r_asli["n_episode"]), rp(r_asli["n_hari_episode"]), rp(r_asli["hari_berbahaya"]), fl(r_asli["kw_tipe_epsilon2"], 3)],
                           "05 (bertopeng)": [fl(r_m["rata2_bjb01"]), r_m["stasiun_tertinggi"], fl(r_m["rata2_tipe_gambut"]), rp(r_m["n_episode"]), rp(r_m["n_hari_episode"]), rp(r_m["hari_berbahaya"]), fl(r_m["kw_tipe_epsilon2"], 3)]})
def tbl(df): return df.to_markdown(index=False, disable_numparse=True)

rek = pd.DataFrame([
    ["Gambut", f"{int(bt.loc['Gambut', 'n_stasiun'])} stasiun; PM2.5 rata-rata {fl(bt.loc['Gambut', 'pm25_rata2_harian'])} µg/m³; {fl(bt.loc['Gambut', 'persen_hari_gt55'])}% hari > 55; {int(bt.loc['Gambut', 'hari_episode'])} hari episode (terbanyak); skor risiko rata-rata {fl(bt.loc['Gambut', 'skor_risiko_rata2'])} (tertinggi); RR balita {fl(bt.loc['Gambut', 'rr_balita_7hari_per10'], 2)}/+10 (respons jenuh)",
     "BPBD/BNPB: siaga asap Agustus–Oktober; DLH: pemantauan real-time dengan alarm flatline/hari tanpa data; Dinkes & RS rujukan Kalimantan Tengah/Sumatera: siapkan lonjakan kasus balita dan lansia saat PM2.5 > 150 selama ≥ 3 hari", "05_5_skor_risiko, 05_2_kasus_berlebih_episode, 04_B5_linimasa_episode_asap"],
    ["Industri", f"{int(bt.loc['Industri', 'n_stasiun'])} stasiun; {fl(bt.loc['Industri', 'persen_hari_gt55'])}% hari > 55 (tertinggi); skor risiko rata-rata {fl(bt.loc['Industri', 'skor_risiko_rata2'])}; RR balita {fl(bt.loc['Industri', 'rr_balita_7hari_per10'], 2)}/+10; stasiun berisiko tertinggi {bt.loc['Industri', 'stasiun_risiko_tertinggi']}",
     f"DLH: audit emisi dan inspeksi kawasan sekitar JKT-02 ({fl(jkt02, 0)}% hari > 55) dan stasiun industri lain; perbaiki SM2-01 (sensor saturasi pada musim asap) sebelum menilai kepatuhan", "04_A2_hari_lewat_baku_mutu, 05_5_skor_risiko"],
    ["Perkotaan", f"{int(bt.loc['Perkotaan', 'n_stasiun'])} stasiun; PM2.5 {fl(bt.loc['Perkotaan', 'pm25_rata2_harian'])} µg/m³; {fl(bt.loc['Perkotaan', 'persen_hari_gt55'])}% hari > 55; puncak NO₂ pukul {fl(KT['Perkotaan'], 0)} lokal; stasiun berisiko tertinggi {bt.loc['Perkotaan', 'stasiun_risiko_tertinggi']} (peringkat {int(risiko.loc[bt.loc['Perkotaan', 'stasiun_risiko_tertinggi'], 'peringkat'])})",
     "Pemda/DLH: manajemen lalu lintas jam sibuk pagi dan sore; Dinkes: imbauan saat PM2.5 mencapai kelas 35–55 µg/m³ (RR ≈ " + fl(ekb.loc['35–55,4', 'rr'], 1) + "×), bukan menunggu > 150", "04_B1_profil_jam_tipe, 05_2_respons_eksposur_kelas"],
    ["Jalan Raya", f"{int(bt.loc['Jalan Raya', 'n_stasiun'])} stasiun; {fl(bt.loc['Jalan Raya', 'persen_hari_gt55'])}% hari > 55; lalu lintas vs median PM2.5 antar-stasiun ρ = {fl(KN04['C']['antar_stasiun']['pm25_ug_m3_traffic_intensity_per_h_semua']['rho'], 2)} (04); puncak NO₂ pukul {fl(KT['Jalan Raya'], 0)} lokal; RR balita {fl(bt.loc['Jalan Raya', 'rr_balita_7hari_per10'], 2)} (hanya 2 RS, tidak signifikan)",
     "DLH: tambah stasiun tepi jalan (hanya 2); Dishub: pengendalian emisi koridor padat; evaluasi dampak kesehatan menunggu lebih banyak RS pasangan", "04_C3_lalu_lintas_jarak, 04_B3_hari_kerja_akhir_pekan"],
    ["Latar Bersih", f"{int(bt.loc['Latar Bersih', 'n_stasiun'])} stasiun; PM2.5 {fl(bt.loc['Latar Bersih', 'pm25_rata2_harian'])} µg/m³ ({fl(KN04['A']['persen_hari_lewat_who15_per_tipe']['Latar Bersih'])}% hari > 15 WHO); {fl(bt.loc['Latar Bersih', 'persen_hari_gt55'])}% hari > 55; RR balita {fl(bt.loc['Latar Bersih', 'rr_balita_7hari_per10'], 2)}/+10 (tertinggi; heterogen antar-RS)",
     "KLHK: pertahankan sebagai stasiun acuan/baseline (jangan dicabut); BMKG/DLH: periksa MER-01 dan perbaiki cakupan data Papua", "04_A2_hari_lewat_baku_mutu, 05_6_peta_jaringan"],
], columns=["Tipe wilayah", "Bukti (angka)", "Rekomendasi per pemangku kepentingan", "Grafik"])

temuan = f"""# 05 — Insight Utama
> Dihasilkan otomatis oleh `notebooks/05_insights.ipynb`; seluruh angka berasal dari `data/processed/` dan identik dengan `reports/key_numbers.json` (bagian `insight`). Data kerja adalah versi **bertopeng** (flatline BJB-01 dibuang, Insight 1); angka notebook 02–04 tetap sebagai acuan "apa adanya" dan selisihnya ditampilkan eksplisit. {int(K_.shape[0])} angka acuan 04 dihitung ulang di sini dan seluruhnya identik. Hubungan kesehatan adalah **asosiasi** pada data sintetis, bukan kausal.

## Ringkasan
| # | Temuan | Bukti kunci | Terutama untuk |
|---|---|---|---|
| 1 | Sensor partikulat BJB-01 macet ±{fl(flat.n_hari.sum() / 30.4, 0)} bulan; sebagian angka notebook 04 tercemar | deret identik {', '.join(f'{rp(r.n_jam)} jam = {fl(r.nilai, 1)} µg/m³' for r in fl_b.itertuples())} vs maks {int(terpanjang.drop(flat.kode_stasiun.unique(), errors='ignore').max())} jam di stasiun lain | KLHK/DLH, tim data |
| 2 | PM2.5 ditentukan tempat dan musim, bukan cuaca harian; model gagal pada ekstrem | R² log {fl(ml1.r2_log, 2)}/{fl(ml2.r2_log, 2)}; cuaca dalam-stasiun {fl(dalam[(dalam.kelompok == G_CUACA)].acak_dalam_stasiun.max(), 2)} | KLHK/DLH |
| 3 | +10 µg/m³ PM2.5 ≈ +{fl(ub.persen_per10, 1)}% kasus; respons jenuh; balita = lansia | RR {fl(ub.rr, 3)} (CI {fl(ub.bawah, 2)}–{fl(ub.atas, 2)}); kelas 55–150: {fl(ekb.loc['55,4–150,4', 'rr'], 1)}× | Dinkes, RS |
| 4 | Episode asap menimbulkan ribuan kasus berlebih | {fl(tb.berlebih_tanpa_potong, 0)}–{fl(tb.berlebih_model, 0)} kasus ISPA balita pada {int(mask_ep.sum())} RS-hari | BNPB/BPBD, RS |
| 5 | Risiko tertinggi pada stasiun gambut Kalimantan/Sumatera; peringkat tahan bobot | {nm_(top5)}; Spearman median {fl(SB['rho_dirichlet_median'], 2)} | KLHK/DLH, BNPB |
| 6 | {len(prov_tanpa)} dari 38 provinsi tanpa stasiun; {int((qc.status_analisis != 'Layak').sum())} stasiun belum Layak | prioritas A: {nm_(A_.index[:3])}; B: {nm_(B_.index[:3])} | KLHK/DLH |
| 7 | "Polusi tinggi, kasus rendah": di BJB-01 sebagian artefak sensor, sisanya (dan SM2-01) bukan salah ukur paparan dalam-RS | RR dalam-RS normal; z residual {fl(bjb_.z_topeng, 1)}/{fl(sm2_.z_topeng, 1)} | Dinkes, RS |

## Insight 1 — Sensor BJB-01 macet; angka tercemar harus dikoreksi
- **Temuan.** BJB-01 tidak mengalami "pembalikan musim", pergeseran timestamp, maupun salah satuan: sensor partikulatnya menahan nilai tetap selama berbulan-bulan.
- **Bukti.** {len(fl_b)} deret identik: {'; '.join(f'{rp(r.n_jam)} jam ({fl(r.n_hari, 0)} hari) = {fl(r.nilai, 1)} µg/m³ ({r.awal:%d-%m-%Y}…{r.akhir:%d-%m-%Y})' for r in fl_b.itertuples())}, sd = 0, padahal stasiun lain paling banyak {int(terpanjang.drop(flat.kode_stasiun.unique(), errors='ignore').max())} jam identik. Di luar dataran, tingkat normal sama ({fl(tab.iloc[0]['median'], 1)} vs {fl(tab.iloc[1]['median'], 1)} µg/m³), korelasi dengan stasiun gambut lain {fl(r_sb, 2)} → {fl(pm_m, 2)} setelah dataran dibuang. Geseran timestamp hanya {fl(abs(geser_menit), 2)} menit; rasio PM2.5/PM10 konstan {fl(rasio.median(), 2)}; perubahan tingkat terkonsentrasi pada PM2.5/PM10 (rasio {fl(pt25.rasio_sebelum_sesudah, 2)}×) sedangkan CO, NO₂, SO₂ ≤ {fl(cp_tab.set_index('polutan').loc[['co', 'no2', 'so2'], 'rasio_sebelum_sesudah'].max(), 2)}×.
- **Dampak pada angka notebook 04.**

{tbl(tbl_dampak)}

- **Implikasi.** Episode 204 hari BJB-01 adalah artefak, bukan kabut asap; peringkat "stasiun tertinggi", komposisi ISPU, dan sebagian anomali "kasus rendah" di 04 terpengaruh. Sebab pasti (kegagalan alat vs nilai tertahan di logger) tidak dapat ditentukan dari data.
- **Rekomendasi.** (1) Tambahkan aturan flatline (≥ 24 jam identik → NaN) ke `src/clean.py` lalu jalankan ulang notebook 03–05 agar seluruh dashboard memakai data yang sama; (2) DLH: periksa/kalibrasi kanal partikulat BJB-01; (3) pantau secara rutin deret identik sebagai alarm kualitas data.

## Insight 2 — PM2.5 ditentukan oleh tempat dan musim; cuaca harian hampir tidak berperan; model tidak boleh dipakai untuk puncak asap
- **Temuan.** Variasi PM2.5 terutama berupa perbedaan level antar-stasiun dan pola musiman di dalam stasiun.
- **Bukti.** Random Forest dengan time-split: R² log {fl(ml1.r2_log, 2)} (uji 2025) dan {fl(ml2.r2_log, 2)} (uji 2026), regresi berganda {fl(ols1.r2_log, 2)}/{fl(ols2.r2_log, 2)}. Rata-rata per stasiun saja sudah R² log {fl(base.loc[L1, 'r2_log_stasiun_saja'], 2)}/{fl(base.loc[L2, 'r2_log_stasiun_saja'], 2)}; stasiun × bulan {fl(base.loc[L1, 'r2_log_stasiun_x_bulan'], 2)}/{fl(base.loc[L2, 'r2_log_stasiun_x_bulan'], 2)} (≈ model penuh). Cuaca bila diacak di dalam stasiun menurunkan R² hanya {fl(dalam[dalam.kelompok == G_CUACA].acak_dalam_stasiun.max(), 2)}, waktu/musim {fl(dalam[dalam.kelompok == G_WAKTU].acak_dalam_stasiun.min(), 2)}–{fl(dalam[dalam.kelompok == G_WAKTU].acak_dalam_stasiun.max(), 2)}. Pada uji 2026 puncak {fl(F.loc[FOLD[L2][1], 'y'].max(), 0)} µg/m³ jauh di atas maksimum latih ({fl(F.loc[FOLD[L2][0], 'y'].max(), 0)}): R² skala asli hanya {fl(ml2.r2, 2)} dan prediksi maksimum {fl(KN['1_ml']['lipatan2_maks_teramati_vs_prediksi_rf'][1], 0)} µg/m³. Klaster stasiun: {'; '.join(f"K{k} '{r.nama}' ({int(r.n)})" for k, r in ring_k.iterrows())}.
- **Implikasi.** Faktor "lalu lintas/jarak jalan" dan tipe lokasi konstan per stasiun, sehingga berfungsi sebagai pengenal stasiun, bukan bukti kausal. Dashboard sebaiknya memakai tipe lokasi dan kalender musim sebagai dimensi utama; model berbasis pohon tidak mampu memperingatkan episode di luar pengalaman data.
- **Rekomendasi.** KLHK/DLH: peringatan dini berbasis kalender musim (Agustus–Oktober) dan pemantauan langsung/titik panas, bukan prediksi ML; gunakan klaster untuk menyeragamkan standar pelaporan stasiun sejenis.

## Insight 3 — Dampak kesehatan nyata sejak kelas PM2.5 menengah, respons jenuh pada ekstrem, balita ≈ lansia
- **Temuan.** Kenaikan PM2.5 berasosiasi dengan kenaikan kasus pernapasan, kardiovaskular, dan kematian; bentuk responsnya tidak linear.
- **Bukti.** RR kumulatif lag 0–7 per +10 µg/m³ (quasi-Poisson, efek tetap RS, spline waktu, eksposur dipotong 250,4): ISPA balita {fl(ub.rr, 3)} (CI {fl(ub.bawah, 3)}–{fl(ub.atas, 3)}), ISPA lansia {fl(ul.rr, 3)} ({fl(ul.bawah, 3)}–{fl(ul.atas, 3)}); {int((cum.p < 0.05).sum())} dari {len(cum)} jenis kasus signifikan. Tahan terhadap spline waktu df 10–30 ({fl(sj.loc['Spline waktu df 10', 'balita_rr'], 3)} dan {fl(sj.loc['Spline waktu df 30', 'balita_rr'], 3)}) dan negative binomial ({fl(nb.iloc[0].rr_NB, 3)}). Tanpa pemotongan eksposur RR hanya {fl(sj.loc['Tanpa pemotongan eksposur', 'balita_rr'], 3)}. RR per kelas (vs ≤ 15,5 µg/m³): 35–55: {fl(ekb.loc['35–55,4', 'rr'], 2)}×; 55–150: {fl(ekb.loc['55,4–150,4', 'rr'], 2)}×; > 250: {fl(ekb.loc['>250,4', 'rr'], 2)}× (ekstrapolasi linear {fl(ekb.loc['>250,4', 'rr_linier_model_utama'], 2)}×). Rasio RR balita/lansia {fl(sd_cum['rasio_rr'], 4)} (CI {fl(sd_cum['rasio_bawah'], 4)}–{fl(sd_cum['rasio_atas'], 4)}).
- **Implikasi.** Efek tidak menunggu episode ekstrem; RR per +10 µg/m³ bergantung rentang eksposur dan tidak boleh diekstrapolasi. Tidak ada bukti perbedaan kerentanan balita vs lansia pada data ini (keduanya praktis identik). Bentuk jeda tidak dapat dipastikan (lag kolinear; RR lag tunggal {fl(tabel_rr.iloc[0, 0], 3)} pada lag 0 menurun landai ke {fl(tabel_rr.iloc[0, -1], 3)} pada lag 7).
- **Rekomendasi.** Dinkes: pertimbangkan imbauan kesehatan bertingkat sejak PM2.5 harian ≈ 35–55 µg/m³, bukan hanya saat kategori Tidak Sehat (perlu diuji ulang pada data riil); prioritaskan balita dan lansia setara; RS: pantau PM2.5 hingga 7 hari sebelumnya untuk penjadwalan IGD.

## Insight 4 — Episode kabut asap menimbulkan ribuan kasus berlebih
- **Temuan.** Pada musim asap Agustus–Oktober, terutama 2026, kasus ISPA di RS pasangan stasiun gambut/hutan meningkat tajam.
- **Bukti.** {epi_n} episode ({epi_h} hari; tanpa artefak BJB-01) seluruhnya pada Agu–Okt (04); {int(mask_ep.sum())} RS-hari pada 5 RS memiliki paparan lengkap. Kasus ISPA balita berlebih: {fl(tb.berlebih_tanpa_potong, 0)} (tanpa pemotongan), {fl(tb.berlebih_empiris, 0)} (selisih empiris), {fl(tb.berlebih_model, 0)} (model; jumlah batas CI per RS {fl(tb.bawah, 0)}–{fl(tb.atas, 0)}) = {fl(100 * tb.berlebih_empiris / tb.teramati, 0)}–{fl(tb.persen_dari_kasus, 0)}% dari {fl(tb.teramati, 0)} kasus teramati; ISPA lansia {fl(tl.berlebih_tanpa_potong, 0)}–{fl(tl.berlebih_model, 0)}. Per RS (balita, model): {', '.join(f'{k} {fl(r.berlebih_model, 0)} ({fl(r.persen_dari_kasus, 0)}%)' for k, r in ex_b.iterrows())}. {KN04['B']['episode']['n_berlanjut_di_akhir_data']} episode masih berjalan pada hari terakhir data (tersensor kanan).
- **Implikasi.** Beban terkonsentrasi pada beberapa RS dan dapat diperkirakan dari kalender musim dan PM2.5 beberapa hari sebelumnya; estimasi besar sangat bergantung asumsi bentuk respons (kisaran 2,4–3,4 ribu untuk balita), jadi dilaporkan sebagai rentang.
- **Rekomendasi.** BNPB/BPBD: status siaga Agustus–Oktober, posko dan masker prioritas balita/lansia di wilayah {kota_ex}; RS rujukan: rencana lonjakan kapasitas IGD/rawat inap (kasus ISPA balita selama episode ±{fl(kali_lo, 1)}–{fl(kali_hi, 1)}× tingkat dasar menurut estimasi empiris–model).

## Insight 5 — Risiko wilayah tertinggi pada stasiun gambut Kalimantan dan Sumatera; peringkat tahan terhadap bobot
- **Temuan.** Gabungan beban polusi, beban kesehatan, kerentanan, dan tren menempatkan stasiun gambut dan beberapa stasiun lintas Sumatera paling atas.
- **Bukti.** Lima teratas: {nm_(f'{k} ({fl(risiko.loc[k, "skor"], 0)})' for k in top5)}; peluang tetap di lima besar pada 5.000 bobot acak: {nm_(f'{k} {fl(100 * pt5[k], 0)}%' for k in top5)}; korelasi peringkat median {fl(SB['rho_dirichlet_median'], 2)}. Per pulau: {nm_(f'{k} {fl(v, 1)}' for k, v in wil_p.skor_rata2.items())}; per tipe skor rata-rata {nm_(f'{t} {fl(bt.loc[t, "skor_risiko_rata2"], 0)}' for t in TIPE)}. Bobot dasar: {nm_(f'{k} {fl(v, 2)}' for k, v in BOBOT.items())}.
- **Implikasi.** Skor adalah peringkat relatif, bukan risiko absolut; {int((komp_f.komponen_netral > 0).sum())} stasiun memakai nilai netral pada tren. PSW-01 (Lampung, tipe perkotaan) masuk lima besar karena beban polusi dan kesehatan tinggi, bukan karena gambut.
- **Rekomendasi.** Fokuskan sumber daya pengendalian dan kesiapsiagaan pada {nm_(top5[:3])}; tinjau ulang bobot bersama pemangku kepentingan karena ini keputusan kebijakan, bukan data.

## Insight 6 — Jaringan belum memadai secara cakupan dan keandalan
- **Temuan.** Cakupan antar-provinsi jauh dari lengkap dan sebagian stasiun belum andal, sementara setiap stasiun memberi informasi yang tidak redundan.
- **Bukti.** {len(prov_stasiun)} dari 38 provinsi memiliki stasiun (tanpa stasiun: {nm_(prov_tanpa)}); status stasiun: {status_txt}; jarak median ke stasiun terdekat {fl(nn.jarak_km.median(), 0)} km (maks {fl(nn.jarak_km.max(), 0)} km, {nn.jarak_km.idxmax()}). Korelasi harian antar-stasiun tidak meluruh dengan jarak (ρ = {fl(sp_raw[0], 2)}, {pv(sp_raw[1])}); setelah musim dibuang ≈ {fl(dc.r_anomali.median(), 2)}. Prioritas penambahan (A, sentroid kasar): {nm_(f'{k} (skor {fl(A_.loc[k, "skor"], 0)}, peluang top-5 {fl(100 * A_.loc[k, "peluang_top5"], 0)}%)' for k in A_.index[:4])}. Prioritas perbaikan (B): {nm_(f'{k} (skor {fl(B_.loc[k, "skor"], 0)}, top-3 {fl(100 * B_.loc[k, "peluang_top3"], 0)}%)' for k in B_.index[:4])}; Spearman stabilitas B median {fl(SBr['rho_dirichlet_median'], 2)} (hanya tingkat prioritas yang dapat dipercaya).
- **Implikasi.** Pada data ini tidak ada bukti stasiun saling mubazir, sehingga penambahan stasiun memberi informasi baru; JMB-02 berjarak {fl(B_.loc['JMB-02', 'isolasi_km'], 1)} km dari JMB-01 (nilai spasial perbaikannya kecil), MER-01 {fl(B_.loc['MER-01', 'isolasi_km'], 0)} km dari stasiun terdekat (paling bernilai bagi Papua bagian selatan).
- **Rekomendasi.** KLHK/DLH: perbaiki PLB-01, MER-01, dan sensor SM2-01 (aturan 18 jam memangkas musim asap); tambah stasiun pada kandidat A setelah diverifikasi dengan data penduduk, titik panas, dan sebaran gambut; jaga stasiun Latar Bersih sebagai acuan.

## Insight 7 — "Polusi tinggi, kasus rendah" di BJB-01 dan SM2-01: sebagian artefak sensor, sisanya bukan salah ukur paparan dalam-RS
- **Temuan.** Kedua RS memiliki kasus lebih sedikit daripada yang diharapkan dari PM2.5-nya. Pada BJB-01 sebagian kesenjangan berasal dari flatline (Insight 1); setelah itu dibuang, respons dalam-RS terhadap PM2.5 pada kedua RS normal sehingga tidak ada bukti salah ukur paparan yang tersisa.
- **Bukti.** Antar-RS ρ kasus–PM2.5 = {fl(stats.spearmanr(hs.kasus, hs.pm_topeng)[0], 2)}; kapasitas tempat tidur tidak memprediksi kasus (ρ = {fl(stats.spearmanr(hs.kasus, hs.kapasitas)[0], 2)}). Residual BJB-01 {fl(bjb_.z_asli, 2)} → {fl(bjb_.z_topeng, 2)} setelah flatline dibuang; SM2-01 {fl(sm2_.z_asli, 2)} → {fl(sm2_.z_topeng, 2)}. RR dalam-RS BJB-01 {fl(rr_b.rr, 3)} ({fl(rr_b.rr_bawah, 2)}–{fl(rr_b.rr_atas, 2)}), SM2-01 {fl(rr_s.rr, 3)} ({fl(rr_s.rr_bawah, 2)}–{fl(rr_s.rr_atas, 2)}) tidak berbeda dari RS lain. Korelasi harian SM2-01 dengan kasus {fl(cmp.iloc[0].spearman, 2)} (aturan 18 jam) → {fl(cmp.iloc[1].spearman, 2)} (12 jam).
- **Implikasi.** Kekurangan kasus berasal dari tingkat dasar yang rendah dan, untuk BJB-01, sebagian dari artefak sensor; data tidak membedakannya dari perbedaan pencatatan RS. Kasus rendah tidak boleh ditafsirkan sebagai wilayah aman.
- **Rekomendasi.** Dinkes: audit kelengkapan pencatatan RS-XXX-25 dan RS-XXX-26; analisis sensitivitas dengan `pm25_ug_m3_longgar` untuk SM2-01; jangan memakai jumlah kasus mentah antar-RS sebagai indikator kerentanan.

## Rekomendasi kebijakan per tipe wilayah
{tbl(rek)}

## Keterbatasan
1. **Data sintetis dan asosiasi.** Tidak ada klaim sebab-akibat; efek kesehatan hanya asosiasi dengan kontrol efek tetap RS, hari dalam minggu, suhu, kelembapan, dan spline waktu. Faktor yang tidak diamati (aktivitas, demografi, wabah) tidak dikendalikan. Rekomendasi di dokumen ini adalah usulan analitis dari data sintetis dan perlu divalidasi dengan data riil sebelum menjadi kebijakan.
2. **Bentuk respons.** RR per +10 µg/m³ sangat bergantung rentang eksposur: {fl(ub.rr, 3)} (dipotong 250,4) vs {fl(sj.loc['Tanpa pemotongan eksposur', 'balita_rr'], 3)} (tanpa pemotongan). DLM Almon berderajat 2 dengan lag kolinear tidak dapat mengidentifikasi bentuk jeda; bentuk U pada RR per lag bersifat artefak. Estimasi kasus berlebih dilaporkan sebagai rentang; jumlah batas CI per RS bukan CI gabungan.
3. **Data bertopeng.** Flatline BJB-01 dibuang pada analisis lanjutan tetapi belum pada pipeline (notebook 03); angka 02–04 tidak diubah. Penyebab flatline tidak dapat ditentukan. Stasiun lain mungkin memiliki artefak lain yang tidak terdeteksi oleh aturan ≥ 24 jam identik.
4. **Pemangkasan data musim asap.** SM2-01 dan PLB-01 kehilangan hari puncak (aturan VALID dan 18 jam), sehingga eksposur dan episode mereka bias ke bawah; MER-01 praktis tidak punya data harian; JMB-02 dikecualikan.
5. **Model PM2.5.** Berbasis pohon sehingga tidak mengekstrapolasi; fitur lokasi konstan per stasiun berfungsi sebagai pengenal stasiun; hanya 26 stasiun dan 2,8 tahun data.
6. **Kecukupan jaringan.** Daftar 38 provinsi dan sentroidnya adalah acuan eksternal (pengetahuan umum, ± ratusan km), bukan data proyek; proksi polusi/asap kandidat berasal dari 3 stasiun Layak terdekat (IDW), tanpa data penduduk, titik panas, atau sebaran gambut. Radius representatif tidak dapat diestimasi karena korelasi antar-stasiun tidak meluruh dengan jarak.
7. **Skor risiko dan prioritas.** Bobot adalah penilaian analis, bukan hasil data; sensitivitas bobot ditampilkan tetapi tidak menggantikan keputusan pemangku kepentingan. Peringkat perbaikan (n = {len(B_)}) tidak stabil. Tidak ada data penduduk/kemiskinan untuk kerentanan; proksi memakai kapasitas RS dan RR per RS.
8. **Klaster.** Silhouette Ward {fl(sil.set_index('k').loc[k_pilih, 'silhouette_ward'], 2)} pada 26 stasiun; batas klaster indikatif dan profil stasiun Terbatas kurang pasti.

## Lampiran: grafik dan konsistensi
Grafik: {', '.join('`' + g + '`' for g in GAMBAR)} (PNG 1600 px + HTML di `reports/figures/`). Pemeriksaan konsistensi: {int(K_.shape[0])}/{int(K_.shape[0])} angka acuan notebook 02–04 identik (bagian 8 notebook).
"""
(cfg.FINDINGS_DIR / "05_insights.md").write_text(temuan, encoding="utf-8")
print(f"Tersimpan: reports/findings/05_insights.md ({len(temuan.splitlines())} baris)")
display(Markdown(temuan))

Tersimpan: reports/findings/05_insights.md (89 baris)


# 05 — Insight Utama
> Dihasilkan otomatis oleh `notebooks/05_insights.ipynb`; seluruh angka berasal dari `data/processed/` dan identik dengan `reports/key_numbers.json` (bagian `insight`). Data kerja adalah versi **bertopeng** (flatline BJB-01 dibuang, Insight 1); angka notebook 02–04 tetap sebagai acuan "apa adanya" dan selisihnya ditampilkan eksplisit. 12 angka acuan 04 dihitung ulang di sini dan seluruhnya identik. Hubungan kesehatan adalah **asosiasi** pada data sintetis, bukan kausal.

## Ringkasan
| # | Temuan | Bukti kunci | Terutama untuk |
|---|---|---|---|
| 1 | Sensor partikulat BJB-01 macet ±10 bulan; sebagian angka notebook 04 tercemar | deret identik 4.886 jam = 336,7 µg/m³, 2.443 jam = 3,5 µg/m³ vs maks 4 jam di stasiun lain | KLHK/DLH, tim data |
| 2 | PM2.5 ditentukan tempat dan musim, bukan cuaca harian; model gagal pada ekstrem | R² log 0,94/0,86; cuaca dalam-stasiun 0,00 | KLHK/DLH |
| 3 | +10 µg/m³ PM2.5 ≈ +8,6% kasus; respons jenuh; balita = lansia | RR 1,086 (CI 1,05–1,12); kelas 55–150: 2,7× | Dinkes, RS |
| 4 | Episode asap menimbulkan ribuan kasus berlebih | 2.402–3.403 kasus ISPA balita pada 148 RS-hari | BNPB/BPBD, RS |
| 5 | Risiko tertinggi pada stasiun gambut Kalimantan/Sumatera; peringkat tahan bobot | KTG-02, PSW-01, PKN-01, PLB-01, PKU-01; Spearman median 0,89 | KLHK/DLH, BNPB |
| 6 | 17 dari 38 provinsi tanpa stasiun; 5 stasiun belum Layak | prioritas A: Kepulauan Bangka Belitung, Kalimantan Utara, Papua; B: MER-01, JMB-02, PLB-01 | KLHK/DLH |
| 7 | "Polusi tinggi, kasus rendah": di BJB-01 sebagian artefak sensor, sisanya (dan SM2-01) bukan salah ukur paparan dalam-RS | RR dalam-RS normal; z residual -2,8/-3,3 | Dinkes, RS |

## Insight 1 — Sensor BJB-01 macet; angka tercemar harus dikoreksi
- **Temuan.** BJB-01 tidak mengalami "pembalikan musim", pergeseran timestamp, maupun salah satuan: sensor partikulatnya menahan nilai tetap selama berbulan-bulan.
- **Bukti.** 2 deret identik: 4.886 jam (204 hari) = 336,7 µg/m³ (07-09-2024…29-03-2025); 2.443 jam (102 hari) = 3,5 µg/m³ (29-03-2025…09-07-2025), sd = 0, padahal stasiun lain paling banyak 4 jam identik. Di luar dataran, tingkat normal sama (37,0 vs 37,1 µg/m³), korelasi dengan stasiun gambut lain 0,17 → 0,77 setelah dataran dibuang. Geseran timestamp hanya 1,26 menit; rasio PM2.5/PM10 konstan 0,69; perubahan tingkat terkonsentrasi pada PM2.5/PM10 (rasio 2,21×) sedangkan CO, NO₂, SO₂ ≤ 1,08×.
- **Dampak pada angka notebook 04.**

| angka                               | 04 (apa adanya)   | 05 (bertopeng)   |
|:------------------------------------|:------------------|:-----------------|
| Rata-rata PM2.5 BJB-01 (µg/m³)      | 102,1             | 49,2             |
| Stasiun dengan rata-rata tertinggi  | BJB-01            | PLB-01           |
| Rata-rata PM2.5 tipe Gambut (µg/m³) | 57,9              | 50,0             |
| Episode kabut asap                  | 21                | 20               |
| Hari episode                        | 396               | 192              |
| Hari berkategori Berbahaya          | 301               | 98               |
| ε² KW tipe lokasi                   | 0,436             | 0,450            |

- **Implikasi.** Episode 204 hari BJB-01 adalah artefak, bukan kabut asap; peringkat "stasiun tertinggi", komposisi ISPU, dan sebagian anomali "kasus rendah" di 04 terpengaruh. Sebab pasti (kegagalan alat vs nilai tertahan di logger) tidak dapat ditentukan dari data.
- **Rekomendasi.** (1) Tambahkan aturan flatline (≥ 24 jam identik → NaN) ke `src/clean.py` lalu jalankan ulang notebook 03–05 agar seluruh dashboard memakai data yang sama; (2) DLH: periksa/kalibrasi kanal partikulat BJB-01; (3) pantau secara rutin deret identik sebagai alarm kualitas data.

## Insight 2 — PM2.5 ditentukan oleh tempat dan musim; cuaca harian hampir tidak berperan; model tidak boleh dipakai untuk puncak asap
- **Temuan.** Variasi PM2.5 terutama berupa perbedaan level antar-stasiun dan pola musiman di dalam stasiun.
- **Bukti.** Random Forest dengan time-split: R² log 0,94 (uji 2025) dan 0,86 (uji 2026), regresi berganda 0,75/0,64. Rata-rata per stasiun saja sudah R² log 0,60/0,45; stasiun × bulan 0,93/0,86 (≈ model penuh). Cuaca bila diacak di dalam stasiun menurunkan R² hanya 0,00, waktu/musim 0,54–0,64. Pada uji 2026 puncak 1.128 µg/m³ jauh di atas maksimum latih (292): R² skala asli hanya 0,26 dan prediksi maksimum 195 µg/m³. Klaster stasiun: K1 'Musiman lemah · dua puncak harian kuat · tingkat sedang' (14); K2 'Kemarau tegas · variasi harian kecil · tingkat tinggi' (2); K3 'Kemarau tegas · dua puncak harian kuat · tingkat tinggi' (10).
- **Implikasi.** Faktor "lalu lintas/jarak jalan" dan tipe lokasi konstan per stasiun, sehingga berfungsi sebagai pengenal stasiun, bukan bukti kausal. Dashboard sebaiknya memakai tipe lokasi dan kalender musim sebagai dimensi utama; model berbasis pohon tidak mampu memperingatkan episode di luar pengalaman data.
- **Rekomendasi.** KLHK/DLH: peringatan dini berbasis kalender musim (Agustus–Oktober) dan pemantauan langsung/titik panas, bukan prediksi ML; gunakan klaster untuk menyeragamkan standar pelaporan stasiun sejenis.

## Insight 3 — Dampak kesehatan nyata sejak kelas PM2.5 menengah, respons jenuh pada ekstrem, balita ≈ lansia
- **Temuan.** Kenaikan PM2.5 berasosiasi dengan kenaikan kasus pernapasan, kardiovaskular, dan kematian; bentuk responsnya tidak linear.
- **Bukti.** RR kumulatif lag 0–7 per +10 µg/m³ (quasi-Poisson, efek tetap RS, spline waktu, eksposur dipotong 250,4): ISPA balita 1,086 (CI 1,051–1,121), ISPA lansia 1,086 (1,053–1,119); 7 dari 7 jenis kasus signifikan. Tahan terhadap spline waktu df 10–30 (1,088 dan 1,087) dan negative binomial (1,091). Tanpa pemotongan eksposur RR hanya 1,021. RR per kelas (vs ≤ 15,5 µg/m³): 35–55: 1,72×; 55–150: 2,73×; > 250: 4,54× (ekstrapolasi linear 7,15×). Rasio RR balita/lansia 1,0000 (CI 0,9997–1,0003).
- **Implikasi.** Efek tidak menunggu episode ekstrem; RR per +10 µg/m³ bergantung rentang eksposur dan tidak boleh diekstrapolasi. Tidak ada bukti perbedaan kerentanan balita vs lansia pada data ini (keduanya praktis identik). Bentuk jeda tidak dapat dipastikan (lag kolinear; RR lag tunggal 1,088 pada lag 0 menurun landai ke 1,074 pada lag 7).
- **Rekomendasi.** Dinkes: pertimbangkan imbauan kesehatan bertingkat sejak PM2.5 harian ≈ 35–55 µg/m³, bukan hanya saat kategori Tidak Sehat (perlu diuji ulang pada data riil); prioritaskan balita dan lansia setara; RS: pantau PM2.5 hingga 7 hari sebelumnya untuk penjadwalan IGD.

## Insight 4 — Episode kabut asap menimbulkan ribuan kasus berlebih
- **Temuan.** Pada musim asap Agustus–Oktober, terutama 2026, kasus ISPA di RS pasangan stasiun gambut/hutan meningkat tajam.
- **Bukti.** 20 episode (192 hari; tanpa artefak BJB-01) seluruhnya pada Agu–Okt (04); 148 RS-hari pada 5 RS memiliki paparan lengkap. Kasus ISPA balita berlebih: 2.402 (tanpa pemotongan), 2.864 (selisih empiris), 3.403 (model; jumlah batas CI per RS 2.640–3.857) = 58–69% dari 4.957 kasus teramati; ISPA lansia 2.776–3.821. Per RS (balita, model): KTG-01 1.113 (80%), KTG-02 1.119 (80%), STG-01 161 (74%), PKN-01 350 (77%), PLB-01 660 (45%). 5 episode masih berjalan pada hari terakhir data (tersensor kanan).
- **Implikasi.** Beban terkonsentrasi pada beberapa RS dan dapat diperkirakan dari kalender musim dan PM2.5 beberapa hari sebelumnya; estimasi besar sangat bergantung asumsi bentuk respons (kisaran 2,4–3,4 ribu untuk balita), jadi dilaporkan sebagai rentang.
- **Rekomendasi.** BNPB/BPBD: status siaga Agustus–Oktober, posko dan masker prioritas balita/lansia di wilayah Kotawaringin Barat, Kotawaringin Timur, Palangka Raya, Palembang, Sintang; RS rujukan: rencana lonjakan kapasitas IGD/rawat inap (kasus ISPA balita selama episode ±2,4–3,2× tingkat dasar menurut estimasi empiris–model).

## Insight 5 — Risiko wilayah tertinggi pada stasiun gambut Kalimantan dan Sumatera; peringkat tahan terhadap bobot
- **Temuan.** Gabungan beban polusi, beban kesehatan, kerentanan, dan tren menempatkan stasiun gambut dan beberapa stasiun lintas Sumatera paling atas.
- **Bukti.** Lima teratas: KTG-02 (81), PSW-01 (77), PKN-01 (74), PLB-01 (72), PKU-01 (71); peluang tetap di lima besar pada 5.000 bobot acak: KTG-02 98%, PSW-01 80%, PKN-01 93%, PLB-01 42%, PKU-01 67%; korelasi peringkat median 0,89. Per pulau: Sumatera 55,4, Jawa 45,0, Kalimantan 63,6, Sulawesi 35,3, Papua 41,0; per tipe skor rata-rata Perkotaan 50, Jalan Raya 42, Industri 57, Gambut 68, Latar Bersih 35. Bobot dasar: polusi 0,35, kesehatan 0,25, kerentanan 0,25, tren 0,15.
- **Implikasi.** Skor adalah peringkat relatif, bukan risiko absolut; 5 stasiun memakai nilai netral pada tren. PSW-01 (Lampung, tipe perkotaan) masuk lima besar karena beban polusi dan kesehatan tinggi, bukan karena gambut.
- **Rekomendasi.** Fokuskan sumber daya pengendalian dan kesiapsiagaan pada KTG-02, PSW-01, PKN-01; tinjau ulang bobot bersama pemangku kepentingan karena ini keputusan kebijakan, bukan data.

## Insight 6 — Jaringan belum memadai secara cakupan dan keandalan
- **Temuan.** Cakupan antar-provinsi jauh dari lengkap dan sebagian stasiun belum andal, sementara setiap stasiun memberi informasi yang tidak redundan.
- **Bukti.** 21 dari 38 provinsi memiliki stasiun (tanpa stasiun: Kepulauan Bangka Belitung, Banten, Jawa Tengah, Bali, Nusa Tenggara Barat, Nusa Tenggara Timur, Kalimantan Utara, Sulawesi Utara, Gorontalo, Sulawesi Barat, Sulawesi Tenggara, Maluku, Maluku Utara, Papua Barat, Papua, Papua Tengah, Papua Pegunungan); status stasiun: 22 Layak, 4 Terbatas, 1 Dikecualikan; jarak median ke stasiun terdekat 186 km (maks 1.319 km, MER-01). Korelasi harian antar-stasiun tidak meluruh dengan jarak (ρ = -0,04, p = 0,580); setelah musim dibuang ≈ 0,03. Prioritas penambahan (A, sentroid kasar): Kepulauan Bangka Belitung (skor 75, peluang top-5 97%), Kalimantan Utara (skor 54, peluang top-5 100%), Papua (skor 46, peluang top-5 77%), Papua Pegunungan (skor 45, peluang top-5 68%). Prioritas perbaikan (B): MER-01 (skor 53, top-3 58%), JMB-02 (skor 53, top-3 74%), PLB-01 (skor 50, top-3 89%), SM2-01 (skor 39, top-3 40%); Spearman stabilitas B median 0,54 (hanya tingkat prioritas yang dapat dipercaya).
- **Implikasi.** Pada data ini tidak ada bukti stasiun saling mubazir, sehingga penambahan stasiun memberi informasi baru; JMB-02 berjarak 0,9 km dari JMB-01 (nilai spasial perbaikannya kecil), MER-01 1.319 km dari stasiun terdekat (paling bernilai bagi Papua bagian selatan).
- **Rekomendasi.** KLHK/DLH: perbaiki PLB-01, MER-01, dan sensor SM2-01 (aturan 18 jam memangkas musim asap); tambah stasiun pada kandidat A setelah diverifikasi dengan data penduduk, titik panas, dan sebaran gambut; jaga stasiun Latar Bersih sebagai acuan.

## Insight 7 — "Polusi tinggi, kasus rendah" di BJB-01 dan SM2-01: sebagian artefak sensor, sisanya bukan salah ukur paparan dalam-RS
- **Temuan.** Kedua RS memiliki kasus lebih sedikit daripada yang diharapkan dari PM2.5-nya. Pada BJB-01 sebagian kesenjangan berasal dari flatline (Insight 1); setelah itu dibuang, respons dalam-RS terhadap PM2.5 pada kedua RS normal sehingga tidak ada bukti salah ukur paparan yang tersisa.
- **Bukti.** Antar-RS ρ kasus–PM2.5 = 0,73; kapasitas tempat tidur tidak memprediksi kasus (ρ = 0,18). Residual BJB-01 -4,47 → -2,79 setelah flatline dibuang; SM2-01 -2,25 → -3,25. RR dalam-RS BJB-01 1,116 (1,07–1,16), SM2-01 1,098 (0,84–1,43) tidak berbeda dari RS lain. Korelasi harian SM2-01 dengan kasus 0,24 (aturan 18 jam) → 0,45 (12 jam).
- **Implikasi.** Kekurangan kasus berasal dari tingkat dasar yang rendah dan, untuk BJB-01, sebagian dari artefak sensor; data tidak membedakannya dari perbedaan pencatatan RS. Kasus rendah tidak boleh ditafsirkan sebagai wilayah aman.
- **Rekomendasi.** Dinkes: audit kelengkapan pencatatan RS-XXX-25 dan RS-XXX-26; analisis sensitivitas dengan `pm25_ug_m3_longgar` untuk SM2-01; jangan memakai jumlah kasus mentah antar-RS sebagai indikator kerentanan.

## Rekomendasi kebijakan per tipe wilayah
| Tipe wilayah   | Bukti (angka)                                                                                                                                                    | Rekomendasi per pemangku kepentingan                                                                                                                                                                                              | Grafik                                                                     |
|:---------------|:-----------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:---------------------------------------------------------------------------|
| Gambut         | 8 stasiun; PM2.5 rata-rata 50,0 µg/m³; 26,1% hari > 55; 128 hari episode (terbanyak); skor risiko rata-rata 67,7 (tertinggi); RR balita 1,05/+10 (respons jenuh) | BPBD/BNPB: siaga asap Agustus–Oktober; DLH: pemantauan real-time dengan alarm flatline/hari tanpa data; Dinkes & RS rujukan Kalimantan Tengah/Sumatera: siapkan lonjakan kasus balita dan lansia saat PM2.5 > 150 selama ≥ 3 hari | 05_5_skor_risiko, 05_2_kasus_berlebih_episode, 04_B5_linimasa_episode_asap |
| Industri       | 4 stasiun; 36,2% hari > 55 (tertinggi); skor risiko rata-rata 57,1; RR balita 1,21/+10; stasiun berisiko tertinggi BTM-01                                        | DLH: audit emisi dan inspeksi kawasan sekitar JKT-02 (100% hari > 55) dan stasiun industri lain; perbaiki SM2-01 (sensor saturasi pada musim asap) sebelum menilai kepatuhan                                                      | 04_A2_hari_lewat_baku_mutu, 05_5_skor_risiko                               |
| Perkotaan      | 7 stasiun; PM2.5 42,8 µg/m³; 8,6% hari > 55; puncak NO₂ pukul 8 lokal; stasiun berisiko tertinggi PSW-01 (peringkat 2)                                           | Pemda/DLH: manajemen lalu lintas jam sibuk pagi dan sore; Dinkes: imbauan saat PM2.5 mencapai kelas 35–55 µg/m³ (RR ≈ 1,7×), bukan menunggu > 150                                                                                 | 04_B1_profil_jam_tipe, 05_2_respons_eksposur_kelas                         |
| Jalan Raya     | 2 stasiun; 17,3% hari > 55; lalu lintas vs median PM2.5 antar-stasiun ρ = 0,77 (04); puncak NO₂ pukul 17 lokal; RR balita 1,05 (hanya 2 RS, tidak signifikan)    | DLH: tambah stasiun tepi jalan (hanya 2); Dishub: pengendalian emisi koridor padat; evaluasi dampak kesehatan menunggu lebih banyak RS pasangan                                                                                   | 04_C3_lalu_lintas_jarak, 04_B3_hari_kerja_akhir_pekan                      |
| Latar Bersih   | 6 stasiun; PM2.5 14,2 µg/m³ (19,2% hari > 15 WHO); 0,1% hari > 55; RR balita 1,25/+10 (tertinggi; heterogen antar-RS)                                            | KLHK: pertahankan sebagai stasiun acuan/baseline (jangan dicabut); BMKG/DLH: periksa MER-01 dan perbaiki cakupan data Papua                                                                                                       | 04_A2_hari_lewat_baku_mutu, 05_6_peta_jaringan                             |

## Keterbatasan
1. **Data sintetis dan asosiasi.** Tidak ada klaim sebab-akibat; efek kesehatan hanya asosiasi dengan kontrol efek tetap RS, hari dalam minggu, suhu, kelembapan, dan spline waktu. Faktor yang tidak diamati (aktivitas, demografi, wabah) tidak dikendalikan. Rekomendasi di dokumen ini adalah usulan analitis dari data sintetis dan perlu divalidasi dengan data riil sebelum menjadi kebijakan.
2. **Bentuk respons.** RR per +10 µg/m³ sangat bergantung rentang eksposur: 1,086 (dipotong 250,4) vs 1,021 (tanpa pemotongan). DLM Almon berderajat 2 dengan lag kolinear tidak dapat mengidentifikasi bentuk jeda; bentuk U pada RR per lag bersifat artefak. Estimasi kasus berlebih dilaporkan sebagai rentang; jumlah batas CI per RS bukan CI gabungan.
3. **Data bertopeng.** Flatline BJB-01 dibuang pada analisis lanjutan tetapi belum pada pipeline (notebook 03); angka 02–04 tidak diubah. Penyebab flatline tidak dapat ditentukan. Stasiun lain mungkin memiliki artefak lain yang tidak terdeteksi oleh aturan ≥ 24 jam identik.
4. **Pemangkasan data musim asap.** SM2-01 dan PLB-01 kehilangan hari puncak (aturan VALID dan 18 jam), sehingga eksposur dan episode mereka bias ke bawah; MER-01 praktis tidak punya data harian; JMB-02 dikecualikan.
5. **Model PM2.5.** Berbasis pohon sehingga tidak mengekstrapolasi; fitur lokasi konstan per stasiun berfungsi sebagai pengenal stasiun; hanya 26 stasiun dan 2,8 tahun data.
6. **Kecukupan jaringan.** Daftar 38 provinsi dan sentroidnya adalah acuan eksternal (pengetahuan umum, ± ratusan km), bukan data proyek; proksi polusi/asap kandidat berasal dari 3 stasiun Layak terdekat (IDW), tanpa data penduduk, titik panas, atau sebaran gambut. Radius representatif tidak dapat diestimasi karena korelasi antar-stasiun tidak meluruh dengan jarak.
7. **Skor risiko dan prioritas.** Bobot adalah penilaian analis, bukan hasil data; sensitivitas bobot ditampilkan tetapi tidak menggantikan keputusan pemangku kepentingan. Peringkat perbaikan (n = 6) tidak stabil. Tidak ada data penduduk/kemiskinan untuk kerentanan; proksi memakai kapasitas RS dan RR per RS.
8. **Klaster.** Silhouette Ward 0,53 pada 26 stasiun; batas klaster indikatif dan profil stasiun Terbatas kurang pasti.

## Lampiran: grafik dan konsistensi
Grafik: `05_0_flatline_per_stasiun`, `05_1_kinerja_model`, `05_1_permutation_importance`, `05_1_shap`, `05_1_prediksi_vs_teramati`, `05_2_rr_lag_dan_kumulatif`, `05_2_respons_eksposur_kelas`, `05_2_kasus_berlebih_episode`, `05_3_klaster_dendrogram_profil`, `05_3_klaster_pca`, `05_4_bjb01_flatline`, `05_4_polusi_tinggi_kasus_rendah`, `05_5_skor_risiko`, `05_6_redundansi_kelengkapan`, `05_6_peta_jaringan`, `05_6_skor_prioritas` (PNG 1600 px + HTML di `reports/figures/`). Pemeriksaan konsistensi: 12/12 angka acuan notebook 02–04 identik (bagian 8 notebook).


# 05 — Insight Utama
> Dihasilkan otomatis oleh `notebooks/05_insights.ipynb`; seluruh angka berasal dari `data/processed/` dan identik dengan `reports/key_numbers.json` (bagian `insight`). Data kerja adalah versi **bertopeng** (flatline BJB-01 dibuang, Insight 1); angka notebook 02–04 tetap sebagai acuan "apa adanya" dan selisihnya ditampilkan eksplisit. 12 angka acuan 04 dihitung ulang di sini dan seluruhnya identik. Hubungan kesehatan adalah **asosiasi** pada data sintetis, bukan kausal.

## Ringkasan
| # | Temuan | Bukti kunci | Terutama untuk |
|---|---|---|---|
| 1 | Sensor partikulat BJB-01 macet ±10 bulan; sebagian angka notebook 04 tercemar | deret identik 4.886 jam = 336,7 µg/m³, 2.443 jam = 3,5 µg/m³ vs maks 4 jam di stasiun lain | KLHK/DLH, tim data |
| 2 | PM2.5 ditentukan tempat dan musim, bukan cuaca harian; model gagal pada ekstrem | R² log 0,94/0,86; cuaca dalam-stasiun 0,00 | KLHK/DLH |
| 3 | +10 µg/m³ PM2.5 ≈ +8,6% kasus; respons jenuh; balita = lansia | RR 1,086 (CI 1,05–1,12); kelas 55–150: 2,7× | Dinkes, RS |
| 4 | Episode asap menimbulkan ribuan kasus berlebih | 2.402–3.403 kasus ISPA balita pada 148 RS-hari | BNPB/BPBD, RS |
| 5 | Risiko tertinggi pada stasiun gambut Kalimantan/Sumatera; peringkat tahan bobot | KTG-02, PSW-01, PKN-01, PLB-01, PKU-01; Spearman median 0,89 | KLHK/DLH, BNPB |
| 6 | 17 dari 38 provinsi tanpa stasiun; 5 stasiun belum Layak | prioritas A: Kepulauan Bangka Belitung, Kalimantan Utara, Papua; B: MER-01, JMB-02, PLB-01 | KLHK/DLH |
| 7 | "Polusi tinggi, kasus rendah": di BJB-01 sebagian artefak sensor, sisanya (dan SM2-01) bukan salah ukur paparan dalam-RS | RR dalam-RS normal; z residual -2,8/-3,3 | Dinkes, RS |

## Insight 1 — Sensor BJB-01 macet; angka tercemar harus dikoreksi
- **Temuan.** BJB-01 tidak mengalami "pembalikan musim", pergeseran timestamp, maupun salah satuan: sensor partikulatnya menahan nilai tetap selama berbulan-bulan.
- **Bukti.** 2 deret identik: 4.886 jam (204 hari) = 336,7 µg/m³ (07-09-2024…29-03-2025); 2.443 jam (102 hari) = 3,5 µg/m³ (29-03-2025…09-07-2025), sd = 0, padahal stasiun lain paling banyak 4 jam identik. Di luar dataran, tingkat normal sama (37,0 vs 37,1 µg/m³), korelasi dengan stasiun gambut lain 0,17 → 0,77 setelah dataran dibuang. Geseran timestamp hanya 1,26 menit; rasio PM2.5/PM10 konstan 0,69; perubahan tingkat terkonsentrasi pada PM2.5/PM10 (rasio 2,21×) sedangkan CO, NO₂, SO₂ ≤ 1,08×.
- **Dampak pada angka notebook 04.**

| angka                               | 04 (apa adanya)   | 05 (bertopeng)   |
|:------------------------------------|:------------------|:-----------------|
| Rata-rata PM2.5 BJB-01 (µg/m³)      | 102,1             | 49,2             |
| Stasiun dengan rata-rata tertinggi  | BJB-01            | PLB-01           |
| Rata-rata PM2.5 tipe Gambut (µg/m³) | 57,9              | 50,0             |
| Episode kabut asap                  | 21                | 20               |
| Hari episode                        | 396               | 192              |
| Hari berkategori Berbahaya          | 301               | 98               |
| ε² KW tipe lokasi                   | 0,436             | 0,450            |

- **Implikasi.** Episode 204 hari BJB-01 adalah artefak, bukan kabut asap; peringkat "stasiun tertinggi", komposisi ISPU, dan sebagian anomali "kasus rendah" di 04 terpengaruh. Sebab pasti (kegagalan alat vs nilai tertahan di logger) tidak dapat ditentukan dari data.
- **Rekomendasi.** (1) Tambahkan aturan flatline (≥ 24 jam identik → NaN) ke `src/clean.py` lalu jalankan ulang notebook 03–05 agar seluruh dashboard memakai data yang sama; (2) DLH: periksa/kalibrasi kanal partikulat BJB-01; (3) pantau secara rutin deret identik sebagai alarm kualitas data.

## Insight 2 — PM2.5 ditentukan oleh tempat dan musim; cuaca harian hampir tidak berperan; model tidak boleh dipakai untuk puncak asap
- **Temuan.** Variasi PM2.5 terutama berupa perbedaan level antar-stasiun dan pola musiman di dalam stasiun.
- **Bukti.** Random Forest dengan time-split: R² log 0,94 (uji 2025) dan 0,86 (uji 2026), regresi berganda 0,75/0,64. Rata-rata per stasiun saja sudah R² log 0,60/0,45; stasiun × bulan 0,93/0,86 (≈ model penuh). Cuaca bila diacak di dalam stasiun menurunkan R² hanya 0,00, waktu/musim 0,54–0,64. Pada uji 2026 puncak 1.128 µg/m³ jauh di atas maksimum latih (292): R² skala asli hanya 0,26 dan prediksi maksimum 195 µg/m³. Klaster stasiun: K1 'Musiman lemah · dua puncak harian kuat · tingkat sedang' (14); K2 'Kemarau tegas · variasi harian kecil · tingkat tinggi' (2); K3 'Kemarau tegas · dua puncak harian kuat · tingkat tinggi' (10).
- **Implikasi.** Faktor "lalu lintas/jarak jalan" dan tipe lokasi konstan per stasiun, sehingga berfungsi sebagai pengenal stasiun, bukan bukti kausal. Dashboard sebaiknya memakai tipe lokasi dan kalender musim sebagai dimensi utama; model berbasis pohon tidak mampu memperingatkan episode di luar pengalaman data.
- **Rekomendasi.** KLHK/DLH: peringatan dini berbasis kalender musim (Agustus–Oktober) dan pemantauan langsung/titik panas, bukan prediksi ML; gunakan klaster untuk menyeragamkan standar pelaporan stasiun sejenis.

## Insight 3 — Dampak kesehatan nyata sejak kelas PM2.5 menengah, respons jenuh pada ekstrem, balita ≈ lansia
- **Temuan.** Kenaikan PM2.5 berasosiasi dengan kenaikan kasus pernapasan, kardiovaskular, dan kematian; bentuk responsnya tidak linear.
- **Bukti.** RR kumulatif lag 0–7 per +10 µg/m³ (quasi-Poisson, efek tetap RS, spline waktu, eksposur dipotong 250,4): ISPA balita 1,086 (CI 1,051–1,121), ISPA lansia 1,086 (1,053–1,119); 7 dari 7 jenis kasus signifikan. Tahan terhadap spline waktu df 10–30 (1,088 dan 1,087) dan negative binomial (1,091). Tanpa pemotongan eksposur RR hanya 1,021. RR per kelas (vs ≤ 15,5 µg/m³): 35–55: 1,72×; 55–150: 2,73×; > 250: 4,54× (ekstrapolasi linear 7,15×). Rasio RR balita/lansia 1,0000 (CI 0,9997–1,0003).
- **Implikasi.** Efek tidak menunggu episode ekstrem; RR per +10 µg/m³ bergantung rentang eksposur dan tidak boleh diekstrapolasi. Tidak ada bukti perbedaan kerentanan balita vs lansia pada data ini (keduanya praktis identik). Bentuk jeda tidak dapat dipastikan (lag kolinear; RR lag tunggal 1,088 pada lag 0 menurun landai ke 1,074 pada lag 7).
- **Rekomendasi.** Dinkes: pertimbangkan imbauan kesehatan bertingkat sejak PM2.5 harian ≈ 35–55 µg/m³, bukan hanya saat kategori Tidak Sehat (perlu diuji ulang pada data riil); prioritaskan balita dan lansia setara; RS: pantau PM2.5 hingga 7 hari sebelumnya untuk penjadwalan IGD.

## Insight 4 — Episode kabut asap menimbulkan ribuan kasus berlebih
- **Temuan.** Pada musim asap Agustus–Oktober, terutama 2026, kasus ISPA di RS pasangan stasiun gambut/hutan meningkat tajam.
- **Bukti.** 20 episode (192 hari; tanpa artefak BJB-01) seluruhnya pada Agu–Okt (04); 148 RS-hari pada 5 RS memiliki paparan lengkap. Kasus ISPA balita berlebih: 2.402 (tanpa pemotongan), 2.864 (selisih empiris), 3.403 (model; jumlah batas CI per RS 2.640–3.857) = 58–69% dari 4.957 kasus teramati; ISPA lansia 2.776–3.821. Per RS (balita, model): KTG-01 1.113 (80%), KTG-02 1.119 (80%), STG-01 161 (74%), PKN-01 350 (77%), PLB-01 660 (45%). 5 episode masih berjalan pada hari terakhir data (tersensor kanan).
- **Implikasi.** Beban terkonsentrasi pada beberapa RS dan dapat diperkirakan dari kalender musim dan PM2.5 beberapa hari sebelumnya; estimasi besar sangat bergantung asumsi bentuk respons (kisaran 2,4–3,4 ribu untuk balita), jadi dilaporkan sebagai rentang.
- **Rekomendasi.** BNPB/BPBD: status siaga Agustus–Oktober, posko dan masker prioritas balita/lansia di wilayah Kotawaringin Barat, Kotawaringin Timur, Palangka Raya, Palembang, Sintang; RS rujukan: rencana lonjakan kapasitas IGD/rawat inap (kasus ISPA balita selama episode ±2,4–3,2× tingkat dasar menurut estimasi empiris–model).

## Insight 5 — Risiko wilayah tertinggi pada stasiun gambut Kalimantan dan Sumatera; peringkat tahan terhadap bobot
- **Temuan.** Gabungan beban polusi, beban kesehatan, kerentanan, dan tren menempatkan stasiun gambut dan beberapa stasiun lintas Sumatera paling atas.
- **Bukti.** Lima teratas: KTG-02 (81), PSW-01 (77), PKN-01 (74), PLB-01 (72), PKU-01 (71); peluang tetap di lima besar pada 5.000 bobot acak: KTG-02 98%, PSW-01 80%, PKN-01 93%, PLB-01 42%, PKU-01 67%; korelasi peringkat median 0,89. Per pulau: Sumatera 55,4, Jawa 45,0, Kalimantan 63,6, Sulawesi 35,3, Papua 41,0; per tipe skor rata-rata Perkotaan 50, Jalan Raya 42, Industri 57, Gambut 68, Latar Bersih 35. Bobot dasar: polusi 0,35, kesehatan 0,25, kerentanan 0,25, tren 0,15.
- **Implikasi.** Skor adalah peringkat relatif, bukan risiko absolut; 5 stasiun memakai nilai netral pada tren. PSW-01 (Lampung, tipe perkotaan) masuk lima besar karena beban polusi dan kesehatan tinggi, bukan karena gambut.
- **Rekomendasi.** Fokuskan sumber daya pengendalian dan kesiapsiagaan pada KTG-02, PSW-01, PKN-01; tinjau ulang bobot bersama pemangku kepentingan karena ini keputusan kebijakan, bukan data.

## Insight 6 — Jaringan belum memadai secara cakupan dan keandalan
- **Temuan.** Cakupan antar-provinsi jauh dari lengkap dan sebagian stasiun belum andal, sementara setiap stasiun memberi informasi yang tidak redundan.
- **Bukti.** 21 dari 38 provinsi memiliki stasiun (tanpa stasiun: Kepulauan Bangka Belitung, Banten, Jawa Tengah, Bali, Nusa Tenggara Barat, Nusa Tenggara Timur, Kalimantan Utara, Sulawesi Utara, Gorontalo, Sulawesi Barat, Sulawesi Tenggara, Maluku, Maluku Utara, Papua Barat, Papua, Papua Tengah, Papua Pegunungan); status stasiun: 22 Layak, 4 Terbatas, 1 Dikecualikan; jarak median ke stasiun terdekat 186 km (maks 1.319 km, MER-01). Korelasi harian antar-stasiun tidak meluruh dengan jarak (ρ = -0,04, p = 0,580); setelah musim dibuang ≈ 0,03. Prioritas penambahan (A, sentroid kasar): Kepulauan Bangka Belitung (skor 75, peluang top-5 97%), Kalimantan Utara (skor 54, peluang top-5 100%), Papua (skor 46, peluang top-5 77%), Papua Pegunungan (skor 45, peluang top-5 68%). Prioritas perbaikan (B): MER-01 (skor 53, top-3 58%), JMB-02 (skor 53, top-3 74%), PLB-01 (skor 50, top-3 89%), SM2-01 (skor 39, top-3 40%); Spearman stabilitas B median 0,54 (hanya tingkat prioritas yang dapat dipercaya).
- **Implikasi.** Pada data ini tidak ada bukti stasiun saling mubazir, sehingga penambahan stasiun memberi informasi baru; JMB-02 berjarak 0,9 km dari JMB-01 (nilai spasial perbaikannya kecil), MER-01 1.319 km dari stasiun terdekat (paling bernilai bagi Papua bagian selatan).
- **Rekomendasi.** KLHK/DLH: perbaiki PLB-01, MER-01, dan sensor SM2-01 (aturan 18 jam memangkas musim asap); tambah stasiun pada kandidat A setelah diverifikasi dengan data penduduk, titik panas, dan sebaran gambut; jaga stasiun Latar Bersih sebagai acuan.

## Insight 7 — "Polusi tinggi, kasus rendah" di BJB-01 dan SM2-01: sebagian artefak sensor, sisanya bukan salah ukur paparan dalam-RS
- **Temuan.** Kedua RS memiliki kasus lebih sedikit daripada yang diharapkan dari PM2.5-nya. Pada BJB-01 sebagian kesenjangan berasal dari flatline (Insight 1); setelah itu dibuang, respons dalam-RS terhadap PM2.5 pada kedua RS normal sehingga tidak ada bukti salah ukur paparan yang tersisa.
- **Bukti.** Antar-RS ρ kasus–PM2.5 = 0,73; kapasitas tempat tidur tidak memprediksi kasus (ρ = 0,18). Residual BJB-01 -4,47 → -2,79 setelah flatline dibuang; SM2-01 -2,25 → -3,25. RR dalam-RS BJB-01 1,116 (1,07–1,16), SM2-01 1,098 (0,84–1,43) tidak berbeda dari RS lain. Korelasi harian SM2-01 dengan kasus 0,24 (aturan 18 jam) → 0,45 (12 jam).
- **Implikasi.** Kekurangan kasus berasal dari tingkat dasar yang rendah dan, untuk BJB-01, sebagian dari artefak sensor; data tidak membedakannya dari perbedaan pencatatan RS. Kasus rendah tidak boleh ditafsirkan sebagai wilayah aman.
- **Rekomendasi.** Dinkes: audit kelengkapan pencatatan RS-XXX-25 dan RS-XXX-26; analisis sensitivitas dengan `pm25_ug_m3_longgar` untuk SM2-01; jangan memakai jumlah kasus mentah antar-RS sebagai indikator kerentanan.

## Rekomendasi kebijakan per tipe wilayah
| Tipe wilayah   | Bukti (angka)                                                                                                                                                    | Rekomendasi per pemangku kepentingan                                                                                                                                                                                              | Grafik                                                                     |
|:---------------|:-----------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:---------------------------------------------------------------------------|
| Gambut         | 8 stasiun; PM2.5 rata-rata 50,0 µg/m³; 26,1% hari > 55; 128 hari episode (terbanyak); skor risiko rata-rata 67,7 (tertinggi); RR balita 1,05/+10 (respons jenuh) | BPBD/BNPB: siaga asap Agustus–Oktober; DLH: pemantauan real-time dengan alarm flatline/hari tanpa data; Dinkes & RS rujukan Kalimantan Tengah/Sumatera: siapkan lonjakan kasus balita dan lansia saat PM2.5 > 150 selama ≥ 3 hari | 05_5_skor_risiko, 05_2_kasus_berlebih_episode, 04_B5_linimasa_episode_asap |
| Industri       | 4 stasiun; 36,2% hari > 55 (tertinggi); skor risiko rata-rata 57,1; RR balita 1,21/+10; stasiun berisiko tertinggi BTM-01                                        | DLH: audit emisi dan inspeksi kawasan sekitar JKT-02 (100% hari > 55) dan stasiun industri lain; perbaiki SM2-01 (sensor saturasi pada musim asap) sebelum menilai kepatuhan                                                      | 04_A2_hari_lewat_baku_mutu, 05_5_skor_risiko                               |
| Perkotaan      | 7 stasiun; PM2.5 42,8 µg/m³; 8,6% hari > 55; puncak NO₂ pukul 8 lokal; stasiun berisiko tertinggi PSW-01 (peringkat 2)                                           | Pemda/DLH: manajemen lalu lintas jam sibuk pagi dan sore; Dinkes: imbauan saat PM2.5 mencapai kelas 35–55 µg/m³ (RR ≈ 1,7×), bukan menunggu > 150                                                                                 | 04_B1_profil_jam_tipe, 05_2_respons_eksposur_kelas                         |
| Jalan Raya     | 2 stasiun; 17,3% hari > 55; lalu lintas vs median PM2.5 antar-stasiun ρ = 0,77 (04); puncak NO₂ pukul 17 lokal; RR balita 1,05 (hanya 2 RS, tidak signifikan)    | DLH: tambah stasiun tepi jalan (hanya 2); Dishub: pengendalian emisi koridor padat; evaluasi dampak kesehatan menunggu lebih banyak RS pasangan                                                                                   | 04_C3_lalu_lintas_jarak, 04_B3_hari_kerja_akhir_pekan                      |
| Latar Bersih   | 6 stasiun; PM2.5 14,2 µg/m³ (19,2% hari > 15 WHO); 0,1% hari > 55; RR balita 1,25/+10 (tertinggi; heterogen antar-RS)                                            | KLHK: pertahankan sebagai stasiun acuan/baseline (jangan dicabut); BMKG/DLH: periksa MER-01 dan perbaiki cakupan data Papua                                                                                                       | 04_A2_hari_lewat_baku_mutu, 05_6_peta_jaringan                             |

## Keterbatasan
1. **Data sintetis dan asosiasi.** Tidak ada klaim sebab-akibat; efek kesehatan hanya asosiasi dengan kontrol efek tetap RS, hari dalam minggu, suhu, kelembapan, dan spline waktu. Faktor yang tidak diamati (aktivitas, demografi, wabah) tidak dikendalikan. Rekomendasi di dokumen ini adalah usulan analitis dari data sintetis dan perlu divalidasi dengan data riil sebelum menjadi kebijakan.
2. **Bentuk respons.** RR per +10 µg/m³ sangat bergantung rentang eksposur: 1,086 (dipotong 250,4) vs 1,021 (tanpa pemotongan). DLM Almon berderajat 2 dengan lag kolinear tidak dapat mengidentifikasi bentuk jeda; bentuk U pada RR per lag bersifat artefak. Estimasi kasus berlebih dilaporkan sebagai rentang; jumlah batas CI per RS bukan CI gabungan.
3. **Data bertopeng.** Flatline BJB-01 dibuang pada analisis lanjutan tetapi belum pada pipeline (notebook 03); angka 02–04 tidak diubah. Penyebab flatline tidak dapat ditentukan. Stasiun lain mungkin memiliki artefak lain yang tidak terdeteksi oleh aturan ≥ 24 jam identik.
4. **Pemangkasan data musim asap.** SM2-01 dan PLB-01 kehilangan hari puncak (aturan VALID dan 18 jam), sehingga eksposur dan episode mereka bias ke bawah; MER-01 praktis tidak punya data harian; JMB-02 dikecualikan.
5. **Model PM2.5.** Berbasis pohon sehingga tidak mengekstrapolasi; fitur lokasi konstan per stasiun berfungsi sebagai pengenal stasiun; hanya 26 stasiun dan 2,8 tahun data.
6. **Kecukupan jaringan.** Daftar 38 provinsi dan sentroidnya adalah acuan eksternal (pengetahuan umum, ± ratusan km), bukan data proyek; proksi polusi/asap kandidat berasal dari 3 stasiun Layak terdekat (IDW), tanpa data penduduk, titik panas, atau sebaran gambut. Radius representatif tidak dapat diestimasi karena korelasi antar-stasiun tidak meluruh dengan jarak.
7. **Skor risiko dan prioritas.** Bobot adalah penilaian analis, bukan hasil data; sensitivitas bobot ditampilkan tetapi tidak menggantikan keputusan pemangku kepentingan. Peringkat perbaikan (n = 6) tidak stabil. Tidak ada data penduduk/kemiskinan untuk kerentanan; proksi memakai kapasitas RS dan RR per RS.
8. **Klaster.** Silhouette Ward 0,53 pada 26 stasiun; batas klaster indikatif dan profil stasiun Terbatas kurang pasti.

## Lampiran: grafik dan konsistensi
Grafik: `05_0_flatline_per_stasiun`, `05_1_kinerja_model`, `05_1_permutation_importance`, `05_1_shap`, `05_1_prediksi_vs_teramati`, `05_2_rr_lag_dan_kumulatif`, `05_2_respons_eksposur_kelas`, `05_2_kasus_berlebih_episode`, `05_3_klaster_dendrogram_profil`, `05_3_klaster_pca`, `05_4_bjb01_flatline`, `05_4_polusi_tinggi_kasus_rendah`, `05_5_skor_risiko`, `05_6_redundansi_kelengkapan`, `05_6_peta_jaringan`, `05_6_skor_prioritas` (PNG 1600 px + HTML di `reports/figures/`). Pemeriksaan konsistensi: 12/12 angka acuan notebook 02–04 identik (bagian 8 notebook).
